In [1]:
import pandas as pd
import numpy as np
from supabase import create_client, Client
import os
from dotenv import load_dotenv
from datetime import datetime
from dateutil.relativedelta import relativedelta
import time
import gc
inicio = time.time()
load_dotenv()
# Opção 1: Configurar globalmente
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 casas decimais

# Opção 3: Para um número específico de linhas
pd.set_option('display.max_rows', 100)  # Mostra até 100 linhas

service_key = os.getenv('SUPABASE_SERVICE_KEY')
print("Chave carregada?", service_key is not None)

Chave carregada? True


# Login no Supabase

Como não está ativado o RLS, não é preciso fazer autenticação.

In [2]:
# URL do Projeto
project_url = 'https://mrjrkkbecjyzzwkvouxx.supabase.co'
# Acesso ao cliente
global supabase
supabase: Client = create_client(project_url, service_key)
print("Supabase conectado!")

Supabase conectado!


# Funções

#### Cria Driver

In [3]:
def criar_driver():
    """
    Cria e configura um driver do Selenium Chrome com opções padrão.
    
    Returns:
        WebDriver: Instância configurada do Chrome WebDriver.
    """
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    
    options = Options()
    options.add_experimental_option("prefs", {
        "download.prompt_for_download": False,
        "profile.default_content_settings.popups": 0,
        "profile.default_content_setting_values.automatic_downloads": 1
    })
    return webdriver.Chrome(options=options)

print("Função Criar Driver carregada!")

Função Criar Driver carregada!


## IGPDI

Ações básicas da função abaixo:
1. Verifica se há recente atualização do IGPDI que não está na base.
2. Se houver, baixa nova planilha, trata e cria as colunas e realiza o upload no supabase.
3. Se não houver, traz a base de dados do supabase

#### Verificar atualização do IGPDI


In [4]:
def verificar_atualizacao_igpdi(xpath_data_igpdi, relatorios_a_realizar=True, driver=None):
    """
    Verifica se o IGPDI no Supabase está atualizado comparando a data extraída de um site
    com a data máxima no Supabase acrescida de 3 meses (considerando um lag de 2 meses + dias).
    
    Args:
        xpath_data_igpdi (str): XPath do elemento no site que contém a data do IGPDI.
        supabase: Cliente do Supabase já configurado para acesso aos dados.
        relatorios_a_realizar (bool): Define o comportamento de retorno. Se True, retorna "Extrair";
                                      se False, retorna "Actualizar".
        driver: Instância do Selenium WebDriver (opcional). Se None, cria um novo.
    
    Returns:
        str: Resultado da verificação ("Extrair", "Actualizar", "Atualizado", ou mensagem de erro).
    """
    from datetime import datetime
    from dateutil.relativedelta import relativedelta
    from selenium.webdriver.common.by import By
    
    base_url = "https://sindusconpr.com.br/igp-di-fgv-308-p/"
    timeout = 5
    data_extraida = None
    
    try:
        driver_local = driver or criar_driver()
        driver_local.get(base_url)
        driver_local.implicitly_wait(timeout)
        elemento = driver_local.find_element(By.XPATH, xpath_data_igpdi)
        data_str = elemento.text
        print(f"Data extraída do site: {data_str}")
        data_extraida = datetime.strptime(data_str, "%d/%m/%Y")
        elemento.click()  # Clica no elemento, se necessário
    except Exception as e:
        print(f"Erro ao extrair a data: {e}")
    finally:
        if not driver and 'driver_local' in locals():
            driver_local.quit()
    
    if data_extraida is None:
        return "Erro na extração da data"
    try:
        response = supabase.table("tab_igpdi").select("data").execute()
        dados_supa = response.data if hasattr(response, 'data') else []
    except Exception as e:
        dados_supa = pd.DataFrame({'data':data_extraida.strftime("%Y-%m-%d")})
    if not dados_supa:
        print("Não há dados no Supabase para comparação.")
        return "Sem dados no Supabase"
    
    datas = [datetime.strptime(registro['data'], "%Y-%m-%d") for registro in dados_supa]
    data_maxima = max(datas)
    data_maxima_mais_3_meses = data_maxima + relativedelta(months=3)
    print(f"Data máxima no Supabase: {data_maxima.strftime('%d/%m/%Y')} (+3 meses: {data_maxima_mais_3_meses.strftime('%d/%m/%Y')})")
    
    if data_extraida >= data_maxima_mais_3_meses:
        print("IGPDI desatualizado. Necessária extração.")
        return "Extrair" if relatorios_a_realizar else "Actualizar"
    print("IGPDI atualizado.")
    return "Atualizado"

print("Função Atualizar IGPDI carregada!")

Função Atualizar IGPDI carregada!


#### Tratar IGPDI

In [5]:
def carregar_igpdi(caminho_arquivo, skiprows=2, usecols=[0, 1], colunas=['Data', 'IGPDI']):
    """
    Carrega e limpa dados do IGP-DI da FGV a partir de arquivo Excel.
    
    Parameters:
        caminho_arquivo (str): Caminho completo para o arquivo Excel.
        skiprows (int, default=2): Número de linhas para pular no início.
        usecols (list, default=[0,1]): Colunas para ler do Excel.
        colunas (list, default=['Data','IGPDI']): Nomes das colunas finais.
    
    Returns:
        pd.DataFrame: DataFrame limpo com colunas Data e IGPDI.
    """
    import pandas as pd
    import numpy as np
    
    df = pd.read_excel(caminho_arquivo, skiprows=skiprows, usecols=usecols)
    df = df.dropna(how='all')
    df = df.dropna(subset=[df.columns[1]])
    
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str)
            mask = df[col].str.startswith('=', na=False)
            df.loc[mask, col] = np.nan
    
    df = df.reset_index(drop=True)
    df.columns = colunas
    df['Data'] = pd.to_datetime(df['Data'])
    
    max_igpdi = df['IGPDI'].iloc[-1]
    df['igpdi_atual'] = max_igpdi
    df['deflator'] = df['IGPDI'] / df['igpdi_atual']
    
    data_min = df['Data'].min().strftime('%d/%m/%Y')
    data_max = df['Data'].max().strftime('%d/%m/%Y')
    print(f"✅ Dados carregados: {len(df)} registros (Período: {data_min} a {data_max})")
    print(f"Deflator mais recente: {df['IGPDI'].iloc[-1]:.4f}")
    return df

print("Função Carregar IGPDI carregada!")

Função Carregar IGPDI carregada!


#### Webscrapping

In [6]:
def baixar_dados_igpdi(xpath_igpdi, xpath_data_igpdi, timeout=10, download_wait=7, driver=None):
    """
    Acessa o site do SindusconPR, baixa a série de dados do IGPDI ou extrai do Supabase,
    e retorna um DataFrame com os dados tratados.
    
    Args:
        xpath_igpdi (str): XPath do elemento para download no site.
        supabase: Cliente do Supabase já configurado.
        timeout (int): Tempo máximo de espera para carregamento de elementos (segundos).
        download_wait (int): Tempo de espera após o clique para download (segundos).
        igpdi_atualizado (str): Status de atualização ("Atualizado" para usar Supabase, senão baixa do site).
        driver: Instância do Selenium WebDriver (opcional). Se None, cria um novo.
    
    Returns:
        pd.DataFrame: DataFrame com os dados do IGPDI tratados.
    """
    import os
    import time
    import pandas as pd
    from selenium.webdriver.common.by import By

    # Verificar se o IGPDI está atualizado
    resultado = verificar_atualizacao_igpdi(xpath_data_igpdi)

    # Se estiver atualizado, apenas puxa o bando de dados do Supabase
    if resultado == "Atualizado":
        print("IGPDI atualizado. Extraindo base do Supabase...")
        response = supabase.table("tab_igpdi").select('data','igpdi','igpdi_atual','deflator').execute()
        df_igpdi = pd.DataFrame(response.data)
        df_igpdi['data'] = pd.to_datetime(df_igpdi['data'], errors='coerce')
        for coluna in ['igpdi','igpdi_atual','deflator']:
            # df_igpdi[coluna] = df_igpdi[coluna].str.replace('.', '')
            # df_igpdi[coluna] = df_igpdi[coluna].str.replace(',', '.')
            # Converter para float, substituindo valores inválidos por NaN
            df_igpdi[coluna] = pd.to_numeric(df_igpdi[coluna], errors='coerce')
        print(f"✅ Extração do IGPDI do Supabase concluída ({len(df_igpdi)} registros).")
        return df_igpdi
    
    # Caso não esteja atualizado, realiza o download
    download_dir = os.path.expanduser("~/Downloads")
    arquivos_antes = set(os.listdir(download_dir))
    base_url = "https://sindusconpr.com.br/igp-di-fgv-308-p/"
    
    try:
        driver_local = driver or criar_driver()
        driver_local.get(base_url)
        driver_local.implicitly_wait(timeout)
        serie_link = driver_local.find_element(By.XPATH, xpath_igpdi)
        serie_link.click()
        time.sleep(download_wait)
        
        arquivos_depois = set(os.listdir(download_dir))
        novos_arquivos = arquivos_depois - arquivos_antes
        arquivos_xlsx = [f for f in novos_arquivos if f.endswith('.xlsx')]
        
        if not arquivos_xlsx:
            raise Exception("Arquivo .xlsx não foi baixado.")
        
        arquivo_baixado = os.path.join(download_dir, arquivos_xlsx[0])
        print(f"✅ Arquivo baixado em: {arquivo_baixado}")
    except Exception as e:
        print(f"Erro ao baixar dados: {e}")
        raise
    finally:
        if not driver and 'driver_local' in locals():
            driver_local.quit()
    
    # Tratar dados baixados
    df_igpdi = carregar_igpdi(arquivo_baixado)
    print("✅ Tratamento do IGPDI concluído.")

    dict_rename_igpdi = {
        'Data': 'data',
        'IGPDI': 'igpdi',
        'igpdi_atual': 'igpdi_atual',
        'deflator': 'deflator'
    }
    df_igpdi = df_igpdi.rename(columns=dict_rename_igpdi)    
   # Preparar dados para upload ao Supabase
    df_igpdi_supa = df_igpdi.copy()
    df_igpdi_supa['data'] = df_igpdi_supa['data'].dt.strftime('%Y-%m-%d')
    igpdi_to_supa = df_igpdi_supa.to_dict('records')
    
    # Filtrar linhas novas para upload
    response = supabase.table("tab_igpdi").select("data").execute()
    datas_supa = {r['data'] for r in response.data} if hasattr(response, 'data') else set()
    novas_linhas = [linha for linha in igpdi_to_supa if linha['data'] not in datas_supa]
    
    if novas_linhas:
        try:
            delete_response = supabase.table("tab_igpdi").delete().gte('data', '1900-01-01').execute() # Deletar tudo porque o deflator precisa ser alterado tbm
            upload_response = supabase.table("tab_igpdi").insert(igpdi_to_supa).execute() # Inserir toda a tabela IGPDI
            print(f"✅ {len(novas_linhas)} novas linhas inseridas no Supabase.")
        except Exception as e:
            print(f"Erro ao inserir linhas no Supabase: {e}")
    else:
        print("Não há novas linhas para inserir no Supabase.")
    
    return df_igpdi

print("Função Baixar IGPDI carregada!")

Função Baixar IGPDI carregada!


#### Transformar em numérico

In [7]:
def tratar_colunas_numericas(df, colunas):
    """
    Padroniza e converte colunas numéricas em formato brasileiro para float.
    Remove pontos como separador de milhar, troca vírgula por ponto, converte para float e preenche NaN com zero.
    """
    for col in colunas:
        # Garante que está tudo como string
        df[col] = df[col].astype(str)
        df[col] = pd.to_numeric(df[col], errors='coerce',downcast='integer')
        df[col] = df[col].fillna(0)
    return df

print("Função Transformar Numérico carregada!")

Função Transformar Numérico carregada!


#### Deflacionar

In [8]:
def deflacionar_colunas(df, colunas, nome_deflator='deflator', prefixo_resultado='def_'):
    """
    Cria colunas deflacionadas em relação a um deflator.
    Para cada coluna em 'colunas', cria uma nova coluna prefixada com 'def_' dividindo pelo 'deflator'.
    Garanta que divisões por zero/NaN resultem em NaN.
    """
    for col in colunas:
        coluna_deflacionada = f"{prefixo_resultado}{col}"
        # Usamos .where para evitar dividir por zero ou NaN
        df[coluna_deflacionada] = df[col] / df[nome_deflator]
        df[coluna_deflacionada] = df[coluna_deflacionada].where(df[nome_deflator] != 0)
        df[coluna_deflacionada] = df[coluna_deflacionada].fillna(0)
    return df

print("Função Deflacionar carregada!")

Função Deflacionar carregada!


#### Extração ou Webscrapping

In [9]:
xpath_data_igpdi = '/html/body/div[8]/div/div/div/div[1]/div[1]/div/div/div[3]/div/p[2]/span/strong'
xpath_igpdi = '/html/body/div[8]/div/div/div/div[1]/div[1]/div/div/div[3]/div/div[2]/div/table/tbody/tr/td[3]/a'

df_igpdi = baixar_dados_igpdi(xpath_igpdi, xpath_data_igpdi)

print(df_igpdi.dtypes)
df_igpdi.tail()

Data extraída do site: 07/08/2026
Data máxima no Supabase: 01/05/2026 (+3 meses: 01/08/2026)
IGPDI desatualizado. Necessária extração.
✅ Arquivo baixado em: C:\Users\analy/Downloads\f4c4-serie-historica-igp-di-fgv.xlsx
✅ Dados carregados: 383 registros (Período: 01/08/1994 a 01/06/2026)
Deflator mais recente: 1202.2990
✅ Tratamento do IGPDI concluído.
✅ 1 novas linhas inseridas no Supabase.
data           datetime64[ns]
igpdi                 float64
igpdi_atual           float64
deflator              float64
dtype: object


,data,igpdi,igpdi_atual,deflator
378,2026-02-01,1159.79,1202.30,0.96
379,2026-03-01,1173.04,1202.30,0.98
380,2026-04-01,1201.36,1202.30,1.00
381,2026-05-01,1211.83,1202.30,1.01
382,2026-06-01,1202.30,1202.30,1.00


## Exportar Banco de Dados

In [10]:
# bd_fazenda = pd.DataFrame(tab_fazenda.data)
# bd_fazenda = bd_fazenda.loc[bd_fazenda['ID'].isin([140,88,676])]


# bd_consultor = pd.DataFrame(tab_consultor.data)

# bd_produtor = pd.DataFrame(tab_produtor.data)

# bd_agroindustria = pd.DataFrame(tab_agroindustria.data) 

# bd_rendabruta = pd.DataFrame(tab_rendabruta.data)
# bd_rendabruta = bd_rendabruta.loc[bd_rendabruta['idFazenda'].isin([140,88,676])]

# bd_leiteconsumido = pd.DataFrame(tab_leiteconsumido.data)
# bd_leiteconsumido = bd_leiteconsumido.loc[bd_leiteconsumido['idFazenda'].isin([140,88,676])]

# bd_componentescustos = pd.DataFrame(tab_componentescustos.data)
# bd_componentescustos = bd_componentescustos.loc[bd_componentescustos['idFazenda'].isin([140,88,676])]

# bd_saidaestoque = pd.DataFrame(tab_saidaestoque.data)
# bd_saidaestoque = bd_saidaestoque.loc[bd_saidaestoque['idFazenda'].isin([140,88,676])]

# bd_entradaestoque = pd.DataFrame(tab_entradaestoque.data)
# bd_entradaestoque = bd_entradaestoque.loc[bd_entradaestoque['idFazenda'].isin([140,88,676])]

# bd_custoforrageira = pd.DataFrame(tab_custoforrageira.data)
# bd_custoforrageira = bd_custoforrageira.loc[bd_custoforrageira['idFazenda'].isin([140,88,676])]

# bd_producao = pd.DataFrame(tab_producao.data)
# bd_producao = bd_producao.loc[bd_producao['idFazenda'].isin([140,88,676])]

# bd_itensalimento = pd.DataFrame(tab_itensalimento.data)

# bd_alimentacao = pd.DataFrame(tab_alimentacao.data)
# bd_alimentacao = bd_alimentacao.loc[bd_alimentacao['idFazenda'].isin([140,88,676])]

# bd_energiacombustivel = pd.DataFrame(tab_energiacombustivel.data)
# bd_energiacombustivel = bd_energiacombustivel.loc[bd_energiacombustivel['idFazenda'].isin([140,88,676])]

# bd_maodeobra = pd.DataFrame(tab_maodeobra.data)
# bd_maodeobra = bd_maodeobra.loc[bd_maodeobra['idFazenda'].isin([140,88,676])]

# bd_rebanho = pd.DataFrame(tab_rebanho.data)
# bd_rebanho = bd_rebanho.loc[bd_rebanho['idFazenda'].isin([140,88,676])]

# bd_area = pd.DataFrame(tab_area.data)
# bd_area = bd_area.loc[bd_area['idFazenda'].isin([140,88,676])]

# bd_cultura = pd.DataFrame(tab_cultura.data)
# bd_cultura = bd_cultura.loc[bd_cultura['idFazenda'].isin([140,88,676])]

# bd_qualidadeleite = pd.DataFrame(tab_qualidadeleite.data)
# bd_qualidadeleite = bd_qualidadeleite.loc[bd_qualidadeleite['idFazenda'].isin([140,88,676])]

# bd_patrimonio = pd.DataFrame(tab_patrimonio.data)
# bd_patrimonio = bd_patrimonio.loc[bd_patrimonio['idFazenda'].isin([140,88,676])]

# bd_itenscultura = pd.DataFrame(tab_itenscultura.data)

# bd_sistema = pd.DataFrame(tab_sistema.data)
# bd_sistema = bd_sistema.loc[bd_sistema['idFazenda'].isin([140,88,676])]

# bd_parcela = pd.DataFrame(tab_parcela.data)

# # Criar dicionário com todos os DataFrames
# dataframes = {
#     'Fazenda': bd_fazenda,
#     'Consultor': bd_consultor,
#     'Produtor': bd_produtor,
#     'Agroindustria': bd_agroindustria,
#     'RendaBruta': bd_rendabruta,
#     'LeiteConsumido': bd_leiteconsumido,
#     'ComponentesCustos': bd_componentescustos,
#     'SaidaEstoque': bd_saidaestoque,
#     'EntradaEstoque': bd_entradaestoque,
#     'CustoForrageira': bd_custoforrageira,
#     'Producao': bd_producao,
#     'ItensAlimento': bd_itensalimento,
#     'Alimentacao': bd_alimentacao,
#     'EnergiaCombustivel': bd_energiacombustivel,
#     'MaoDeObra': bd_maodeobra,
#     'Rebanho': bd_rebanho,
#     'Area': bd_area,
#     'Cultura': bd_cultura,
#     'QualidadeLeite': bd_qualidadeleite,
#     'Patrimonio': bd_patrimonio,
#     'ItensCultura': bd_itenscultura,
#     'Sistema': bd_sistema,
#     'Parcela': bd_parcela,
#     'IGPDI': df_igpdi
# }

# # Salvar todos os DataFrames no Excel
# with pd.ExcelWriter('bd_elabore_teste.xlsx', engine='openpyxl') as writer:
#     for nome_aba, df in dataframes.items():
#         df.to_excel(writer, sheet_name=nome_aba, index=False)

# print("Arquivo Excel criado com sucesso!")

## Extrair Parcelas por Lotes

In [11]:
import pandas as pd
from supabase import Client

def extrair_parcelas_em_lotes(supabase: Client, tamanho_lote: int = 10000) -> pd.DataFrame:
    """
    Extrai tab_parcelas em lotes para evitar timeout.

    Args:
        supabase: Cliente do Supabase
        tamanho_lote: Quantidade de linhas por requisição (padrão: 10.000)

    Returns:
        DataFrame consolidado com todas as parcelas
    """
    todas_parcelas = []
    offset = 0

    while True:
        # Buscar lote
        response = (
            supabase.table('tab_parcelas')
            .select('ID', 'idInventario', 'Item', 'nomeTela', 'valor', 'dataPagamento', 'numeroParcela','previsaoPagamento', 'dataPagamento')
            .eq('Excluido', 0)
            .range(offset, offset + tamanho_lote - 1)  # range é inclusivo
            .execute()
        )

        # Se não retornou dados, terminou
        if not response.data:
            break

        todas_parcelas.extend(response.data)

        # Log de progresso (opcional)
        print(f"Extraídas {len(todas_parcelas)} linhas...")

        # Se retornou menos que o lote completo, chegou ao fim
        if len(response.data) < tamanho_lote:
            break

        # Avançar para o próximo lote
        offset += tamanho_lote

    # Converter para DataFrame
    df_parcelas = pd.DataFrame(todas_parcelas)

    return df_parcelas

## Extrair Parcelas por Lotes (Excel)

In [12]:
import pandas as pd
from supabase import Client

def extrair_parcelas_em_lotes_excel(supabase: Client, tamanho_lote: int = 10000) -> pd.DataFrame:
    """
    Extrai tab_parcelas em lotes para evitar timeout.

    Args:
        supabase: Cliente do Supabase
        tamanho_lote: Quantidade de linhas por requisição (padrão: 10.000)

    Returns:
        DataFrame consolidado com todas as parcelas
    """
    todas_parcelas = []
    offset = 0

    while True:
        # Buscar lote
        response = (
            supabase.table('xlsx_parcelas')
            .select('idParcela', 'idInventario', 'Item', 'nomeTela', 'valor', 'dataPagamento', 'numeroParcela','previsaoPagamento', 'dataPagamento')
            .eq('Excluido', 0)
            .range(offset, offset + tamanho_lote - 1)  # range é inclusivo
            .execute()
        )

        # Se não retornou dados, terminou
        if not response.data:
            break

        todas_parcelas.extend(response.data)

        # Log de progresso (opcional)
        print(f"Extraídas {len(todas_parcelas)} linhas...")

        # Se retornou menos que o lote completo, chegou ao fim
        if len(response.data) < tamanho_lote:
            break

        # Avançar para o próximo lote
        offset += tamanho_lote

    # Converter para DataFrame
    df_parcelas = pd.DataFrame(todas_parcelas)
    df_parcelas.rename(columns={'idParcela':'ID'}, inplace=True)

    return df_parcelas

## Extrair Entrada de Estoque em Lotes

In [13]:
def extrair_entradaestoque_em_lotes(supabase: Client, tamanho_lote: int = 10000) -> pd.DataFrame:
    """
    Extrai tab_entradaestoque em lotes para evitar timeout.

    Args:
        supabase: Cliente do Supabase
        tamanho_lote: Quantidade de linhas por requisição (padrão: 10.000)

    Returns:
        DataFrame consolidado com todas as entradaestoque
    """
    todas_entradaestoque = []
    offset = 0

    while True:
        # Buscar lote
        response = (
            supabase.table("tab_entradaestoque")
            .select('idFazenda', 'ID','Tela','idRelacionamento', 'idItem','qntComprada','qntConsumida','valorUnitario',
                    'Excluido','mesReferencia')
            .eq("Excluido", 0)
            .range(offset, offset + tamanho_lote - 1)
            .execute()
        )

        # Se não retornou dados, terminou
        if not response.data:
            break

        todas_entradaestoque.extend(response.data)

        # Log de progresso (opcional)
        print(f"Extraídas {len(todas_entradaestoque)} linhas...")

        # Se retornou menos que o lote completo, chegou ao fim
        if len(response.data) < tamanho_lote:
            break

        # Avançar para o próximo lote
        offset += tamanho_lote

    # Converter para DataFrame
    df_entradaestoque = pd.DataFrame(todas_entradaestoque)

    return df_entradaestoque

# Tratamento das Tabelas Dimensionais
1. Fazenda
2. Consultor
3. Produtor
4. Agroindústria
5. Dimensão fazenda: Merge das 4 etapas anterior

## 1. Fazenda

In [14]:
# Tab_Fazenda
tab_fazenda = (
    supabase.table("tab_fazenda").
    select("id", "Excluido", "aprovacao", "idProdutor", "idConsultor",
           "idAgroindustria", "nomeFazenda", "ufFazenda", "codAgroindustria",
           "dteEntradaInicial", "dteSaida", "regiaoLeiteira", "Status")
    .eq("Status", "Ativo")
    .eq("Excluido", 0)
    .eq("aprovacao","Aprovado")
    .execute()
)

# Transformar Supabase em DF
df_fazenda = pd.DataFrame(tab_fazenda.data)
# Selecionar colunas desejadas. Usaremos apenas as necessárias para apresentação de dados
df_fazenda = df_fazenda.loc[:, ['id', 'idProdutor', 'idConsultor', 'idAgroindustria',
                                'nomeFazenda', 'ufFazenda', 'codAgroindustria',
                                'dteEntradaInicial', 'dteSaida', 'regiaoLeiteira']]

# 1. Converter para datetime (o Pandas reconhece perfeitamente este formato ISO 8601)
df_fazenda['dteEntradaInicial'] = pd.to_datetime(pd.to_datetime(df_fazenda['dteEntradaInicial'], errors='coerce').dt.date, format='%Y-%m-%d')
# df_fazenda['dteEntradaInicial'] = df_fazenda['dteEntradaInicial'].dt.date
df_fazenda['dteSaida'] = pd.to_datetime(pd.to_datetime(df_fazenda['dteSaida'], errors='coerce').dt.date, format='%Y-%m-%d')
## Tornar compartilhamento de fazenda por 2+ consultores em linhas diferentes
# Passo 1: Substituir None por NaN (já é o padrão no pandas, mas garantimos)
df_fazenda['idConsultor'] = df_fazenda['idConsultor'].replace({None: pd.NA})
# Passo 2: Separar os valores com ';' em linhas diferentes
# Primeiro, convertemos a coluna para string para usar str.split(), depois explodimos as listas
df_fazenda['idConsultor'] = df_fazenda['idConsultor'].astype(str).str.split(';')
df_fazenda = df_fazenda.explode('idConsultor')
# Passo 3: Converter os valores para inteiro, tratando NaN
# Valores como 'nan' ou pd.NA serão mantidos como NaN
tratar_colunas_numericas(df_fazenda, ['idConsultor', 'id', 'idAgroindustria'])

# Resetar o índice, se necessário
df_fazenda.reset_index(drop=True, inplace=True)

# Mostrar colunas e tipos
print(df_fazenda.dtypes)

# Mostrar últimas 5 linhas
# df_fazenda.tail()

id                            int16
idProdutor                    int64
idConsultor                 float64
idAgroindustria             float64
nomeFazenda                  object
ufFazenda                    object
codAgroindustria             object
dteEntradaInicial    datetime64[ns]
dteSaida             datetime64[ns]
regiaoLeiteira               object
dtype: object


## Consultor

In [15]:
# Tab_Consultores
tab_consultor = (
    supabase.table("tab_consultor")
    .select('ID', 'nomeConsultor', 'Status', 'Excluido', 'aprovacao')
    .eq("Status", "Ativo")
    .eq("Excluido", 0)
    .execute()
)

# Transformar Supabase em DF
df_consultores = pd.DataFrame(tab_consultor.data)
# Selecionar colunas
df_consultores = df_consultores[['ID', 'nomeConsultor']]
# Renomear o id -> idConsultor para poder dar o merge depois
df_consultores.columns = ['idConsultor', 'nomeconsultor']
# Mostrar tipo das colunas
print(df_consultores.dtypes)
# Mostrar últimas 5 linhas
# df_consultores.tail()

idConsultor       int64
nomeconsultor    object
dtype: object


## Produtor

In [16]:

# Tab_Produtor
tab_produtor = (
    supabase.table("tab_produtor")
    .select("ID", 'nomeProdutor', 'Status', 'Excluido', 'aprovacao')
    .eq("Status", "Ativo")
    .eq("Excluido", 0)
    .execute()
)

# Transformar Supabase em DF
df_produtor = pd.DataFrame(tab_produtor.data)
# Selecionar colunas
df_produtor = df_produtor[['ID', 'nomeProdutor']]
# Renomear o id -> idConsultor para poder dar o merge depois
df_produtor.columns = ['idProdutor', 'nomeprodutor']
# Mostrar tipo das colunas
print(df_produtor.dtypes)
# Mostrar últimas 5 linhas
df_produtor.tail()

# df_produtor.loc[df_produtor['idProdutor']==947]

idProdutor       int64
nomeprodutor    object
dtype: object


,idProdutor,nomeprodutor
950,1150,FABIO HENRIQUE MENDES COSTA
951,1140,FABRICIO BEZERRA DIDIER LEITE
952,1145,GUSTAVO GOULART DE CASTRO
953,1149,JOSE DE AZEVEDO JUNIOR
954,1148,JOAQUIM DANIEL LUIZ


## Agroindústria

In [17]:
# Tab_Agroindustria
tab_agroindustria = (
    supabase.table("tab_agroindustria")
    .select("ID", "nomeAgroindustria", "Status", "Excluido")
    .eq("Status", "Ativo")
    .eq("Excluido", 0)
    .execute()
)
# Transformar Supabase em DF
df_agroindustria = pd.DataFrame(tab_agroindustria.data)
# Selecionar colunas
df_agroindustria = df_agroindustria[['ID', 'nomeAgroindustria']]
# Renomear o id -> idConsultor para poder dar o merge depois
df_agroindustria.columns = ['idAgroindustria', 'nomeagroindustria']
# Mostrar tipo das colunas
print(df_agroindustria.dtypes)
# Mostrar últimas 5 linhas
# df_agroindustria.tail()

idAgroindustria       int64
nomeagroindustria    object
dtype: object


## Dimensão Fazenda

In [18]:
d_fazenda = df_fazenda.copy()
# Merge com Consultor
d_fazenda = d_fazenda.merge(
    df_consultores[['idConsultor', 'nomeconsultor']],
    on='idConsultor',
    how='left'
)

# Merge com Produtor
d_fazenda = d_fazenda.merge(
    df_produtor[['idProdutor', 'nomeprodutor']],
    on='idProdutor',
    how='left'
)

# Merge com Agroindustria
d_fazenda = d_fazenda.merge(
    df_agroindustria[['idAgroindustria', 'nomeagroindustria']],
    on='idAgroindustria',
    how='left'
)

# Renomear a coluna 'ID' para 'idFazenda'
d_fazenda = d_fazenda.rename(columns={'id': 'idFazenda'})

# Mostra colunas e seus tipos
print(d_fazenda.dtypes)

# Mostrar últimas 5 linhas
d_fazenda.tail()

# Deletar tabelas que formaram o d_fazenda 
#del df_fazenda, df_consultores, df_produtor, df_agroindustria, tab_fazenda, tab_produtor, tab_consultor, tab_agroindustria
#gc.collect()

print(d_fazenda.head())

idFazenda                     int16
idProdutor                    int64
idConsultor                 float64
idAgroindustria             float64
nomeFazenda                  object
ufFazenda                    object
codAgroindustria             object
dteEntradaInicial    datetime64[ns]
dteSaida             datetime64[ns]
regiaoLeiteira               object
nomeconsultor                object
nomeprodutor                 object
nomeagroindustria            object
dtype: object
   idFazenda  idProdutor  idConsultor  idAgroindustria  \
0       1171        1185        95.00             2.00   
1       1132         541       112.00             2.00   
2       1135        1155       124.00             3.00   
3       1177        1208       120.00             2.00   
4       1187        1186        98.00             7.00   

                           nomeFazenda ufFazenda codAgroindustria  \
0                FAZENDA COPACABANA II        BA          LR11833   
1             FAZENDA RIBEIRAO 

## Itens Alimentação/Forrageira

In [19]:
# Dimensão dos Itens de Alimento/Produção de Forrageira 
tab_itensalimento = (
    supabase.table("tab_itensalimento")
    .select("ID","cultura","producao","tipo")
    .execute()
)

# Acessar Itens Alimento
df_itensalimento = pd.DataFrame(tab_itensalimento.data)

## Leite Consumido

### Sharepoint

In [20]:
# Leite Consumido
tab_leiteconsumido = (
    supabase.table("tab_componentescusto")
    .select('idFazenda', 'ID', 'mesReferencia', 'consumoLeiteBezerro', 'consumoMaoObraFamiliar',
           'consumoMaoObraContratada', 'consumoLeiteDescartado')
    .execute()
)

df_leiteconsumido = pd.DataFrame(tab_leiteconsumido.data)

# Criar consumo total
df_leiteconsumido['leiteConsumido'] = df_leiteconsumido[['consumoLeiteBezerro','consumoMaoObraFamiliar','consumoMaoObraContratada','consumoLeiteDescartado']].sum(axis=1)


## Transformar mes em date
df_leiteconsumido['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_leiteconsumido['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)


# # Deletar tabela original 
# del tab_leiteconsumido

df_leiteconsumido = df_leiteconsumido.groupby(['idFazenda','mesReferencia']).sum().reset_index()

# Mostrar colunas e tipos
print(df_leiteconsumido.dtypes)

# Mostrar primeiras 5 linhas
#df_leiteconsumido.loc[df_leiteconsumido['idFazenda']==99]

idFazenda                            int64
mesReferencia               datetime64[ns]
ID                                   int64
consumoLeiteBezerro                float64
consumoMaoObraFamiliar             float64
consumoMaoObraContratada           float64
consumoLeiteDescartado             float64
leiteConsumido                     float64
dtype: object


### Excel

In [21]:
# Leite Consumido
tab_leiteconsumido_excel = (
    supabase.table("tab_componentescusto")
    .select('idFazenda', 'ID', 'mesReferencia', 'consumoLeiteBezerro', 'consumoMaoObraFamiliar',
           'consumoMaoObraContratada', 'consumoLeiteDescartado')
    .execute()
)

df_leiteconsumido_excel = pd.DataFrame(tab_leiteconsumido_excel.data)

# Criar consumo total
df_leiteconsumido_excel['leiteConsumido'] = df_leiteconsumido_excel[['consumoLeiteBezerro','consumoMaoObraFamiliar','consumoMaoObraContratada','consumoLeiteDescartado']].sum(axis=1)


## Transformar mes em date
df_leiteconsumido_excel['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_leiteconsumido_excel['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)


# # Deletar tabela original 
# del tab_leiteconsumido

df_leiteconsumido_excel = df_leiteconsumido_excel.groupby(['idFazenda','mesReferencia']).sum().reset_index()

# Mostrar colunas e tipos
print(df_leiteconsumido_excel.dtypes)

# Mostrar primeiras 5 linhas
#df_leiteconsumido_excel.loc[df_leiteconsumido_excel['idFazenda']==99]

idFazenda                            int64
mesReferencia               datetime64[ns]
ID                                   int64
consumoLeiteBezerro                float64
consumoMaoObraFamiliar             float64
consumoMaoObraContratada           float64
consumoLeiteDescartado             float64
leiteConsumido                     float64
dtype: object


## Renda Bruta

### Sharepoint

In [22]:
# Tab_ComponenteReceita
tab_rendabruta = (
    supabase.table("tab_componentesrendabruta")
    .select('idFazenda', 'mesReferencia', 'leiteVendido', 'precoLeite',
           'vendaAnimais', 'qtdeVendaVolumoso', 'outrasReceitas', 'derivadosVenda',
           'derivadosPreco', 'bonificacaoPreco', 'divisaodesobrasPreco', 'qtdeVendaConcentrado',
           'parcela', 'emprestimosRecebidos', 'parcelaEmprestimosRecebidos')
    .execute()
)

df_rendabruta_shp = pd.DataFrame(tab_rendabruta.data)
# Nomes novos das colunas
nome_colunas_rb = {
    'qtdeVendaVolumoso':'vendaVolumoso',
    'qtdeVendaConcentrado':'vendaConcentrado'
}
# Renomear colunas
df_rendabruta_shp = df_rendabruta_shp.rename(columns=nome_colunas_rb)

print(f"Tipo das Colunas antes: {df_rendabruta_shp.dtypes}")

## Transformar mes em date
df_rendabruta_shp['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_rendabruta_shp['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

## Converter colunas de object para float
# # Lista de colunas a converter
colunas_para_float = ['vendaVolumoso', 'vendaConcentrado']
# Converter colunas para float, tratando vírgulas e erros
for coluna in colunas_para_float:
    # Substituir vírgula por ponto, se necessário
    if df_rendabruta_shp[coluna].dtype == 'object':
        df_rendabruta_shp[coluna] = df_rendabruta_shp[coluna].str.replace('.', '')
        df_rendabruta_shp[coluna] = df_rendabruta_shp[coluna].str.replace(',', '.')
    # Converter para float, substituindo valores inválidos por NaN
    df_rendabruta_shp[coluna] = pd.to_numeric(df_rendabruta_shp[coluna], errors='coerce')
# Opcional: Substituir NaN por 0.0
df_rendabruta_shp[colunas_para_float] = df_rendabruta_shp[colunas_para_float].fillna(0.0)

## Deflacionar preços
# Merge Renda Bruta com IGPDI
df_rendabruta_shp = df_rendabruta_shp.merge(
    df_igpdi[['data', 'deflator']],
    left_on='mesReferencia',
    right_on='data',
    how='left')

colunas_deflacionar = ['precoLeite', 'vendaAnimais', 'vendaVolumoso',
                       'outrasReceitas', 'derivadosPreco', 'bonificacaoPreco',
                       'divisaodesobrasPreco', 'vendaConcentrado', 'emprestimosRecebidos']

colunas_manter = ['idFazenda','mesReferencia', 'leiteVendido',
                 'derivadosVenda', 'parcela', 'parcelaEmprestimosRecebidos']
for coluna in colunas_deflacionar:
    nome_coluna = 'def_'+coluna
    df_rendabruta_shp[nome_coluna] = df_rendabruta_shp[coluna]/df_rendabruta_shp['deflator']
    colunas_manter.append(nome_coluna)

# Substituir colunas monetárias por colunas deflacionadas
df_rendabruta_shp = df_rendabruta_shp.loc[:,colunas_manter]

## Mesclar com Leite Consumido
# Merge com o DF do leite consumido
df_rendabruta_shp = (
    df_rendabruta_shp
    .merge(df_leiteconsumido.loc[:, ['idFazenda', 'mesReferencia', 'leiteConsumido']],
           on=['idFazenda', 'mesReferencia'],
          how='left')
)

# Criar o leite produzido = Vendido + Consumido
df_rendabruta_shp['leiteProduzido'] = df_rendabruta_shp[['leiteVendido', 'derivadosVenda', 'leiteConsumido']].sum(axis=1)

# Criar coluna de Renda do Leite Vendido
df_rendabruta_shp['def_rendaLeiteProduzido'] = df_rendabruta_shp[['leiteVendido', 'leiteConsumido']].sum(axis=1) * df_rendabruta_shp['def_precoLeite']
# Criar coluna de Renda do Derivado
df_rendabruta_shp['def_rendaDerivado'] = df_rendabruta_shp['derivadosVenda'] * df_rendabruta_shp['def_derivadosPreco']
# Criar coluna de Renda do Leite = Leite Vendido + Derivado + Bonificações
df_rendabruta_shp['def_rendaLeite'] = df_rendabruta_shp[['def_rendaLeiteProduzido', 'def_rendaDerivado', 'def_bonificacaoPreco']].sum(axis=1)
# Criar Renda da Atividade 
df_rendabruta_shp['def_rendaAtividade'] = df_rendabruta_shp[['def_rendaLeite', 'def_vendaAnimais', 'def_vendaConcentrado', 'def_vendaVolumoso',
                                                     'def_outrasReceitas', 'def_emprestimosRecebidos']].sum(axis=1)

# Deletar tabela original 
# del tab_rendabruta 
# gc.collect()

# Mostra tipos das colunas
print(df_rendabruta_shp.dtypes)
df_rendabruta_shp = df_rendabruta_shp[~df_rendabruta_shp.duplicated(subset=['idFazenda', 'mesReferencia'], keep='first')].sort_values(['idFazenda','mesReferencia'])

Tipo das Colunas antes: idFazenda                      float64
mesReferencia                   object
leiteVendido                   float64
precoLeite                     float64
vendaAnimais                   float64
vendaVolumoso                   object
outrasReceitas                 float64
derivadosVenda                 float64
derivadosPreco                 float64
bonificacaoPreco               float64
divisaodesobrasPreco           float64
vendaConcentrado                object
parcela                        float64
emprestimosRecebidos           float64
parcelaEmprestimosRecebidos    float64
dtype: object
idFazenda                             float64
mesReferencia                  datetime64[ns]
leiteVendido                          float64
derivadosVenda                        float64
parcela                               float64
parcelaEmprestimosRecebidos           float64
def_precoLeite                        float64
def_vendaAnimais                      float64
def_venda

### Excel

In [23]:
tab_rendabruta = (
    supabase.table("xlsx_receita")
    .select('idFazenda', 'mesReferencia', 'leiteVendido', 'precoLeite',
           'vendaAnimais', 'qtdeVendaVolumoso', 'outrasReceitas', 'derivadosVenda',
           'derivadosPreco', 'bonificacaoPreco', 'divisaodesobrasPreco', 'qtdeVendaConcentrado',
           'parcela', 'emprestimosRecebidos', 'parcelaEmprestimosRecebidos')
    .execute()
)

df_rendabruta_excel = pd.DataFrame(tab_rendabruta.data)

# Nomes novos das colunas
nome_colunas_rb = {
    'ID':'idRendaBruta',
    'qtdeVendaVolumoso':'vendaVolumoso',
    'qtdeVendaConcentrado':'vendaConcentrado'
}
# Renomear colunas
df_rendabruta_excel = df_rendabruta_excel.rename(columns=nome_colunas_rb)

print(f"Tipo das Colunas antes: {df_rendabruta_excel.dtypes}")

## Transformar mes em date
df_rendabruta_excel['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_rendabruta_excel['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

## Converter colunas de object para float
# # Lista de colunas a converter
colunas_para_float = ['vendaVolumoso', 'vendaConcentrado']
# Converter colunas para float, tratando vírgulas e erros
for coluna in colunas_para_float:
    # Substituir vírgula por ponto, se necessário
    if df_rendabruta_excel[coluna].dtype == 'object':
        df_rendabruta_excel[coluna] = df_rendabruta_excel[coluna].str.replace('.', '')
        df_rendabruta_excel[coluna] = df_rendabruta_excel[coluna].str.replace(',', '.')
    # Converter para float, substituindo valores inválidos por NaN
    df_rendabruta_excel[coluna] = pd.to_numeric(df_rendabruta_excel[coluna], errors='coerce')
# Opcional: Substituir NaN por 0.0
df_rendabruta_excel[colunas_para_float] = df_rendabruta_excel[colunas_para_float].fillna(0.0)

## Deflacionar preços
# Merge Renda Bruta com IGPDI
df_rendabruta_excel = df_rendabruta_excel.merge(
    df_igpdi[['data', 'deflator']],
    left_on='mesReferencia',
    right_on='data',
    how='left')

colunas_deflacionar = ['precoLeite', 'vendaAnimais', 'vendaVolumoso',
                       'outrasReceitas', 'derivadosPreco', 'bonificacaoPreco',
                       'divisaodesobrasPreco', 'vendaConcentrado', 'emprestimosRecebidos']

colunas_manter = ['idFazenda','mesReferencia', 'leiteVendido',
                 'derivadosVenda', 'parcela', 'parcelaEmprestimosRecebidos']
for coluna in colunas_deflacionar:
    nome_coluna = 'def_'+coluna
    df_rendabruta_excel[nome_coluna] = df_rendabruta_excel[coluna]/df_rendabruta_excel['deflator']
    colunas_manter.append(nome_coluna)

# Substituir colunas monetárias por colunas deflacionadas
df_rendabruta_excel = df_rendabruta_excel.loc[:,colunas_manter]

## Mesclar com Leite Consumido
# Merge com o DF do leite consumido
df_rendabruta_excel = (
    df_rendabruta_excel
    .merge(df_leiteconsumido_excel.loc[:, ['idFazenda', 'mesReferencia', 'leiteConsumido']],
           on=['idFazenda', 'mesReferencia'],
          how='left')
)

# Criar o leite produzido = Vendido + Consumido
df_rendabruta_excel['leiteProduzido'] = df_rendabruta_excel[['leiteVendido', 'derivadosVenda', 'leiteConsumido']].sum(axis=1)

# Criar coluna de Renda do Leite Vendido
df_rendabruta_excel['def_rendaLeiteProduzido'] = df_rendabruta_excel[['leiteVendido', 'leiteConsumido']].sum(axis=1) * df_rendabruta_excel['def_precoLeite']
# Criar coluna de Renda do Derivado
df_rendabruta_excel['def_rendaDerivado'] = df_rendabruta_excel['derivadosVenda'] * df_rendabruta_excel['def_derivadosPreco']
# Criar coluna de Renda do Leite = Leite Vendido + Derivado + Bonificações
df_rendabruta_excel['def_rendaLeite'] = df_rendabruta_excel[['def_rendaLeiteProduzido', 'def_rendaDerivado', 'def_bonificacaoPreco']].sum(axis=1)
# Criar Renda da Atividade 
df_rendabruta_excel['def_rendaAtividade'] = df_rendabruta_excel[['def_rendaLeite', 'def_vendaAnimais', 'def_vendaConcentrado', 'def_vendaVolumoso',
                                                     'def_outrasReceitas', 'def_emprestimosRecebidos']].sum(axis=1)

# Passar IdFazenda para float
df_rendabruta_excel['idFazenda'] = df_rendabruta_excel['idFazenda'].astype('float')

# Deletar tabela original 
# del tab_rendabruta 
# gc.collect()

# Mostra tipos das colunas
print(df_rendabruta_excel.dtypes)
df_rendabruta_excel = df_rendabruta_excel[~df_rendabruta_excel.duplicated(subset=['idFazenda', 'mesReferencia'], keep='first')].sort_values(['idFazenda','mesReferencia'])

#df_rendabruta_excel.head()

Tipo das Colunas antes: idFazenda                        int64
mesReferencia                   object
leiteVendido                   float64
precoLeite                     float64
vendaAnimais                   float64
vendaVolumoso                  float64
outrasReceitas                 float64
derivadosVenda                 float64
derivadosPreco                 float64
bonificacaoPreco               float64
divisaodesobrasPreco           float64
vendaConcentrado               float64
parcela                        float64
emprestimosRecebidos           float64
parcelaEmprestimosRecebidos    float64
dtype: object
idFazenda                             float64
mesReferencia                  datetime64[ns]
leiteVendido                          float64
derivadosVenda                        float64
parcela                               float64
parcelaEmprestimosRecebidos           float64
def_precoLeite                        float64
def_vendaAnimais                      float64
def_venda

### Unificar

In [24]:
# Unificar sharepoint e excel
df_rendabruta = pd.concat([df_rendabruta_excel, df_rendabruta_shp]) 
# Visualizar 
# print(df_rendabruta.loc[df_rendabruta['idFazenda']==131].head())

## Custo de Forrageira

In [25]:
# Custo de Forrageira
tab_custoforrageira = (
    supabase.table("tab_custoforrageira")
    .select('idFazenda','ID','mesReferencia','idCultura','item','id_item','Excluido','AlimentoCultura',
            'Operacao','qtdeComprada','qtdeConsumida','valorUnitario','idArea','etapa','culturasPlantadas')
    .eq("Excluido",0)
    .execute()
)

df_custoforrageira = pd.DataFrame(tab_custoforrageira.data)
# Mostrar tipo de colunas antes do tratamento
print(f"Tipos de colunas antes do tratamento:\n{df_custoforrageira.dtypes}")

# Trocar tipo de coluna: qntConsumida e valorUnitario -> float, MesReferencia -> Datetime
# Transformar mes em date
df_custoforrageira['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_custoforrageira['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# Mostrar tipos de colunas
print(f"Formato das colunas depois:\n{df_custoforrageira.dtypes}")

# Criar coluna com o primeiro dia do mês, para poder usar como referência no merge
df_custoforrageira['priDiaMes'] = df_custoforrageira['mesReferencia'].dt.to_period('M').dt.to_timestamp()

# Mesclar com o IGPDI para poder deflacionar cada item
df_custoforrageira = (
    df_custoforrageira
    .merge(df_igpdi[['data','deflator']],
           left_on='priDiaMes',
           right_on='data',
           how='left')
)

# Deletar tabela original 
del tab_custoforrageira 
gc.collect() 

# Deflacionar os valores unitários
df_custoforrageira['def_valorUnitario'] = df_custoforrageira['valorUnitario']/df_custoforrageira['deflator']

# Mostrar o tamanho do dataframe
print(f"Shape: {df_custoforrageira.shape}")
#df_custoforrageira.head()

Tipos de colunas antes do tratamento:
idFazenda              int64
ID                     int64
mesReferencia         object
idCultura            float64
item                  object
id_item                int64
Excluido               int64
AlimentoCultura       object
Operacao              object
qtdeComprada         float64
qtdeConsumida        float64
valorUnitario        float64
idArea               float64
etapa                 object
culturasPlantadas     object
dtype: object
Formato das colunas depois:
idFazenda                     int64
ID                            int64
mesReferencia        datetime64[ns]
idCultura                   float64
item                         object
id_item                       int64
Excluido                      int64
AlimentoCultura              object
Operacao                     object
qtdeComprada                float64
qtdeConsumida               float64
valorUnitario               float64
idArea                      float64
etapa            

## Produção

In [ ]:
# Produção de Forrageira
tab_producao = (
    supabase.table("tab_producao")
    .select("idFazenda","ID","idCultura","idItem","areaProduzida","producao",
            "dataColheita","Excluido")
    .eq("Excluido",0)
    .execute()
)

df_producao = pd.DataFrame(tab_producao.data)
# Mostrar tipo de colunas antes do tratamento
print(f"Tipos de colunas antes do tratamento:\n{df_producao.dtypes}")

# Trocar tipo de coluna: qntConsumida e valorUnitario -> float, MesReferencia -> Datetime
# Transformar mes em date
df_producao['dataColheita'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_producao['dataColheita'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# Mostrar tipos de colunas
print(f"Formato das colunas depois:\n{df_producao.dtypes}")
#df_producao.head()

Tipos de colunas antes do tratamento:
idFazenda          int64
ID                 int64
idCultura          int64
idItem             int64
areaProduzida    float64
producao         float64
dataColheita      object
Excluido           int64
dtype: object
Formato das colunas depois:
idFazenda                 int64
ID                        int64
idCultura                 int64
idItem                    int64
areaProduzida           float64
producao                float64
dataColheita     datetime64[ns]
Excluido                  int64
dtype: object


## Encontrar o Valor Unitário da Forrageira Produzida
1. A partir da Tabela de Custo de Forrageira, precisa-se
    1. Excluir os que possuem idCultura e Etapa nula.
    2. Criar coluna de valorTotal (valorUnitario * qtdeConsumida).
    3. Agregar segundo Etapa, idCultura, idFazenda.
2. Unir essa nova tabela com tab_producao.
    1.  Importar a coluna de valorTotal.
    2.  Unir o idItem com o tipo de alimento.
    3.  Criar tabela auxiliar com a área total da área produzida.
    4.  Ratear por área.
    5.  Dividir os custos por etapa e tipo.
    6.  Agrupar e dividir por produção.

In [27]:
# =============== ETAPA 1 - CUSTO DE FORRAGEIRA =============================
# Achar valor unitário na tabela de custo de forrageira
df_custounitarioforrageira = df_custoforrageira.loc[
    df_custoforrageira['idCultura'].notna() &
    df_custoforrageira['idArea'].notna() &
    df_custoforrageira['etapa'].notna(),
    ['idFazenda','idCultura','etapa','qtdeConsumida','def_valorUnitario']
].copy()

# Criar coluna de valorTotal do Custo de Forrageira por idCultura
df_custounitarioforrageira['valorTotal'] = df_custounitarioforrageira['qtdeConsumida']*df_custounitarioforrageira['def_valorUnitario']

# Agregar por etapa e idCultura
df_custounitarioforrageira = (
    df_custounitarioforrageira[['idCultura','etapa','valorTotal']]
    .groupby(['idCultura','etapa'])
    .sum()
    .reset_index()
)
# =============== ETAPA 2 - PRODUÇÃO DE FORRAGEIRA ===========================
# Criar dataframe de ETL
df_producao_etl = df_producao.copy()
# Mesclar com o Item Cultura
df_producao_etl = df_producao_etl.merge(df_itensalimento[['ID','tipo']], left_on='idItem', right_on='ID', how='left')
# Renomear as colunas com mesmo nome
df_producao_etl = df_producao_etl.rename(columns={'ID_x':'idProducao', 'ID_y':'idItem2'})
# Mesclar com custoTotal da Forrageira
df_producao_etl = df_producao_etl.merge(df_custounitarioforrageira, on='idCultura', how='left')
# Criar df com área total por idCultura
df_area_produzida = (
    df_producao_etl[['idCultura','areaProduzida']]
    .drop_duplicates()
    .groupby('idCultura')
    .sum()
    .reset_index()
).copy()
# Renomear a coluna
df_area_produzida = df_area_produzida.rename(columns={'areaProduzida':'areaProduzidaTotal'})
# Mesclar com a área total produzida
df_producao_etl = df_producao_etl.merge(df_area_produzida, on='idCultura', how='left')
# Criar proporcao de área
df_producao_etl['propAreaProduzida'] = df_producao_etl['areaProduzida']/df_producao_etl['areaProduzidaTotal']
# Agregar por idCultura e idProducao e criar uma contagem
df_contar_producao = (
    df_producao_etl[['idCultura','idProducao','tipo']]
    .drop_duplicates()
    .groupby(['idCultura','tipo'])
    .count()
    .reset_index()
)
df_contar_producao = df_contar_producao.rename(columns={'idProducao':'countProducao'})
# Mesclar a nova contagem com o df de ETL
df_producao_etl = df_producao_etl.merge(df_contar_producao, on=['idCultura','tipo'], how='left')
# Define condições para cada etapa
condicoes_rateio = [
    (df_producao_etl['etapa'] == 'Colheita+Ensilagem Planta Inteira') & (df_producao_etl['tipo']=='Volumoso') & (df_producao_etl['countProducao'] == 1),
    (df_producao_etl['etapa'] == 'Colheita+Ensilagem Planta Inteira') & (df_producao_etl['tipo']=='Concentrado') & (df_producao_etl['countProducao'] == 1),
    (df_producao_etl['etapa'] == 'Colheita + Ensilagem Grão') & (df_producao_etl['tipo']=='Concentrado') & (df_producao_etl['countProducao'] == 1),
    (df_producao_etl['etapa'] == 'Colheita + Ensilagem Grão') & (df_producao_etl['tipo']=='Volumoso') & (df_producao_etl['countProducao'] == 1)
]
resultado_rateio = [1,0,1,0]
# Inserir nova coluna
df_producao_etl['rateioCusto'] = np.select(condicoes_rateio,resultado_rateio,default=df_producao_etl['propAreaProduzida'])
# Criar coluna rateada
df_producao_etl['valorTotalRateado'] = df_producao_etl['valorTotal'] * df_producao_etl['rateioCusto']
# Agrupar sem etapa para obter o custo da produção
df_producao_etl = (
    df_producao_etl[['idFazenda', 'idProducao', 'idCultura', 'idItem','tipo', 'areaProduzida','producao', 'dataColheita',
                     'areaProduzidaTotal', 'propAreaProduzida','valorTotalRateado']]
    .groupby(['idFazenda', 'idProducao', 'idCultura', 'idItem', 'areaProduzida',
       'producao', 'dataColheita', 'areaProduzidaTotal', 'propAreaProduzida','tipo'])
    .sum()
    .reset_index()
)
# Criar coluna de custounitario
df_producao_etl['custoUnitarioForrageira'] = df_producao_etl['valorTotalRateado'] / df_producao_etl['producao']

# Deletar tabelas utilizadas no processo 
del df_custounitarioforrageira, df_area_produzida
gc.collect()
print(f"Formato das colunas depois:\n{df_producao_etl.dtypes}")
#df_producao_etl.head()

Formato das colunas depois:
idFazenda                           int64
idProducao                          int64
idCultura                           int64
idItem                              int64
areaProduzida                     float64
producao                          float64
dataColheita               datetime64[ns]
areaProduzidaTotal                float64
propAreaProduzida                 float64
tipo                               object
valorTotalRateado                 float64
custoUnitarioForrageira           float64
dtype: object


## Encontrar os Valores Unitários finais da Entrada de Estoque

1. Definir tipo das colunas da Entrada de Estoque
2. Deflacionar Valores
3. Merge da Entrada de Estoque com Produção usando ID (Produção) e idRelacionamento (Entrada)
4. Substituir o valor da Tela de Culturas Plantadas pelo da Producao
5. Garantir que os valores que não são Culturas Plantadas sejam os mesmos

In [28]:
# df_entradaestoque = pd.DataFrame(tab_entradaestoque.data)
df_entradaestoque = extrair_entradaestoque_em_lotes(supabase, tamanho_lote=10000)
df_entradaestoque.rename(columns={'ID':'idEstoque'}, inplace=True)
# Mostrar tipos de colunas
print(f"Formato das colunas antes:\n{df_entradaestoque.dtypes}")
# Trocar tipo de coluna: qntConsumida e valorUnitario -> float, MesReferencia -> Datetime
# Transformar mes em date
df_entradaestoque['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_entradaestoque['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)
# Criar coluna de primeira data do mês
df_entradaestoque['priDiaMes'] = df_entradaestoque['mesReferencia'].dt.to_period('M').dt.to_timestamp()

# Mesclar com IGPDI
df_entradaestoque = df_entradaestoque.merge(df_igpdi[['data','deflator']],left_on='priDiaMes',right_on='data',how='left')
# Deflacionar
df_entradaestoque['def_valorUnitario'] = df_entradaestoque['valorUnitario']/df_entradaestoque['deflator']
df_entradaestoque['def_valorUnitario'] = df_entradaestoque['def_valorUnitario'].fillna(0)

# Trazer a tabela de Entrada de Estoque apenas com a tela de culturas plantadas
df_entradaestoque_etl = df_entradaestoque.copy()
# Mesclar pelo idFazenda, idRelacionamento
df_entradaestoque_etl = (
    df_entradaestoque_etl.merge(df_producao_etl[['idFazenda','idProducao','tipo','custoUnitarioForrageira']],
                                       left_on=['idFazenda','idRelacionamento'],
                                       right_on=['idFazenda','idProducao'],
                                       how='left')
)

# Substituir valor unitário das culturas plantadas pela de forrageira
df_entradaestoque_etl.loc[df_entradaestoque_etl['Tela'] == 'CulturasPlantadas', 'def_valorUnitario'] = df_entradaestoque_etl['custoUnitarioForrageira']

# Deletar as tabelas original 
del df_entradaestoque
gc.collect()

# Mostrar tipos de colunas
print(f"Formato das colunas depois:\n{df_entradaestoque_etl.dtypes}")
# Mostrar tabela
print(df_entradaestoque_etl.shape)
#df_entradaestoque_etl.head()

Extraídas 10000 linhas...
Extraídas 20000 linhas...
Extraídas 30000 linhas...
Extraídas 40000 linhas...
Extraídas 50000 linhas...
Extraídas 60000 linhas...
Extraídas 70000 linhas...
Extraídas 79679 linhas...
Formato das colunas antes:
idFazenda           float64
idEstoque             int64
Tela                 object
idRelacionamento    float64
idItem              float64
qntComprada         float64
qntConsumida        float64
valorUnitario       float64
Excluido              int64
mesReferencia        object
dtype: object
Formato das colunas depois:
idFazenda                         float64
idEstoque                           int64
Tela                               object
idRelacionamento                  float64
idItem                            float64
qntComprada                       float64
qntConsumida                      float64
valorUnitario                     float64
Excluido                            int64
mesReferencia              datetime64[ns]
priDiaMes              

## Repassar o Valor Unitário para Saída de Estoque

In [29]:
# Saída de Estoque
tab_saidaestoque = (
    supabase.table("tab_saidaestoque")
    .select('idFazenda', 'ID','Tela','idRelacionamento', 'idItem','qntConsumida','valorUnitario',
            'idEstoque','Excluido','mesReferencia')
    .eq("Excluido", 0)
    .execute()
)
df_saidaestoque = pd.DataFrame(tab_saidaestoque.data)
# Mostrar tipos de colunas
print(f"Formato das colunas antes:\n{df_saidaestoque.dtypes}")
# Trocar tipo de coluna: qntConsumida e valorUnitario -> float, MesReferencia -> Datetime
# Transformar mes em date
df_saidaestoque['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_saidaestoque['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# Trazer o valor deflacionar do valor unitário
df_saidaestoque = df_saidaestoque.merge(df_entradaestoque_etl[['idEstoque','def_valorUnitario']],on='idEstoque',how='left')

# Mostrar tipos de colunas
print(f"Formato das colunas depois:\n{df_saidaestoque.dtypes}")

# # Deletar tabelas original 
# del tab_saidaestoque, df_entradaestoque_etl
gc.collect()

# Mostrar tabela
#df_saidaestoque.head()

Formato das colunas antes:
idFazenda           float64
ID                    int64
Tela                 object
idRelacionamento    float64
idItem              float64
qntConsumida        float64
valorUnitario       float64
idEstoque           float64
Excluido              int64
mesReferencia        object
dtype: object
Formato das colunas depois:
idFazenda                   float64
ID                            int64
Tela                         object
idRelacionamento            float64
idItem                      float64
qntConsumida                float64
valorUnitario               float64
idEstoque                   float64
Excluido                      int64
mesReferencia        datetime64[ns]
def_valorUnitario           float64
dtype: object


50

## Alimentação

### Sharepoint

In [30]:
# Alimentação
tab_alimentacao = (
    supabase.table('tab_alimentacao')
    .select('ID','idFazenda','mesReferencia','categoria','item','id_item',
            'operacao','quantidadeComprada','quantidadeConsumida','valorUnitario', 'Excluido')
    .eq('Excluido',0)
    .execute()
)
# Importar tabela de alimentação
df_alimentacao_shp = pd.DataFrame(tab_alimentacao.data)
# Mostrar tipo das colunas
print(f"Tipo das colunas antes:\n{df_alimentacao_shp.dtypes}")
df_alimentacao_shp['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_alimentacao_shp['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# Alimentação sem custo unitário = 0
df_alimentacao_a = df_alimentacao_shp.loc[df_alimentacao_shp['valorUnitario']!=0].copy()
# Fazer o merge com df_idpgi
df_alimentacao_a = df_alimentacao_a.merge(df_igpdi[['data','deflator']], left_on='mesReferencia', right_on='data', how='left')
# Deflacionar
df_alimentacao_a['def_valorUnitario'] = df_alimentacao_a['valorUnitario'] / df_alimentacao_a['deflator']

# Deletar tabela original 
del tab_alimentacao
gc.collect()
#Mostrar tipo das colunas
print(f"Tipo das colunas epois:\n{df_alimentacao_a.dtypes}")
#df_alimentacao_a.head()

Tipo das colunas antes:
ID                       int64
idFazenda                int64
mesReferencia           object
categoria               object
item                    object
id_item                float64
operacao                object
quantidadeComprada     float64
quantidadeConsumida    float64
valorUnitario          float64
Excluido                 int64
dtype: object
Tipo das colunas epois:
ID                              int64
idFazenda                       int64
mesReferencia          datetime64[ns]
categoria                      object
item                           object
id_item                       float64
operacao                       object
quantidadeComprada            float64
quantidadeConsumida           float64
valorUnitario                 float64
Excluido                        int64
data                   datetime64[ns]
deflator                      float64
def_valorUnitario             float64
dtype: object


### Excel

In [31]:
# Alimentação
tab_alimentacao_excel = (
    supabase.table('xlsx_alimentacao')
    .select('idFazenda','mesReferencia','categoria','item',
            'operacao','quantidadeComprada','quantidadeConsumida','valorUnitario')
    .execute()
)
# Importar tabela de alimentação
df_alimentacao_excel = pd.DataFrame(tab_alimentacao_excel.data)
# Mostrar tipo das colunas
print(f"Tipo das colunas antes:\n{df_alimentacao_excel.dtypes}")
df_alimentacao_excel['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_alimentacao_excel['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# Alimentação sem custo unitário = 0
df_alimentacao_excel = df_alimentacao_excel.loc[df_alimentacao_excel['valorUnitario']!=0].copy()
# Fazer o merge com df_idpgi
df_alimentacao_excel = df_alimentacao_excel.merge(df_igpdi[['data','deflator']], left_on='mesReferencia', right_on='data', how='left')
# Deflacionar
df_alimentacao_excel['def_valorUnitario'] = df_alimentacao_excel['valorUnitario'] / df_alimentacao_excel['deflator']

# Criar coluna de Custo com Alimentação
df_alimentacao_excel['custoAlimentacao'] = df_alimentacao_excel['def_valorUnitario'] * df_alimentacao_excel['quantidadeConsumida']

# =============== ETAPA 6 - CRIAR CUSTO DE ALIMENTAÇÃO ===========================
# Criar custo com Concentrado e Volumoso
df_custo_alimentacao_excel = df_alimentacao_excel.loc[df_alimentacao_excel['operacao']!='Estocar', ['idFazenda','mesReferencia','categoria','custoAlimentacao','quantidadeConsumida']].copy()
df_custo_alimentacao_excel = pd.pivot_table(
    df_custo_alimentacao_excel,
    index=['idFazenda', 'mesReferencia'],
    columns='categoria',
    values=['custoAlimentacao','quantidadeConsumida'],
    aggfunc='sum'
).reset_index()
df_custo_alimentacao_excel.columns = ['_'.join([str(i) for i in col if i]) for col in df_custo_alimentacao_excel.columns]  # Para "achatar" o multiindex das colunas
# Trocar NaN por 0
colunas_alimentacao = ['custoAlimentacao_Concentrado','custoAlimentacao_Minerais','custoAlimentacao_Volumoso']
for col in colunas_alimentacao:
    df_custo_alimentacao_excel[col] = df_custo_alimentacao_excel[col].fillna(0)
# Criar agregado de concentrado + minerais
df_custo_alimentacao_excel['custoAlimentacao_Concentrado_Minerais'] = df_custo_alimentacao_excel[['custoAlimentacao_Concentrado','custoAlimentacao_Minerais']].sum(axis=1)

# Mostrar tipo das colunas
print(f"Tipo das colunas epois:\n{df_alimentacao_excel.dtypes}")
#df_custo_alimentacao_excel.head()

Tipo das colunas antes:
idFazenda                int64
mesReferencia           object
categoria               object
item                    object
operacao                object
quantidadeComprada     float64
quantidadeConsumida    float64
valorUnitario          float64
dtype: object
Tipo das colunas epois:
idFazenda                       int64
mesReferencia          datetime64[ns]
categoria                      object
item                           object
operacao                       object
quantidadeComprada            float64
quantidadeConsumida           float64
valorUnitario                 float64
data                   datetime64[ns]
deflator                      float64
def_valorUnitario             float64
custoAlimentacao              float64
dtype: object


## Repassar o valor da Saída de Estoque para Alimentação

In [32]:
# =============== ETAPA 5 - ALIMENTAÇÃO ===========================
# Trazer a tabela de alimentacao
df_alimentacao_b = df_alimentacao_shp.loc[df_alimentacao_shp['valorUnitario']==0].copy()
# Criar cópia da saída de estoque
df_saidaestoque_etl = df_saidaestoque.copy()
# Filtrar saida de estoque para alimentacao
df_saida_alimentacao = df_saidaestoque_etl.loc[df_saidaestoque_etl['Tela']=='Alimentação']
df_alimentacao_b = (
    df_alimentacao_b.merge(df_saida_alimentacao[['idRelacionamento','def_valorUnitario']],
                             left_on='ID',
                             right_on='idRelacionamento',
                             how='left')
)
# Dropar coluna do relacionamento do estoque do DF alimentacao B
df_alimentacao_b = df_alimentacao_b.drop(columns=['idRelacionamento'])
# Juntar Dataframe de Alimentação A + B
df_alimentacao_etl = pd.concat([df_alimentacao_a, df_alimentacao_b])
# Criar coluna de Custo com Alimentação
df_alimentacao_etl['custoAlimentacao'] = df_alimentacao_etl['def_valorUnitario'] * df_alimentacao_etl['quantidadeConsumida']

# Deletar tabelas A e B 
del df_alimentacao_a, df_alimentacao_b
gc.collect()

#df_alimentacao_etl.head()

29

## Criar custo com alimentação para usar no COE

In [33]:
# =============== ETAPA 6 - CRIAR CUSTO DE ALIMENTAÇÃO ===========================
# Criar custo com Concentrado e Volumoso
df_custo_alimentacao_shp = df_alimentacao_etl.loc[df_alimentacao_etl['operacao']!='Estocar', ['idFazenda','mesReferencia','categoria','custoAlimentacao','quantidadeConsumida']].copy()
df_custo_alimentacao_shp = pd.pivot_table(
    df_custo_alimentacao_shp,
    index=['idFazenda', 'mesReferencia'],
    columns='categoria',
    values=['custoAlimentacao','quantidadeConsumida'],
    aggfunc='sum'
).reset_index()
df_custo_alimentacao_shp.columns = ['_'.join([str(i) for i in col if i]) for col in df_custo_alimentacao_shp.columns]  # Para "achatar" o multiindex das colunas
# Trocar NaN por 0
colunas_alimentacao = ['custoAlimentacao_Concentrado','custoAlimentacao_Minerais','custoAlimentacao_Volumoso']
for col in colunas_alimentacao:
    df_custo_alimentacao_shp[col] = df_custo_alimentacao_shp[col].fillna(0)
# Criar agregado de concentrado + minerais
df_custo_alimentacao_shp['custoAlimentacao_Concentrado_Minerais'] = df_custo_alimentacao_shp[['custoAlimentacao_Concentrado','custoAlimentacao_Minerais']].sum(axis=1)

# Deletar tabela ETL de Alimentação 
del df_alimentacao_etl
gc.collect() 

#df_custo_alimentacao_shp.head()

0

## Unificar alimentação

In [34]:
df_custo_alimentacao = pd.concat([df_custo_alimentacao_excel, df_custo_alimentacao_shp])

#df_custo_alimentacao.head()

## Custo da Venda de Volumosos/Concentrados

In [35]:
receita_forrageira = ['ReceitasVolumoso','ReceitasConcentrado']
# Cria df para custo da receita
df_receitaforrageira = df_saidaestoque.loc[df_saidaestoque['Tela'].isin(receita_forrageira)].copy() # Os custos de venda de alimentação só estão na saída de estoque.
# Criar coluna de Custo da Receita 
df_receitaforrageira.loc[:,'custoReceita'] = df_receitaforrageira['qntConsumida'] * df_receitaforrageira['def_valorUnitario']
# Criar a coluna de custo de Concentrado e Volumoso separamente e agregar o Custo

df_receitaforrageira = df_receitaforrageira.pivot_table(
    index=['idFazenda', 'mesReferencia'], # Agregando por par de Fazenda e Data
    columns='Tela', # Sobre a Tela
    values='custoReceita', 
    aggfunc='sum'
).reset_index()
# Renomear as colunas
df_receitaforrageira.columns = ['idFazenda','mesReferencia','custoReceitasConcentrado','custoReceitasVolumoso']
# Preencher NaN com 0
for col in ['custoReceitasConcentrado','custoReceitasVolumoso']:
    df_receitaforrageira[col] = df_receitaforrageira[col].fillna(0)

# Mostrar tabela final
#df_receitaforrageira.head()

## Energia e Combustível

### Sharepoint

In [36]:
# Energia e Combustível
tab_energiacombustivel = (
    supabase.table('tab_energiacombustivel')
    .select('idFazenda','mesReferencia','qtdeEnergia','valorEnergia','qtdeEnergiaSolar','valorEnergiaSolar',
            'qtdeAlcoolgasolina','valorAlcoolgasolina','qtdeDiesel','valorDiesel')
    .execute()
)
# Criar dataframe de energia e combustível
df_energiacombustivel_shp = pd.DataFrame(tab_energiacombustivel.data)
# Mostrar tipo dos dados
print(f"Tipo dos dados antes do ETL:\n{df_energiacombustivel_shp.dtypes}")
# Transformar mesReferencia em Datetime
df_energiacombustivel_shp['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_energiacombustivel_shp['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)
# Trazer o IGPDI para deflacionar
df_energiacombustivel_shp = (
    df_energiacombustivel_shp.merge(df_igpdi[['data','deflator']],
                                left_on='mesReferencia',
                                right_on='data',
                                how='left')
)

# Definir colunas a deflacionar
colunas_deflacionar = ['valorEnergia','valorEnergiaSolar','valorAlcoolgasolina','valorDiesel']
# Loop para deflacionar
for col in colunas_deflacionar:
    coluna_deflacionada = 'def_'+col
    df_energiacombustivel_shp[coluna_deflacionada] = df_energiacombustivel_shp[col]/df_energiacombustivel_shp['deflator']

# Criar coluna de valor de energia
df_energiacombustivel_shp['def_custoEnergia'] = df_energiacombustivel_shp[['def_valorEnergia', 'def_valorEnergiaSolar']].sum(axis=1)
# Criar coluna de valor de combustível
df_energiacombustivel_shp['def_custoCombustivel'] = df_energiacombustivel_shp[['def_valorAlcoolgasolina', 'def_valorDiesel']].sum(axis=1)

# Deletar tabela original
del tab_energiacombustivel

print(f"\nTipo dos dados após ETL:\n{df_energiacombustivel_shp.dtypes}\n")
#df_energiacombustivel_shp.head()

Tipo dos dados antes do ETL:
idFazenda                int64
mesReferencia           object
qtdeEnergia            float64
valorEnergia           float64
qtdeEnergiaSolar       float64
valorEnergiaSolar      float64
qtdeAlcoolgasolina     float64
valorAlcoolgasolina    float64
qtdeDiesel             float64
valorDiesel            float64
dtype: object

Tipo dos dados após ETL:
idFazenda                           int64
mesReferencia              datetime64[ns]
qtdeEnergia                       float64
valorEnergia                      float64
qtdeEnergiaSolar                  float64
valorEnergiaSolar                 float64
qtdeAlcoolgasolina                float64
valorAlcoolgasolina               float64
qtdeDiesel                        float64
valorDiesel                       float64
data                       datetime64[ns]
deflator                          float64
def_valorEnergia                  float64
def_valorEnergiaSolar             float64
def_valorAlcoolgasolina          

### Excel

In [37]:
# Energia e Combustível
tab_energiacombustivel_excel = (
    supabase.table('xlsx_energia')
    .select('idFazenda','mesReferencia','qtdeEnergia','valorEnergia',
            'qtdeAlcoolgasolina','valorAlcoolgasolina','qtdeDiesel','valorDiesel')
    .execute()
)
# Criar dataframe de energia e combustível
df_energiacombustivel_excel = pd.DataFrame(tab_energiacombustivel_excel.data)
# Mostrar tipo dos dados
print(f"Tipo dos dados antes do ETL:\n{df_energiacombustivel_excel.dtypes}")

# Inserir colunas criadas depois
df_energiacombustivel_excel['qtdeEnergiaSolar'] = 0
df_energiacombustivel_excel['valorEnergiaSolar'] = 0

# Transformar mesReferencia em Datetime
df_energiacombustivel_excel['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_energiacombustivel_excel['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)
# Trazer o IGPDI para deflacionar
df_energiacombustivel_excel = (
    df_energiacombustivel_excel.merge(df_igpdi[['data','deflator']],
                                left_on='mesReferencia',
                                right_on='data',
                                how='left')
)

# Definir colunas a deflacionar
colunas_deflacionar = ['valorEnergia','valorEnergiaSolar','valorAlcoolgasolina','valorDiesel']
# Loop para deflacionar
for col in colunas_deflacionar:
    coluna_deflacionada = 'def_'+col
    df_energiacombustivel_excel[coluna_deflacionada] = df_energiacombustivel_excel[col]/df_energiacombustivel_excel['deflator']

# Criar coluna de valor de energia
df_energiacombustivel_excel['def_custoEnergia'] = df_energiacombustivel_excel[['def_valorEnergia', 'def_valorEnergiaSolar']].sum(axis=1)
# Criar coluna de valor de combustível
df_energiacombustivel_excel['def_custoCombustivel'] = df_energiacombustivel_excel[['def_valorAlcoolgasolina', 'def_valorDiesel']].sum(axis=1)

# Deletar tabela original
del tab_energiacombustivel_excel

print(f"\nTipo dos dados após ETL:\n{df_energiacombustivel_excel.dtypes}\n")
#df_energiacombustivel_excel.head()

Tipo dos dados antes do ETL:
idFazenda                int64
mesReferencia           object
qtdeEnergia            float64
valorEnergia           float64
qtdeAlcoolgasolina     float64
valorAlcoolgasolina    float64
qtdeDiesel             float64
valorDiesel            float64
dtype: object

Tipo dos dados após ETL:
idFazenda                           int64
mesReferencia              datetime64[ns]
qtdeEnergia                       float64
valorEnergia                      float64
qtdeAlcoolgasolina                float64
valorAlcoolgasolina               float64
qtdeDiesel                        float64
valorDiesel                       float64
qtdeEnergiaSolar                    int64
valorEnergiaSolar                   int64
data                       datetime64[ns]
deflator                          float64
def_valorEnergia                  float64
def_valorEnergiaSolar             float64
def_valorAlcoolgasolina           float64
def_valorDiesel                   float64
def_custoEn

### Unificar

In [38]:
df_energiacombustivel = pd.concat([df_energiacombustivel_excel, df_energiacombustivel_shp])

#df_energiacombustivel.head()

## Mão de Obra

### Sharepoint

In [39]:
# Mão de Obra
tab_maodeobra = (
    supabase.table('tab_maodeobra')
    .select('idFazenda','mesReferencia','familiarQtd','familiarValortotal','contratataordenhadorQtd','contratataordenhadorValortotal',
            'ctdServicosgeraisQtd','ctdServicosgeraisValor','ctdFolguistaQtde','ctdFolguistaValor','ctdTratoristaQtde','ctdTratoristaValor',
            'encargostrabalhistasValor','demaisdespesasValor')
    .execute()
)
# Criar Dataframe
df_mdo_shp = pd.DataFrame(tab_maodeobra.data)
# Mostrar tipo das colunas
print(f"Tipo das colunas antes do ETL:\n{df_mdo_shp.dtypes}")
# Transformar mesReferencia em Datetime
df_mdo_shp['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_mdo_shp['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)
# Há valores duplicados na base. É preciso retirar 
df_mdo_shp['ordem_par'] = (
    df_mdo_shp.groupby(['idFazenda', 'mesReferencia'])
    .cumcount() + 1  # começa em 0, soma 1 pra começar em 1
)# Conta o número de valores duplicados
# Filtra os valores 
df_mdo_shp = df_mdo_shp[df_mdo_shp['ordem_par'] == 1]

# Definir colunas para transformar para numérico
colunas_alterar = ['familiarQtd', 'familiarValortotal','contratataordenhadorQtd', 'contratataordenhadorValortotal',
                   'ctdServicosgeraisQtd', 'ctdServicosgeraisValor','ctdFolguistaQtde','ctdFolguistaValor', 'ctdTratoristaQtde',
                   'ctdTratoristaValor','encargostrabalhistasValor', 'demaisdespesasValor']

#Aplicar função para transformar em numérico
df_mdo_shp = tratar_colunas_numericas(df_mdo_shp, colunas_alterar)
# Colunas a deflacionar 
colunas_deflacionar = ['familiarValortotal', 'contratataordenhadorValortotal', 'ctdServicosgeraisValor', 'ctdFolguistaValor', 'ctdTratoristaValor',
                       'encargostrabalhistasValor', 'demaisdespesasValor']
# Fazer merge com idpgi 
df_mdo_shp = df_mdo_shp.merge(df_igpdi[['data','deflator']], left_on='mesReferencia', right_on='data', how='left')
df_mdo_shp = deflacionar_colunas(df_mdo_shp, colunas_deflacionar)
# Definir colunas a somar
colunas_soma = ['def_contratataordenhadorValortotal','def_ctdServicosgeraisValor','def_ctdFolguistaValor',
                'def_ctdTratoristaValor','def_encargostrabalhistasValor', 'def_demaisdespesasValor']
# Criar coluna de Custo de mão de obra Familiar
df_mdo_shp['custoMDOContratada'] = df_mdo_shp.loc[:, colunas_soma].sum(axis=1)
# Definir colunas de quantidade a somar
colunas_quantidade = ['contratataordenhadorQtd','ctdServicosgeraisQtd','ctdFolguistaQtde','ctdTratoristaQtde']
# Criar coluna de quantidade de mão de obra Contratada
df_mdo_shp['qtdMDOContratada'] = df_mdo_shp.loc[:, colunas_quantidade].sum(axis=1)
# Criar coluna de MDO Total
df_mdo_shp['qtdMDOTotal'] = df_mdo_shp[['familiarQtd', 'qtdMDOContratada']].sum(axis=1)

# Encontrar mão de obra diária 
df_mdo_shp['MDOContratadaDiaria'] = df_mdo_shp['qtdMDOContratada'] / 30.42
df_mdo_shp['MDOTotalDiaria'] = df_mdo_shp['qtdMDOTotal'] / 30.42
df_mdo_shp['MDOFamiliarDiaria'] = df_mdo_shp['familiarQtd'] / 30.42

# # Deletar tabela original 
# del tab_maodeobra

# Mostrar tipo das colunas no fim do tratamento
print(f"\nTipo das colunas depois do ETL:\n{df_mdo_shp.dtypes}")
# Mostrar a tabela
df_mdo_shp = df_mdo_shp.sort_values(['idFazenda','mesReferencia']).reset_index(drop=True)
#df_mdo_shp.loc[df_mdo_shp['idFazenda']==99]

Tipo das colunas antes do ETL:
idFazenda                           int64
mesReferencia                      object
familiarQtd                       float64
familiarValortotal                float64
contratataordenhadorQtd           float64
contratataordenhadorValortotal    float64
ctdServicosgeraisQtd              float64
ctdServicosgeraisValor            float64
ctdFolguistaQtde                  float64
ctdFolguistaValor                 float64
ctdTratoristaQtde                 float64
ctdTratoristaValor                float64
encargostrabalhistasValor         float64
demaisdespesasValor               float64
dtype: object

Tipo das colunas depois do ETL:
idFazenda                                      int64
mesReferencia                         datetime64[ns]
familiarQtd                                  float64
familiarValortotal                           float64
contratataordenhadorQtd                      float64
contratataordenhadorValortotal               float64
ctdServicosgerai

### Excel

In [40]:
# Mão de Obra
tab_maodeobra = (
    supabase.table('xlsx_mdo')
    .select('idFazenda','mesReferencia','familiarQtd','familiarValortotal','contratataordenhadorQtd','contratataordenhadorValortotal',
            'ctdServicosgeraisQtd','ctdServicosgeraisValor','ctdFolguistaQtde','ctdFolguistaValor','ctdTratoristaQtde','ctdTratoristaValor',
            'encargostrabalhistasValor','demaisdespesasValor')
    .execute()
)
# Criar Dataframe
df_mdo_excel = pd.DataFrame(tab_maodeobra.data)
# Mostrar tipo das colunas
print(f"Tipo das colunas antes do ETL:\n{df_mdo_excel.dtypes}")
# Transformar mesReferencia em Datetime
df_mdo_excel['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_mdo_excel['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)
# Há valores duplicados na base. É preciso retirar 
df_mdo_excel['ordem_par'] = (
    df_mdo_excel.groupby(['idFazenda', 'mesReferencia'])
    .cumcount() + 1  # começa em 0, soma 1 pra começar em 1
)# Conta o número de valores duplicados
# Filtra os valores 
df_mdo_excel = df_mdo_excel[df_mdo_excel['ordem_par'] == 1]

# Definir colunas para transformar para numérico
colunas_alterar = ['familiarQtd', 'familiarValortotal','contratataordenhadorQtd', 'contratataordenhadorValortotal',
                   'ctdServicosgeraisQtd', 'ctdServicosgeraisValor','ctdFolguistaQtde','ctdFolguistaValor', 'ctdTratoristaQtde',
                   'ctdTratoristaValor','encargostrabalhistasValor', 'demaisdespesasValor']

#Aplicar função para transformar em numérico
df_mdo_excel = tratar_colunas_numericas(df_mdo_excel, colunas_alterar)
# Colunas a deflacionar 
colunas_deflacionar = ['familiarValortotal', 'contratataordenhadorValortotal', 'ctdServicosgeraisValor', 'ctdFolguistaValor', 'ctdTratoristaValor',
                       'encargostrabalhistasValor', 'demaisdespesasValor']
# Fazer merge com idpgi 
df_mdo_excel = df_mdo_excel.merge(df_igpdi[['data','deflator']], left_on='mesReferencia', right_on='data', how='left')
df_mdo_excel = deflacionar_colunas(df_mdo_excel, colunas_deflacionar)
# Definir colunas a somar
colunas_soma = ['def_contratataordenhadorValortotal','def_ctdServicosgeraisValor','def_ctdFolguistaValor',
                'def_ctdTratoristaValor','def_encargostrabalhistasValor', 'def_demaisdespesasValor']
# Criar coluna de Custo de mão de obra Familiar
df_mdo_excel['custoMDOContratada'] = df_mdo_excel.loc[:, colunas_soma].sum(axis=1)
# Definir colunas de quantidade a somar
colunas_quantidade = ['contratataordenhadorQtd','ctdServicosgeraisQtd','ctdFolguistaQtde','ctdTratoristaQtde']
# Criar coluna de quantidade de mão de obra Contratada
df_mdo_excel['qtdMDOContratada'] = df_mdo_excel.loc[:, colunas_quantidade].sum(axis=1)
# Criar coluna de MDO Total
df_mdo_excel['qtdMDOTotal'] = df_mdo_excel[['familiarQtd', 'qtdMDOContratada']].sum(axis=1)

# Encontrar mão de obra diária 
df_mdo_excel['MDOContratadaDiaria'] = df_mdo_excel['qtdMDOContratada'] / 30.42
df_mdo_excel['MDOTotalDiaria'] = df_mdo_excel['qtdMDOTotal'] / 30.42
df_mdo_excel['MDOFamiliarDiaria'] = df_mdo_excel['familiarQtd'] / 30.42

# # Deletar tabela original 
# del tab_maodeobra

# Mostrar tipo das colunas no fim do tratamento
print(f"\nTipo das colunas depois do ETL:\n{df_mdo_excel.dtypes}")
# Mostrar a tabela
df_mdo_excel = df_mdo_excel.sort_values(['idFazenda','mesReferencia']).reset_index(drop=True)
#df_mdo_excel.loc[df_mdo_excel['idFazenda']==99]

Tipo das colunas antes do ETL:
idFazenda                           int64
mesReferencia                      object
familiarQtd                       float64
familiarValortotal                float64
contratataordenhadorQtd           float64
contratataordenhadorValortotal    float64
ctdServicosgeraisQtd              float64
ctdServicosgeraisValor            float64
ctdFolguistaQtde                  float64
ctdFolguistaValor                 float64
ctdTratoristaQtde                 float64
ctdTratoristaValor                float64
encargostrabalhistasValor         float64
demaisdespesasValor               float64
dtype: object

Tipo das colunas depois do ETL:
idFazenda                                      int64
mesReferencia                         datetime64[ns]
familiarQtd                                  float64
familiarValortotal                           float64
contratataordenhadorQtd                      float64
contratataordenhadorValortotal               float64
ctdServicosgerai

### Unificar

In [41]:
# Unir sharepoint com excel
df_mdo = pd.concat([df_mdo_excel, df_mdo_shp])
# Visualizar
#df_mdo.head()

## Componentes Custos
1. Deflacionar todos os valores monetários
2. Importar os consumos de leite
3. Importar custo com alimentação
4. Importar custo com energia
5. Importar custo com receita
6. Importar custo com mão de obra
7. Calcular o COE

### Sharepoint

In [42]:
# Componentes Custos Usada em Leite Consumido e Componentes Custos
tab_componentescustos = (
    supabase.table("tab_componentescusto")
    .select('ID','idFazenda', 'mesReferencia', 'consumoLeiteBezerro', 'consumoMaoObraFamiliar',
           'consumoMaoObraContratada', 'consumoLeiteDescartado', 'gastoSucedaneo',
           'gastoMaterialOrdenha', 'gastoReproducao', 'gastoHormonios', 'gastoMedicamentosVacinas',
           'gastoAssistenciaTecnica', 'gastoImpostoTaxas', 'gastoArrendamento', 'gastoReparosConsertos',
           'gastoAdministrativo', 'gastoAcessoriosDespesasGerais', 'gastoCamaAreia',
            'gastoEnergiaEletrica', 'gastoCombustivel', 'compra_Animais', 'compra_Terras', 'gastoEmprestimosJurosPagos')
    .execute()
)
# Importar dataframe
df_componentescustos_shp = pd.DataFrame(tab_componentescustos.data)

# # Definir colunas de parcelas para transformar em int
# colunas_custo = ['ID','consumoLeiteBezerro', 'consumoMaoObraFamiliar',
#                     'consumoMaoObraContratada','consumoLeiteDescartado','gastoSucedaneo',
#                     'gastoMaterialOrdenha', 'gastoReproducao','gastoHormonios',
#                     'gastoMedicamentosVacinas','gastoAssistenciaTecnica','gastoImpostoTaxas',
#                     'gastoArrendamento','gastoReparosConsertos','gastoAdministrativo',
#                     'gastoAcessoriosDespesasGerais','gastoCamaAreia','compra_Animais','compra_Terras','gastoEmprestimosJurosPagos']
# # Iterar para definir formatos
# for col in colunas_custo:
#     df_componentescustos_shp[col] = df_componentescustos_shp[col].str.replace('.', '')
#     df_componentescustos_shp[col] = df_componentescustos_shp[col].str.replace(',', '.')
#     # Transformar qntconsumida
#     df_componentescustos_shp[col] = pd.to_numeric(df_componentescustos_shp[col], errors='coerce')
#     df_componentescustos_shp[col] = df_componentescustos_shp[col].fillna(0)

# Transformar data em datetime
df_componentescustos_shp['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_componentescustos_shp['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# Trazer o preço do leite
df_componentescustos_shp = df_componentescustos_shp.merge(df_rendabruta[['mesReferencia', 'idFazenda', 'def_precoLeite']],
                                                 on=['idFazenda', 'mesReferencia'], how='left')

# Deflacionar os dados
df_componentescustos_shp = df_componentescustos_shp.merge(df_igpdi[['data','deflator']],left_on='mesReferencia',right_on='data', how='left')

# Colunas a deflacionar
colunas_deflacionar = ['gastoSucedaneo','gastoMaterialOrdenha', 'gastoReproducao','gastoHormonios','gastoMedicamentosVacinas',
                       'gastoAssistenciaTecnica','gastoImpostoTaxas','gastoArrendamento','gastoReparosConsertos','gastoAdministrativo',
                       'gastoAcessoriosDespesasGerais','gastoCamaAreia','compra_Animais',
                       'compra_Terras','gastoEmprestimosJurosPagos']

# Deflacionar as colunas
for variable in colunas_deflacionar:
    nome_var = 'def_'+variable
    df_componentescustos_shp[nome_var] = df_componentescustos_shp[variable]/df_componentescustos_shp['deflator']

# =============== CONSUMO DE LEITE ===========================
# Criar custo com consumo de leite
colunas_consumo_leite = ['consumoLeiteBezerro', 'consumoMaoObraFamiliar', 'consumoMaoObraContratada', 'consumoLeiteDescartado']
for consumo in colunas_consumo_leite:
    gasto_consumo = 'gasto'+consumo
    df_componentescustos_shp[gasto_consumo] = df_componentescustos_shp[consumo]*df_componentescustos_shp['def_precoLeite']

# =============== ALIMENTAÇÃO ===========================
# Importar os custos com alimentação
df_componentescustos_shp = (
    df_componentescustos_shp.merge(df_custo_alimentacao[['idFazenda','mesReferencia',
                                                     'custoAlimentacao_Concentrado_Minerais','custoAlimentacao_Volumoso']],
                               on=['idFazenda','mesReferencia'],
                               how='left')
)

# =============== ENERGIA E COMBUSTÍVEL ===========================
df_energiacombustivel_etl = df_energiacombustivel[['idFazenda','mesReferencia','def_custoEnergia','def_custoCombustivel']].copy()
# Mesclar com componentes de custo
df_componentescustos_shp = df_componentescustos_shp.merge(df_energiacombustivel_etl, on=['idFazenda','mesReferencia'],how='left')

# =============== CUSTO COM RECEITA ===========================
df_componentescustos_shp = df_componentescustos_shp.merge(df_receitaforrageira, on=['idFazenda','mesReferencia'],how='left')

# =============== MÃO DE OBRA ===========================
df_componentescustos_shp = df_componentescustos_shp.merge(df_mdo[['idFazenda','mesReferencia','custoMDOContratada']],on=['idFazenda','mesReferencia'],how='left')

# =============== Inserir Leite Consumido nos Custos específicos ==============================
# Custo da MDO Contradada = Custo da MDO + Custo do Leite Consumido da MDO Contratada
df_componentescustos_shp['custoMDOContratada'] = df_componentescustos_shp[['custoMDOContratada', 'gastoconsumoMaoObraContratada']].sum(axis=1)
# Custo com Qualidade do Leite = Custo com Qualidade do leite + Custo do Leite Descartado
df_componentescustos_shp['def_gastoMaterialOrdenha'] = df_componentescustos_shp[['def_gastoMaterialOrdenha', 'gastoconsumoLeiteDescartado']].sum(axis=1)

# =============== COE ===========================
# Lista de colunas a serem somadas
colunas_coe = [
    'def_gastoSucedaneo',
    'def_gastoMaterialOrdenha',
    'def_gastoReproducao',
    'def_gastoHormonios',
    'def_gastoMedicamentosVacinas',
    'def_gastoAssistenciaTecnica',
    'def_gastoImpostoTaxas',
    'def_gastoArrendamento',
    'def_gastoReparosConsertos',
    'def_gastoAdministrativo',
    'def_gastoAcessoriosDespesasGerais',
    'def_gastoCamaAreia',
    'gastoconsumoLeiteBezerro',
    'custoAlimentacao_Concentrado_Minerais',
    'custoAlimentacao_Volumoso',
    'def_custoEnergia',
    'def_custoCombustivel',
    'custoReceitasConcentrado',
    'custoReceitasVolumoso',
    'custoMDOContratada'
]

# Criar uma nova coluna com a soma de todas as colunas listadas por linha
df_componentescustos_shp['coe'] = df_componentescustos_shp[colunas_coe].sum(axis=1)
 
df_componentescustos_shp = df_componentescustos_shp[~df_componentescustos_shp.duplicated(subset=['idFazenda', 'mesReferencia'], keep='first')].sort_values(['idFazenda','mesReferencia'])

# Mostrar colunas e tipos
print(df_componentescustos_shp.dtypes)

df_componentescustos_shp.to_excel('teste_coe.xlsx', index=False)

ID                                                int64
idFazenda                                         int64
mesReferencia                            datetime64[ns]
consumoLeiteBezerro                             float64
consumoMaoObraFamiliar                          float64
consumoMaoObraContratada                        float64
consumoLeiteDescartado                          float64
gastoSucedaneo                                  float64
gastoMaterialOrdenha                            float64
gastoReproducao                                 float64
gastoHormonios                                  float64
gastoMedicamentosVacinas                        float64
gastoAssistenciaTecnica                         float64
gastoImpostoTaxas                               float64
gastoArrendamento                               float64
gastoReparosConsertos                           float64
gastoAdministrativo                             float64
gastoAcessoriosDespesasGerais                   

### Excel

In [43]:
# Componentes Custos Usada em Leite Consumido e Componentes Custos
tab_componentescustos_excel = (
    supabase.table("xlsx_custo")
    .select('idCusto','idFazenda', 'mesReferencia', 'consumoLeiteBezerro', 'consumoMaoObraFamiliar',
           'consumoMaoObraContratada', 'consumoLeiteDescartado', 'gastoSucedaneo',
           'gastoMaterialOrdenha', 'gastoReproducao', 'gastoHormonios', 'gastoMedicamentosVacinas',
           'gastoAssistenciaTecnica', 'gastoImpostoTaxas', 'gastoArrendamento', 'gastoReparosConsertos',
           'gastoAdministrativo', 'gastoAcessoriosDespesasGerais', 'gastoCamaAreia',
           'compra_Animais', 'compra_Terras', 'gastoEmprestimosJurosPagos')
    .execute()
)
# Importar dataframe
df_componentescustos_excel = pd.DataFrame(tab_componentescustos_excel.data)

# Renomear
df_componentescustos_excel.rename(columns={'idCusto':'ID'}, inplace=True)

# # Definir colunas de parcelas para transformar em int
# colunas_custo = ['ID','consumoLeiteBezerro', 'consumoMaoObraFamiliar',
#                     'consumoMaoObraContratada','consumoLeiteDescartado','gastoSucedaneo',
#                     'gastoMaterialOrdenha', 'gastoReproducao','gastoHormonios',
#                     'gastoMedicamentosVacinas','gastoAssistenciaTecnica','gastoImpostoTaxas',
#                     'gastoArrendamento','gastoReparosConsertos','gastoAdministrativo',
#                     'gastoAcessoriosDespesasGerais','gastoCamaAreia','compra_Animais','compra_Terras','gastoEmprestimosJurosPagos']
# # Iterar para definir formatos
# for col in colunas_custo:
#     df_componentescustos_excel[col] = df_componentescustos_excel[col].str.replace('.', '')
#     df_componentescustos_excel[col] = df_componentescustos_excel[col].str.replace(',', '.')
#     # Transformar qntconsumida
#     df_componentescustos_excel[col] = pd.to_numeric(df_componentescustos_excel[col], errors='coerce')
#     df_componentescustos_excel[col] = df_componentescustos_excel[col].fillna(0)

# Transformar data em datetime
df_componentescustos_excel['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_componentescustos_excel['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# Trazer o preço do leite
df_componentescustos_excel = df_componentescustos_excel.merge(df_rendabruta[['mesReferencia', 'idFazenda', 'def_precoLeite']],
                                                 on=['idFazenda', 'mesReferencia'], how='left')

# Deflacionar os dados
df_componentescustos_excel = df_componentescustos_excel.merge(df_igpdi[['data','deflator']],left_on='mesReferencia',right_on='data', how='left')

# Colunas a deflacionar
colunas_deflacionar = ['gastoSucedaneo','gastoMaterialOrdenha', 'gastoReproducao','gastoHormonios','gastoMedicamentosVacinas',
                       'gastoAssistenciaTecnica','gastoImpostoTaxas','gastoArrendamento','gastoReparosConsertos','gastoAdministrativo',
                       'gastoAcessoriosDespesasGerais','gastoCamaAreia','compra_Animais',
                       'compra_Terras','gastoEmprestimosJurosPagos']

# Deflacionar as colunas
for variable in colunas_deflacionar:
    nome_var = 'def_'+variable
    df_componentescustos_excel[nome_var] = df_componentescustos_excel[variable]/df_componentescustos_excel['deflator']

# =============== CONSUMO DE LEITE ===========================
# Criar custo com consumo de leite
colunas_consumo_leite = ['consumoLeiteBezerro', 'consumoMaoObraFamiliar', 'consumoMaoObraContratada', 'consumoLeiteDescartado']
for consumo in colunas_consumo_leite:
    gasto_consumo = 'gasto'+consumo
    df_componentescustos_excel[gasto_consumo] = df_componentescustos_excel[consumo]*df_componentescustos_excel['def_precoLeite']

# =============== ALIMENTAÇÃO ===========================
# Importar os custos com alimentação
df_componentescustos_excel = (
    df_componentescustos_excel.merge(df_custo_alimentacao[['idFazenda','mesReferencia',
                                                     'custoAlimentacao_Concentrado_Minerais','custoAlimentacao_Volumoso']],
                               on=['idFazenda','mesReferencia'],
                               how='left')
)

# =============== ENERGIA E COMBUSTÍVEL ===========================
df_energiacombustivel_etl = df_energiacombustivel[['idFazenda','mesReferencia','def_custoEnergia','def_custoCombustivel']].copy()
# Mesclar com componentes de custo
df_componentescustos_excel = df_componentescustos_excel.merge(df_energiacombustivel_etl, on=['idFazenda','mesReferencia'],how='left')

# =============== CUSTO COM RECEITA ===========================
df_componentescustos_excel = df_componentescustos_excel.merge(df_receitaforrageira, on=['idFazenda','mesReferencia'],how='left')

# =============== MÃO DE OBRA ===========================
df_componentescustos_excel = df_componentescustos_excel.merge(df_mdo[['idFazenda','mesReferencia','custoMDOContratada']],on=['idFazenda','mesReferencia'],how='left')

# =============== Inserir Leite Consumido nos Custos específicos ==============================
# Custo da MDO Contradada = Custo da MDO + Custo do Leite Consumido da MDO Contratada
df_componentescustos_excel['custoMDOContratada'] = df_componentescustos_excel[['custoMDOContratada', 'gastoconsumoMaoObraContratada']].sum(axis=1)
# Custo com Qualidade do Leite = Custo com Qualidade do leite + Custo do Leite Descartado
df_componentescustos_excel['def_gastoMaterialOrdenha'] = df_componentescustos_excel[['def_gastoMaterialOrdenha', 'gastoconsumoLeiteDescartado']].sum(axis=1)

# =============== COE ===========================
# Lista de colunas a serem somadas
colunas_coe = [
    'def_gastoSucedaneo',
    'def_gastoMaterialOrdenha',
    'def_gastoReproducao',
    'def_gastoHormonios',
    'def_gastoMedicamentosVacinas',
    'def_gastoAssistenciaTecnica',
    'def_gastoImpostoTaxas',
    'def_gastoArrendamento',
    'def_gastoReparosConsertos',
    'def_gastoAdministrativo',
    'def_gastoAcessoriosDespesasGerais',
    'def_gastoCamaAreia',
    'gastoconsumoLeiteBezerro',
    'custoAlimentacao_Concentrado_Minerais',
    'custoAlimentacao_Volumoso',
    'def_custoEnergia',
    'def_custoCombustivel',
    'custoReceitasConcentrado',
    'custoReceitasVolumoso',
    'custoMDOContratada'
]

# Criar uma nova coluna com a soma de todas as colunas listadas por linha
df_componentescustos_excel['coe'] = df_componentescustos_excel[colunas_coe].sum(axis=1)
 
df_componentescustos_excel = df_componentescustos_excel[~df_componentescustos_excel.duplicated(subset=['idFazenda', 'mesReferencia'], keep='first')].sort_values(['idFazenda','mesReferencia'])

# Mostrar colunas e tipos
print(df_componentescustos_excel.dtypes)

df_componentescustos_excel.to_excel('teste_coe.xlsx', index=False)

ID                                               object
idFazenda                                         int64
mesReferencia                            datetime64[ns]
consumoLeiteBezerro                             float64
consumoMaoObraFamiliar                          float64
consumoMaoObraContratada                        float64
consumoLeiteDescartado                          float64
gastoSucedaneo                                  float64
gastoMaterialOrdenha                            float64
gastoReproducao                                 float64
gastoHormonios                                  float64
gastoMedicamentosVacinas                        float64
gastoAssistenciaTecnica                         float64
gastoImpostoTaxas                               float64
gastoArrendamento                               float64
gastoReparosConsertos                           float64
gastoAdministrativo                             float64
gastoAcessoriosDespesasGerais                   

### Unificar

In [44]:
# Unificar
df_componentescustos = pd.concat([df_componentescustos_excel, df_componentescustos_shp])
#df_componentescustos.head()

df_componentescustos.head()

,ID,idFazenda,mesReferencia,consumoLeiteBezerro,consumoMaoObraFamiliar,consumoMaoObraContratada,consumoLeiteDescartado,gastoSucedaneo,gastoMaterialOrdenha,gastoReproducao,...,custoAlimentacao_Concentrado_Minerais,custoAlimentacao_Volumoso,def_custoEnergia,def_custoCombustivel,custoReceitasConcentrado,custoReceitasVolumoso,custoMDOContratada,coe,gastoEnergiaEletrica,gastoCombustivel
5860,M766,1,2022-05-01,2500.00,0.00,0.00,0.00,0.00,337.44,378.75,...,30658.29,0.00,1045.70,1187.31,NaN,NaN,0.00,45546.16,NaN,NaN
5861,M767,1,2022-06-01,3000.00,0.00,0.00,0.00,0.00,358.44,478.75,...,30467.92,0.00,1034.65,1179.94,NaN,NaN,0.00,56692.91,NaN,NaN
5862,M768,1,2022-07-01,3000.00,0.00,0.00,0.00,0.00,358.44,378.75,...,30582.69,0.00,1043.12,0.00,NaN,NaN,0.00,46709.20,NaN,NaN
5863,M769,1,2022-08-01,80.00,20.00,0.00,0.00,0.00,7861.84,545.00,...,34445.15,0.00,1840.22,5649.88,NaN,NaN,0.00,68869.30,NaN,NaN
5865,M770,1,2022-09-01,80.00,20.00,0.00,0.00,0.00,5205.80,4716.90,...,40601.27,0.00,1796.74,8904.13,NaN,NaN,0.00,87895.32,NaN,NaN


## Rebanho

### Sharepoint

In [45]:
# Rebanho
tab_rebanho = (
    supabase.table('tab_rebanho')
    .select('idFazenda','mesReferencia','qtdeVacasEmLactacao','qtdeVacasSecas','qtdeAnimaisAleitamento','qtdeAnimaisRecria','qtdeOutrasCategorias',
            'valorunitVacasEmLactacao','valorunitVacasSecas','valorunitAnimaisAleitamento','valorunitAnimaisRecria','valorunitOutrasCategorias',
            'qtdeMacho','valorunitMacho')
    .execute()
)

# Criar dataframe
df_rebanho_shp = pd.DataFrame(tab_rebanho.data)
# Mostrar tipo das colunas
print(f"Tipo das colunas antes do ETL:\n{df_rebanho_shp.dtypes}")
# 1. Garantir que 'mesReferencia' em df_rebanho_shp é um tipo datetime e remover fuso horário
# Se a coluna já é datetime64[ns] com fuso horário (+00), .dt.tz_localize(None) a torna naive.
# Se for string, pd.to_datetime a converte e depois remove o fuso horário.
df_rebanho_shp['mesReferencia'] = pd.to_datetime(df_rebanho_shp['mesReferencia'], errors='coerce').dt.tz_localize(None)

# 2. Corrigir a inversão de mês e dia
# Extrair o dia e mês
def corrigir_data_invertida(data):
    """
    Corrige datas onde dia e mês foram invertidos.
    Se o dia for maior que 12, assume que está invertido.
    """
    dia = data.day
    mes = data.month
    ano = data.year

    # Se dia > 12, certamente está invertido (mês não pode ser > 12)
    if dia > 1:
        return pd.Timestamp(year=ano, month=dia, day=mes)
    else:
        # Mantém a data original
        return data

# Aplicar a função em cada linha
df_rebanho_shp['mesReferencia'] = df_rebanho_shp['mesReferencia'].apply(corrigir_data_invertida)

# # # Definir colunas a transformar
# # colunas_transformar = ['qtdeOutrasCategorias','valorunitAnimaisRecria','valorunitOutrasCategorias','qtdeMacho','valorunitMacho']
# # # Tranformar colunas em numérico
# # df_rebanho_shp = tratar_colunas_numericas(df_rebanho_shp,colunas_transformar)
# Criar coluna de totais de vacas
df_rebanho_shp['totalVacas'] = df_rebanho_shp.loc[:,['qtdeVacasEmLactacao','qtdeVacasSecas']].sum(axis=1)
# Colunas para total de animais
colunas_animais = ['qtdeVacasEmLactacao','qtdeVacasSecas','qtdeAnimaisAleitamento',
                   'qtdeAnimaisRecria','qtdeOutrasCategorias','qtdeMacho']
# Criar coluna de totais de animais
df_rebanho_shp['totalAnimais'] = df_rebanho_shp.loc[:,colunas_animais].sum(axis=1)
# Criar coluna de vacas em lactação/total de vacas
df_rebanho_shp['vacasLactacao_totalVacas'] = df_rebanho_shp['qtdeVacasEmLactacao'] / df_rebanho_shp['totalVacas']
# Criar coluna de vacas em lactação/total de animais
df_rebanho_shp['vacasLactacao_totalAnimais'] = df_rebanho_shp['qtdeVacasEmLactacao'] / df_rebanho_shp['totalAnimais']

# # Deletar tabela original
# del tab_rebanho

# Remover duplicatas 
df_rebanho_shp = df_rebanho_shp[~df_rebanho_shp.duplicated(subset=['idFazenda', 'mesReferencia'], keep='first')].sort_values(['idFazenda','mesReferencia'])


# Mostrar tipo das colunas
print(f"\nTipo das colunas depois do ETL:\n{df_rebanho_shp.dtypes}")
# Mostrar tabelas
#df_rebanho_shp.head()

Tipo das colunas antes do ETL:
idFazenda                        int64
mesReferencia                   object
qtdeVacasEmLactacao            float64
qtdeVacasSecas                 float64
qtdeAnimaisAleitamento         float64
qtdeAnimaisRecria              float64
qtdeOutrasCategorias           float64
valorunitVacasEmLactacao       float64
valorunitVacasSecas            float64
valorunitAnimaisAleitamento    float64
valorunitAnimaisRecria         float64
valorunitOutrasCategorias      float64
qtdeMacho                      float64
valorunitMacho                 float64
dtype: object

Tipo das colunas depois do ETL:
idFazenda                               int64
mesReferencia                  datetime64[ns]
qtdeVacasEmLactacao                   float64
qtdeVacasSecas                        float64
qtdeAnimaisAleitamento                float64
qtdeAnimaisRecria                     float64
qtdeOutrasCategorias                  float64
valorunitVacasEmLactacao              float64
valoruni

### Excel

In [46]:
# Rebanho
tab_rebanho = (
    supabase.table('xlsx_rebanho')
    .select('idFazenda','mesReferencia','qtdeVacasEmLactacao','qtdeVacasSecas','qtdeAnimaisAleitamento','qtdeAnimaisRecria','qtdeOutrasCategorias',
            'valorunitVacasEmLactacao','valorunitVacasSecas','valorunitAnimaisAleitamento','valorunitAnimaisRecria','valorunitOutrasCategorias',
            'qtdeMacho','valorunitMacho')
    .execute()
)

# Criar dataframe
df_rebanho_excel = pd.DataFrame(tab_rebanho.data)
# Mostrar tipo das colunas
print(f"Tipo das colunas antes do ETL:\n{df_rebanho_excel.dtypes}")
# 1. Garantir que 'mesReferencia' em df_rebanho_excel é um tipo datetime e remover fuso horário
# Se a coluna já é datetime64[ns] com fuso horário (+00), .dt.tz_localize(None) a torna naive.
# Se for string, pd.to_datetime a converte e depois remove o fuso horário.
df_rebanho_excel['mesReferencia'] = pd.to_datetime(df_rebanho_excel['mesReferencia'], errors='coerce').dt.tz_localize(None)

# 2. Corrigir a inversão de mês e dia
# Extrair o dia e mês
def corrigir_data_invertida(data):
    """
    Corrige datas onde dia e mês foram invertidos.
    Se o dia for maior que 12, assume que está invertido.
    """
    dia = data.day
    mes = data.month
    ano = data.year

    # Se dia > 12, certamente está invertido (mês não pode ser > 12)
    if dia > 1:
        return pd.Timestamp(year=ano, month=dia, day=mes)
    else:
        # Mantém a data original
        return data

# Aplicar a função em cada linha
df_rebanho_excel['mesReferencia'] = df_rebanho_excel['mesReferencia'].apply(corrigir_data_invertida)

# # # Definir colunas a transformar
# # colunas_transformar = ['qtdeOutrasCategorias','valorunitAnimaisRecria','valorunitOutrasCategorias','qtdeMacho','valorunitMacho']
# # # Tranformar colunas em numérico
# # df_rebanho_excel = tratar_colunas_numericas(df_rebanho_excel,colunas_transformar)
# Criar coluna de totais de vacas
df_rebanho_excel['totalVacas'] = df_rebanho_excel.loc[:,['qtdeVacasEmLactacao','qtdeVacasSecas']].sum(axis=1)
# Colunas para total de animais
colunas_animais = ['qtdeVacasEmLactacao','qtdeVacasSecas','qtdeAnimaisAleitamento',
                   'qtdeAnimaisRecria','qtdeOutrasCategorias','qtdeMacho']
# Criar coluna de totais de animais
df_rebanho_excel['totalAnimais'] = df_rebanho_excel.loc[:,colunas_animais].sum(axis=1)
# Criar coluna de vacas em lactação/total de vacas
df_rebanho_excel['vacasLactacao_totalVacas'] = df_rebanho_excel['qtdeVacasEmLactacao'] / df_rebanho_excel['totalVacas']
# Criar coluna de vacas em lactação/total de animais
df_rebanho_excel['vacasLactacao_totalAnimais'] = df_rebanho_excel['qtdeVacasEmLactacao'] / df_rebanho_excel['totalAnimais']

# # Deletar tabela original
# del tab_rebanho

# Remover duplicatas 
df_rebanho_excel = df_rebanho_excel[~df_rebanho_excel.duplicated(subset=['idFazenda', 'mesReferencia'], keep='first')].sort_values(['idFazenda','mesReferencia'])


# Mostrar tipo das colunas
print(f"\nTipo das colunas depois do ETL:\n{df_rebanho_excel.dtypes}")
# Mostrar tabelas
#df_rebanho_excel.head()

Tipo das colunas antes do ETL:
idFazenda                        int64
mesReferencia                   object
qtdeVacasEmLactacao              int64
qtdeVacasSecas                   int64
qtdeAnimaisAleitamento           int64
qtdeAnimaisRecria                int64
qtdeOutrasCategorias             int64
valorunitVacasEmLactacao       float64
valorunitVacasSecas            float64
valorunitAnimaisAleitamento    float64
valorunitAnimaisRecria         float64
valorunitOutrasCategorias      float64
qtdeMacho                        int64
valorunitMacho                 float64
dtype: object

Tipo das colunas depois do ETL:
idFazenda                               int64
mesReferencia                  datetime64[ns]
qtdeVacasEmLactacao                     int64
qtdeVacasSecas                          int64
qtdeAnimaisAleitamento                  int64
qtdeAnimaisRecria                       int64
qtdeOutrasCategorias                    int64
valorunitVacasEmLactacao              float64
valoruni

### Unificar

In [47]:
# Unificar
df_rebanho = pd.concat([df_rebanho_excel, df_rebanho_shp])

## Área

In [48]:
%%time
# Área
tab_area = (
    supabase.table('tab_area')
    .select('ID','idFazenda','especificacaoArea','nomeArea','tamanhoArea','arrendamento','dataInicio',
            'dataFim','precoTerraNua','Excluido')
    .eq('Excluido',0)
    .execute()
)
# Definir Dataframe
df_area = pd.DataFrame(tab_area.data)
# Limpar área com data de início vazia
df_area = df_area[~df_area['dataInicio'].isna()]
# Mostrar colunas e seus tipos
print(f"Tipo das colunas antes do ETL:\n{df_area.dtypes}")
# Transformar Data Início e Fim em datetime
for col in ['dataInicio', 'dataFim']:
    df_area[col] = (
        pd.to_datetime(
            pd.to_datetime(
                df_area[col],
                errors='coerce'
            )
            .dt
            .date,
            format='%Y-%m-%d'
        )
    )

# Parâmetros e base
ini = pd.to_datetime('2022-01-01')
# Calcula o primeiro dia do mês atual:
fim = pd.Timestamp(datetime.today().replace(day=1))
meses = pd.date_range(start=ini, end=fim, freq='MS')
fazendas_unicas = df_area['idFazenda'].unique()
df_base = pd.MultiIndex.from_product([fazendas_unicas, meses], names=['idFazenda', 'mesReferencia']).to_frame(index=False)
# Expanda linhas conforme meses ativos
def gerar_periodos(row):
    data_inicio = max(row['dataInicio'], ini)
    data_fim = min(row['dataFim'] if pd.notna(row['dataFim']) else fim, fim)
    return pd.date_range(start=data_inicio, end=data_fim, freq='MS')
df_area_expand = (
    df_area
    .assign(mesReferencia=df_area.apply(gerar_periodos, axis=1))
    .explode('mesReferencia')
    .dropna(subset=['mesReferencia'])
).reset_index(drop=True)
df_area_expand = df_area_expand[
    (df_area_expand['mesReferencia'] >= ini) & (df_area_expand['mesReferencia'] <= fim)
]


# Máscaras de categoria
reserva = ["APP", "Reserva Legal"]
outros = ["Área de Outras Atividades", "Área em Pousio"]
benfeitoria = ['Área de Benfeitorias e Estradas']
df_area_expand['is_sem_reserva'] = ~df_area_expand['especificacaoArea'].isin(reserva + outros)
df_area_expand['is_atividade_reserva'] = ~df_area_expand['especificacaoArea'].isin(outros)
df_area_expand['forrageira'] = ~df_area_expand['especificacaoArea'].isin(reserva + outros + benfeitoria)
# Área Arrendada: Soma de arrendamento
df_area_expand['areaArrendada'] = df_area_expand['arrendamento'].where(df_area_expand['arrendamento'] > 0, 0)
# Área da Atividade sem Reserva: Todas menos APP, Reserva Legal, Pousio e Outras Atividades
df_area_expand['areaAtividadeSemReserva'] = (df_area_expand['tamanhoArea'] + df_area_expand['arrendamento']).where(df_area_expand['is_sem_reserva'], 0)
# Área da Atividade sem Reserva: Todas menos Pousio e Outras Atividades
df_area_expand['areaAtividadeReserva'] = (df_area_expand['tamanhoArea'] + df_area_expand['arrendamento']).where(df_area_expand['is_atividade_reserva'], 0)
# Área de Forrageira: Todas menos Pousio, Outras Atividades, Reserva legal, APP e Benfeitoria
df_area_expand['areaForrageira'] = (df_area_expand['tamanhoArea'] + df_area_expand['arrendamento']).where(df_area_expand['forrageira'], 0)
# Área de Forrageira 
df_area_expand['is_atividade_reserva'] = ~df_area_expand['especificacaoArea'].isin(outros)
# Todas menos arrendamento
df_area_expand['areaPropria'] = df_area_expand['tamanhoArea']
# Todas incluindo arrendamento
df_area_expand['areaTotal'] = df_area_expand[['tamanhoArea', 'arrendamento']].sum(axis=1)  
# Criar Estoque de Terra
df_area_expand['estoqueTerra'] = df_area_expand['areaPropria'] * df_area_expand['precoTerraNua']
# Agrupamento
df_area_ativa = (
    df_area_expand.groupby(['idFazenda', 'mesReferencia'])[
        ['areaArrendada', 'areaAtividadeSemReserva', 'areaAtividadeReserva', 'areaForrageira', 'areaPropria', 'areaTotal','estoqueTerra']
    ].sum().reset_index()
)
# Criar preço da terra nua agregado 
df_area_ativa['precoTerraNua'] = df_area_ativa['estoqueTerra'] / df_area_ativa['areaTotal']

# Mesclar com IGPDI
df_area_ativa = (
    df_area_ativa
    .merge(df_igpdi[['data','deflator']],
           left_on='mesReferencia',
           right_on='data',
           how='left')
)
# Eliminar quando não há deflator
df_area_ativa.dropna(subset=['deflator'], inplace=True)
# Deflacionar preço da terra Nua 
df_area_ativa = deflacionar_colunas(df_area_ativa, ['precoTerraNua', 'estoqueTerra'])

df_area_ativa = df_area_ativa.drop(['data', 'deflator', 'precoTerraNua', 'estoqueTerra'], axis=1)
# Deletar tabelas usadas no tratamento 
del df_area_expand
gc.collect()

# Mostrar Dataframe
df_area_ativa.loc[df_area_ativa['idFazenda']==27]

Tipo das colunas antes do ETL:
ID                     int64
idFazenda              int64
especificacaoArea     object
nomeArea              object
tamanhoArea          float64
arrendamento         float64
dataInicio            object
dataFim               object
precoTerraNua        float64
Excluido               int64
dtype: object
CPU times: total: 6.94 s
Wall time: 7.44 s


,idFazenda,mesReferencia,areaArrendada,areaAtividadeSemReserva,areaAtividadeReserva,areaForrageira,areaPropria,areaTotal,def_precoTerraNua,def_estoqueTerra
1294,27,2022-01-01,15.86,8.00,9.69,8.00,1.69,17.55,7298.63,128090.98
1295,27,2022-02-01,15.86,8.00,9.69,8.00,1.69,17.55,7190.62,126195.43
1296,27,2022-03-01,15.86,8.00,9.69,8.00,1.69,17.55,7024.22,123275.10
1297,27,2022-04-01,15.86,8.00,9.69,8.00,1.69,17.55,6995.31,122767.65
1298,27,2022-05-01,15.86,8.00,9.69,8.00,1.69,17.55,6947.36,121926.15
1299,27,2022-06-01,15.86,8.00,9.69,8.00,1.69,17.55,6904.22,121169.04
1300,27,2022-07-01,15.86,8.00,9.69,8.00,1.69,17.55,6930.23,121625.46
1301,27,2022-08-01,15.86,8.00,9.69,8.00,1.69,17.55,6968.78,122302.11
1302,27,2022-09-01,15.86,8.00,9.69,8.00,1.69,17.55,7054.59,123807.98
1303,27,2022-10-01,15.86,8.00,9.69,8.00,1.69,17.55,7098.32,124575.51


## Qualidade do Leite

### Sharepoint

In [49]:
# ============================================================
# 1. Carregar qualidade SHP (sem merge com renda ainda)
# ============================================================
tab_qualidadeleite_shp = (
    supabase.table('tab_qualidadeleite')
    .select('idFazenda','mesReferencia','CCS','CPP','Gordura','Proteina')
    .execute()
)
df_qualidade_shp = pd.DataFrame(tab_qualidadeleite_shp.data)
df_qualidade_shp['mesReferencia'] = pd.to_datetime(
    df_qualidade_shp['mesReferencia'], errors='coerce'
).dt.normalize()
df_qualidade_shp['idFazenda'] = df_qualidade_shp['idFazenda'].astype('float')

## Transformar mes em date
df_qualidade_shp['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_qualidade_shp['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# ============================================================
# 2. Carregar qualidade Excel (sem merge com renda ainda)
# ============================================================
tab_qualidadeleite_excel = (
    supabase.table('xlsx_qualidadeleite')
    .select('idFazenda','mesReferencia','CCS','CPP','Gordura','Proteina')
    .execute()
)
df_qualidade_excel = pd.DataFrame(tab_qualidadeleite_excel.data)
df_qualidade_excel['mesReferencia'] = pd.to_datetime(
    df_qualidade_excel['mesReferencia'], errors='coerce'
).dt.normalize()
df_qualidade_excel['idFazenda'] = df_qualidade_excel['idFazenda'].astype('float')

## Transformar mes em date
df_qualidade_excel['mesReferencia'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_qualidade_excel['mesReferencia'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# ============================================================
# 3. ✅ Combinar qualidades ANTES do merge com renda
#    Prioridade: excel primeiro, shp completa o restante
# ============================================================
df_qualidade = (
    pd.concat([df_qualidade_excel, df_qualidade_shp])
    .drop_duplicates(subset=['idFazenda', 'mesReferencia'], keep='first')
    .sort_values(['idFazenda', 'mesReferencia'])
    .reset_index(drop=True)
)

print(f"Meses qualidade excel: {len(df_qualidade_excel)}")
print(f"Meses qualidade shp:   {len(df_qualidade_shp)}")
print(f"Meses qualidade total: {len(df_qualidade)}")  # deve ser > que cada um

# ============================================================
# 4. ✅ UM único merge com df_rendabruta como base
# ============================================================
colunas_float = ['CCS', 'CPP', 'Gordura', 'Proteina']

df_leite = (
    df_rendabruta[['idFazenda', 'mesReferencia', 'leiteProduzido', 'def_precoLeite']]
    .merge(
        df_qualidade[['idFazenda', 'mesReferencia'] + colunas_float],
        on=['idFazenda', 'mesReferencia'],
        how='left'
    )
    .drop_duplicates(subset=['idFazenda', 'mesReferencia'], keep='first')
    .sort_values(['idFazenda', 'mesReferencia'])
    .reset_index(drop=True)
)

# 5. Criar colunas derivadas
for col in colunas_float:
    df_leite['abs'+col] = df_leite[col] * df_leite['leiteProduzido']

df_leite['leiteDiario'] = df_leite['leiteProduzido'] / 30.42
df_leite['idFazenda'] = df_leite['idFazenda'].astype('float')

# 6. Validar
print(f"\nLinhas df_leite: {df_leite.shape[0]}")
print(f"Duplicatas: {df_leite.duplicated(subset=['idFazenda','mesReferencia']).sum()}")
print(f"\nFazendas com 12 meses no período:")
cobertura = (
    df_leite[
        (df_leite['mesReferencia'] >= pd.Timestamp('2023-04-01')) &
        (df_leite['mesReferencia'] <= pd.Timestamp('2024-03-01')) &
        (df_leite['leiteProduzido'].notna())
    ]
    .groupby('idFazenda')['mesReferencia'].count()
    .value_counts().sort_index()
)
print(cobertura)

Meses qualidade excel: 994
Meses qualidade shp:   14446
Meses qualidade total: 15424

Linhas df_leite: 20403
Duplicatas: 0

Fazendas com 12 meses no período:
mesReferencia
1      34
2      21
3     118
4      11
5       3
6       4
7       9
8       6
9      18
10      2
11      4
12    116
Name: count, dtype: int64


## Patrimônio

In [50]:
# Patrimônio
tab_patrimonio = (
    supabase.table('tab_inventario')
    .select('ID','idFazenda','dataAquisicao','classificacao','tipoItem','valorAquisicao', 'vidaUtil','dataFim','quantidade','Excluido')
    .eq('Excluido', 0)
    .execute()
)
# Criar dataframe
df_patrimonio = pd.DataFrame(tab_patrimonio.data)
# Mostrar tipo das colunas
print(f"Tipos das colunas antes do ELT:\n{df_patrimonio.dtypes}")
# Transformar mesReferencia em Datetime
datas_patrimonio = ['dataAquisicao', 'dataFim']
for col in datas_patrimonio:
    df_patrimonio[col] = (
        pd.to_datetime(
            pd.to_datetime(
                df_patrimonio[col],
                errors='coerce'
            )
            .dt
            .date,
            format='%Y-%m-%d'
        )
    )
# Valor do bem
df_patrimonio['valorPatrimonio'] = df_patrimonio['quantidade'] * df_patrimonio['valorAquisicao']
# Depreciação
df_patrimonio['depreciacao'] = df_patrimonio['valorPatrimonio'] / (df_patrimonio['vidaUtil']*12)
# Estoque de Capital
df_patrimonio['estoqueCapitalMensal'] = df_patrimonio['valorPatrimonio']/24
# criar coluna de fim da vida útil
df_patrimonio['dataVidaUtil'] = df_patrimonio.apply(
    lambda row: row['dataAquisicao'] + pd.DateOffset(months=int(row['vidaUtil'] * 12)) if pd.notna(row['vidaUtil']) and pd.notna(row['dataAquisicao']) else pd.NaT,
    axis=1
)
# Mostrar tipo das colunas
print(f"Tipos das colunas depois do ELT:\n{df_patrimonio.dtypes}")
# Mostrar tipo das colunas
print(f"Tamanho do dataframe de Patrimônio:\n{df_patrimonio.shape}")
# Mostrar DF
df_patrimonio.head()

Tipos das colunas antes do ELT:
ID                  int64
idFazenda           int64
dataAquisicao      object
classificacao      object
tipoItem            int64
valorAquisicao    float64
vidaUtil            int64
dataFim            object
quantidade        float64
Excluido            int64
dtype: object
Tipos das colunas depois do ELT:
ID                               int64
idFazenda                        int64
dataAquisicao           datetime64[ns]
classificacao                   object
tipoItem                         int64
valorAquisicao                 float64
vidaUtil                         int64
dataFim                 datetime64[ns]
quantidade                     float64
Excluido                         int64
valorPatrimonio                float64
depreciacao                    float64
estoqueCapitalMensal           float64
dataVidaUtil            datetime64[ns]
dtype: object
Tamanho do dataframe de Patrimônio:
(19360, 14)


,ID,idFazenda,dataAquisicao,classificacao,tipoItem,valorAquisicao,vidaUtil,dataFim,quantidade,Excluido,valorPatrimonio,depreciacao,estoqueCapitalMensal,dataVidaUtil
0,11409,752,2021-02-01,Máquinas e equipamentos,107,124300.00,15,NaT,1.00,0,124300.00,690.56,5179.17,2036-02-01
1,17830,165,2025-09-01,Máquinas e equipamentos,68,15000.00,15,NaT,1.00,0,15000.00,83.33,625.00,2040-09-01
2,17831,165,2025-04-01,Máquinas e equipamentos,92,30000.00,15,NaT,6.00,0,180000.00,1000.00,7500.00,2040-04-01
3,17761,1072,2024-04-01,Máquinas e equipamentos,102,10000.00,15,NaT,1.00,0,10000.00,55.56,416.67,2039-04-01
4,17832,165,2026-01-01,Máquinas e equipamentos,92,4000.00,15,NaT,2.00,0,8000.00,44.44,333.33,2041-01-01


## Inventário

In [51]:
# Define Dataframe para primeiro Inventário
df_inventario = pd.DataFrame() # DF vazio
# Selecione o inventário
try:
    # Todos os registros
    response = (
        supabase
        .table('tab_inventario')
        .select('ID', 'idFazenda', 'dataAquisicao', 'classificacao', 'valorAquisicao', 'vidaUtil', 'dataFim', 'quantidade')
        .eq('Excluido', 0)
        .execute()
    )

    if response.data:
        df_inventario = pd.DataFrame(response.data)
        print(f"Foram encontrados {len(df_inventario)} registros na extração completa.")

    else:
        print('Nenhum registro encontrado.')

except Exception as e:
    print(f"Ocorreu um erro ao consultar a tab_inventario no Supabase: {e}")



# Importar a tabela de Revisão do IR
df_novovalor = pd.DataFrame()
# Selecione o inventário
try:
    # Todos os registros
    response = (
        supabase
        .table('tab_novovalor')
        .select('idFazenda', 'idInventario','novoValor', 'dataRevisao')
        .eq('Excluido', 0)
        .execute()
    )

    if response.data:
        df_novovalor = pd.DataFrame(response.data)
        print(f"Foram encontrados {len(df_novovalor)} registros na extração completa do Novo Valor.")

    else:
        print('Nenhum registro encontrado.')
except Exception as e:
    print(f"Ocorreu um erro ao consultar a tab_inventario no Supabase: {e}")
print(df_novovalor.dtypes)    
# Mesclar com novo valor com inventario
df_inventario = (
    df_inventario
    .merge(
        df_novovalor,
        left_on=['idFazenda','ID'],
        right_on= ['idFazenda','idInventario'],
        how='left'
    )
)

# Transformar mesReferencia em Datetime
datas_patrimonio = ['dataAquisicao', 'dataFim', 'dataRevisao']
for col in datas_patrimonio:
    df_inventario[col] = (
        pd.to_datetime(
            pd.to_datetime(
                df_inventario[col],
                errors='coerce'
            )
            .dt
            .date,
            format='%Y-%m-%d'
        )
    )
print(df_inventario.dtypes)

# # criar coluna de fim da vida útil
df_inventario['dataVidaUtil'] = df_inventario.apply(
    lambda row: row['dataAquisicao'] + pd.DateOffset(months=int(row['vidaUtil'] * 12)) if pd.notna(row['vidaUtil']) and pd.notna(row['dataAquisicao']) else pd.NaT,
    axis=1
)

Foram encontrados 19360 registros na extração completa.
Foram encontrados 1243 registros na extração completa do Novo Valor.
idFazenda         int64
idInventario      int64
novoValor       float64
dataRevisao      object
dtype: object
ID                         int64
idFazenda                  int64
dataAquisicao     datetime64[ns]
classificacao             object
valorAquisicao           float64
vidaUtil                   int64
dataFim           datetime64[ns]
quantidade               float64
idInventario             float64
novoValor                float64
dataRevisao       datetime64[ns]
dtype: object


## Depreciação

In [52]:
import pandas as pd
import numpy as np
import gc
from datetime import datetime

# =============================================================================
# 1. CONFIGURAÇÃO INICIAL
# =============================================================================
ini = pd.Timestamp('1994-08-01').to_period('M').to_timestamp()
fim = pd.Timestamp(datetime.today().replace(day=1))

# =============================================================================
# 2. PRÉ-PROCESSAMENTO DO IGP-DI
# Resolve gaps na série temporal antes de qualquer merge.
# ffill() é seguro aqui pois opera sobre série ordenada por data.
# =============================================================================
df_igpdi['data'] = (
    pd.to_datetime(df_igpdi['data'], errors='coerce')
    .dt.to_period('M')
    .dt.to_timestamp()
)

df_igpdi = (
    df_igpdi
    .sort_values('data')
    .set_index('data')
    .reindex(pd.date_range(
        df_igpdi['data'].min(),
        df_igpdi['data'].max(),
        freq='MS'
    ))
    .ffill()
    .reset_index()
    .rename(columns={'index': 'data'})
)

# Âncora para ativos adquiridos antes do início da série IGP-DI
igpdi_mais_antigo = df_igpdi.loc[df_igpdi['data'].idxmin(), 'igpdi']

# =============================================================================
# 3. PREPARAÇÃO DO INVENTÁRIO
# =============================================================================
df_patrimonio_etl = df_inventario[[
    'ID', 'idFazenda', 'classificacao', 'dataAquisicao',
    'dataFim', 'vidaUtil', 'quantidade', 'valorAquisicao'
]].copy()

# ID mantido como int64 para compatibilidade com idInventario
df_patrimonio_etl['ID']        = pd.to_numeric(df_patrimonio_etl['ID'], errors='coerce').astype('Int64')
df_patrimonio_etl['idFazenda'] = df_patrimonio_etl['idFazenda'].astype('Int64')

# Normaliza datas para início do mês
for col in ['dataAquisicao', 'dataFim']:
    df_patrimonio_etl[col] = (
        pd.to_datetime(df_patrimonio_etl[col], errors='coerce')
        .dt.to_period('M')
        .dt.to_timestamp()
    )

# =============================================================================
# 4. NOVO VALOR — busca o registro mais recente por inventário
# Equivalente à função latest_novovalor_per_inventory() do Método 2
# =============================================================================
if df_novovalor is not None and not df_novovalor.empty:

    df_novovalor['dataRevisao'] = (
        pd.to_datetime(df_novovalor['dataRevisao'], errors='coerce')
        .dt.to_period('M')
        .dt.to_timestamp()
    )

    # Mantém apenas o registro mais recente por inventário
    nv_latest = (
        df_novovalor
        .sort_values('dataRevisao', ascending=False)
        .drop_duplicates(subset=['idInventario'], keep='first')
    )[['idInventario', 'novoValor', 'dataRevisao']]
    
    # Alinha o tipo com df_patrimonio_etl['ID']
    nv_latest['idInventario'] = pd.to_numeric(nv_latest['idInventario'], errors='coerce').astype('Int64')

    df_patrimonio_etl = df_patrimonio_etl.merge(
        nv_latest,
        left_on='ID',
        right_on='idInventario',
        how='left'
    ).drop(columns=['idInventario'])

else:
    df_patrimonio_etl['novoValor']   = np.nan
    df_patrimonio_etl['dataRevisao'] = pd.NaT

# Normaliza dataRevisao para início do mês
df_patrimonio_etl['dataRevisao'] = (
    pd.to_datetime(df_patrimonio_etl['dataRevisao'], errors='coerce')
    .dt.to_period('M')
    .dt.to_timestamp()
)

# =============================================================================
# 5. RECALCULAR dataVidaUtil
# Determinístico: dataAquisicao + vidaUtil * 12 meses.
# Não usa o valor cadastrado no banco, que pode estar incorreto.
# =============================================================================
def _calcular_vida_util_end(row):
    if pd.isna(row['dataAquisicao']) or pd.isna(row['vidaUtil']):
        return pd.NaT
    try:
        months = int(float(row['vidaUtil']) * 12)
        return row['dataAquisicao'] + pd.DateOffset(months=months)
    except Exception:
        return pd.NaT

df_patrimonio_etl['dataVidaUtil'] = (
    df_patrimonio_etl
    .apply(_calcular_vida_util_end, axis=1)
)

df_patrimonio_etl['dataVidaUtil'] = (
    pd.to_datetime(df_patrimonio_etl['dataVidaUtil'], errors='coerce')
    .dt.to_period('M')
    .dt.to_timestamp()
)

# =============================================================================
# 6. CALCULAR start_month E end_month POR ATIVO
# start = max(dataAquisicao, ini)
# end   = min(dataFim, dataVidaUtil, fim)  ← corta pelo limite mais restritivo
# =============================================================================

# start_month
df_patrimonio_etl['start_month'] = (
    df_patrimonio_etl['dataAquisicao']
    .where(df_patrimonio_etl['dataAquisicao'].notna(), ini)
    .clip(lower=ini)
)

# end_month: constrói progressivamente ignorando NaT
def _min_ignorando_nat(a, b):
    """Retorna o menor entre dois Timestamps, ignorando NaT."""
    if pd.isna(a):
        return b
    if pd.isna(b):
        return a
    return min(a, b)

df_patrimonio_etl['end_month'] = fim

df_patrimonio_etl['end_month'] = df_patrimonio_etl.apply(
    lambda r: _min_ignorando_nat(r['end_month'], r['dataFim']), axis=1
)
df_patrimonio_etl['end_month'] = df_patrimonio_etl.apply(
    lambda r: _min_ignorando_nat(r['end_month'], r['dataVidaUtil']), axis=1
)

# Remove ativos sem janela válida
df_patrimonio_etl = df_patrimonio_etl[
    df_patrimonio_etl['start_month'].notna() &
    df_patrimonio_etl['end_month'].notna() &
    (df_patrimonio_etl['start_month'] <= df_patrimonio_etl['end_month'])
].copy()

# =============================================================================
# 7. EXPANSÃO DA LINHA DO TEMPO (EXPLODE)
# Gera um registro por mês ativo para cada ativo
# =============================================================================
def _gerar_periodos(row):
    return pd.date_range(
        start=row['start_month'],
        end=row['end_month'],
        freq='MS'
    )

df_depreciacao = (
    df_patrimonio_etl
    .assign(mesAtivo=df_patrimonio_etl.apply(_gerar_periodos, axis=1))
    .explode('mesAtivo')
    .dropna(subset=['mesAtivo'])
    .reset_index(drop=True)
)

df_depreciacao['mesAtivo'] = (
    pd.to_datetime(df_depreciacao['mesAtivo'])
    .dt.to_period('M')
    .dt.to_timestamp()
)

# =============================================================================
# 8. VALOR BASE DO PERÍODO
# Usa novoValor quando dataRevisao existe e já passou; caso contrário valorAquisicao
# =============================================================================
condicao_revisao_ativa = (
    df_depreciacao['dataRevisao'].notna() &
    (df_depreciacao['mesAtivo'] >= df_depreciacao['dataRevisao'])
)

df_depreciacao['valor_base'] = np.where(
    condicao_revisao_ativa,
    df_depreciacao['novoValor'],
    df_depreciacao['valorAquisicao']
)

df_depreciacao['valorPatrimonio'] = (
    df_depreciacao['quantidade'] * df_depreciacao['valor_base']
)

# =============================================================================
# 9. DEPRECIAÇÃO MENSAL BASE (sem correção monetária ainda)
# =============================================================================
vida_meses = df_depreciacao['vidaUtil'].astype(float) * 12.0

df_depreciacao['depreciacao'] = np.where(
    vida_meses > 0,
    df_depreciacao['valorPatrimonio'] / vida_meses,
    0.0
)

# # =============================================================================
# # 10. CORREÇÃO MONETÁRIA — IGP-DI
# # =============================================================================

# Data base do índice: dataRevisao (se revisão ativa) ou dataAquisicao
df_depreciacao['data_base_indice'] = (
    pd.Series(np.where(
        condicao_revisao_ativa,
        df_depreciacao['dataRevisao'],
        df_depreciacao['dataAquisicao']
    ))
    .astype('datetime64[ns]')
    .dt.to_period('M')
    .dt.to_timestamp()
)

# Merge 1: IGP-DI da época (mês corrente do ativo)
df_depreciacao = df_depreciacao.merge(
    df_igpdi[['data', 'igpdi']],
    left_on='mesAtivo',
    right_on='data',
    how='left'
).rename(columns={'igpdi': 'igpdi_epoca'}).drop(columns=['data'])

# Merge 2: IGP-DI da base (data de aquisição ou revisão)
df_depreciacao = df_depreciacao.merge(
    df_igpdi[['data', 'igpdi']],
    left_on='data_base_indice',
    right_on='data',
    how='left'
).rename(columns={'igpdi': 'igpdi_aquisicao'}).drop(columns=['data'])
# Âncora para ativos pré-1994: usa o IGP-DI mais antigo disponível
# fillna aplicado no DENOMINADOR, não no fator final
df_depreciacao['igpdi_aquisicao'] = (
    df_depreciacao['igpdi_aquisicao']
    .fillna(igpdi_mais_antigo)
)

# Fator de correção
df_depreciacao['fator_correcao'] = (
    df_depreciacao['igpdi_epoca'] / df_depreciacao['igpdi_aquisicao']
)

# fillna(1.0) apenas para mês corrente sem IGP-DI publicado
df_depreciacao = df_depreciacao.loc[df_depreciacao['fator_correcao'].notna()]

# Depreciação corrigida
df_depreciacao['def_depreciacao'] = df_depreciacao['depreciacao'] * df_depreciacao['fator_correcao']

df_depreciacao['def_depreciacao'] = (
    df_depreciacao['def_depreciacao']
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

# =============================================================================
# 11. AGRUPAMENTO FINAL
# =============================================================================
df_depreciacao['classificacao'] = (
    df_depreciacao['classificacao']
    .fillna('SemClassificacao')
)

df_final = (
    df_depreciacao[['idFazenda', 'classificacao', 'mesAtivo', 'def_depreciacao']]
    .groupby(['idFazenda', 'classificacao', 'mesAtivo'])
    .sum()
    .reset_index()
)

# Separação por classificação
df_estoquecapital_maquinas = df_final.loc[
    df_final['classificacao'] == 'Máquinas e equipamentos'
].copy()

df_estoquecapital_benfeitorias = df_final.loc[
    df_final['classificacao'] == 'Benfeitorias'
].copy()

# =============================================================================
# 12. LIMPEZA DE MEMÓRIA
# =============================================================================
del df_patrimonio_etl, df_depreciacao
gc.collect()

# =============================================================================
# 13. DIAGNÓSTICO
# =============================================================================
print("\nClassificações encontradas:")
print(df_final['classificacao'].unique())

# print(f"\nTotal de registros: {len(df_final)}")
# print("\nÚltimas linhas:")
# print(df_final.loc[df_final['idFazenda']==34].tail())

C:\Users\analy\AppData\Local\Temp\ipykernel_54632\1745262583.py:68: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period('M')



Classificações encontradas:
['Benfeitorias' 'Máquinas e equipamentos']


## Investimento

In [53]:
# Parcelas Sharepoint
df_parcela_shp = extrair_parcelas_em_lotes(supabase, tamanho_lote=10000)
df_parcela_excel = extrair_parcelas_em_lotes_excel(supabase, tamanho_lote=10000)
df_parcela = pd.concat([df_parcela_shp, df_parcela_excel])
colunas_investimento = ['ID', 'idInventario', 'Item', 'nomeTela', 'valor', 'dataPagamento']
print(f"Total de parcelas extraídas: {len(df_parcela)}")
df_parcela = df_parcela[colunas_investimento]
mascara = (
    ((df_parcela['nomeTela'] == 'Patrimônio') | 
     (df_parcela['Item'].isin(['Compra de terras', 'Compra de animais'])))
    & (df_parcela['dataPagamento'].notna()) &
    (df_parcela['idInventario'].notna()) 
)
df_parcela = df_parcela.loc[mascara].reset_index(drop=True)
# Converter para datetime
df_parcela['dataPagamento'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_parcela['dataPagamento'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)

# Criar a nova coluna com o primeiro dia do mês
df_parcela['mesReferencia'] = df_parcela['dataPagamento'].apply(lambda x: x.replace(day=1))

# Separar entre Patrimônio
df_parcela_pat = df_parcela.loc[df_parcela['nomeTela'] =='Patrimônio']
# Juntar idFazenda 
df_parcela_pat = df_parcela_pat.merge(df_patrimonio[['ID', 'idFazenda', 'classificacao']], left_on='idInventario', right_on='ID', how='left')

# Separar entre custos gerais
df_parcela_custo = df_parcela.loc[df_parcela['nomeTela'] !='Patrimônio']
# Juntar idFazenda 
df_parcela_custo = df_parcela_custo.merge(df_componentescustos[['ID', 'idFazenda']], left_on='idInventario', right_on='ID', how='left')

# Juntar novamente as tabelas
df_investimento = pd.concat([df_parcela_pat, df_parcela_custo])

# Juntar com IGPDI 
df_investimento = df_investimento.merge(df_igpdi[['data','deflator']], left_on='mesReferencia', right_on='data', how='left')

# Deflacionar
df_investimento = deflacionar_colunas(df_investimento, ['valor']) 

# Onde classificação for NaN, trocar para Item 
df_investimento['classificacao'] = df_investimento['classificacao'].fillna(df_investimento['Item'])

df_investimento_agg = df_investimento.pivot_table(
    index=['idFazenda', 'mesReferencia'],
    columns='classificacao',
    values='def_valor',
    aggfunc='sum',
    fill_value=0
).reset_index()

df_investimento_agg['investimento'] = df_investimento_agg[['Benfeitorias', 'Compra de animais',
       'Compra de terras', 'Máquinas e equipamentos']].sum(axis=1)

df_investimento_agg = df_investimento_agg.rename(columns={
    'Benfeitorias':'investimentoBenfeitorias',
    'Compra de animais':'investimentoAnimais',
    'Compra de terras':'investimentoTerras',
    'Máquinas e equipamentos':'investimentoMaquinas'
})

#df_investimento_agg.loc[df_investimento_agg['idFazenda']==88].head()

Extraídas 10000 linhas...
Extraídas 20000 linhas...
Extraídas 30000 linhas...
Extraídas 40000 linhas...
Extraídas 50000 linhas...
Extraídas 60000 linhas...
Extraídas 70000 linhas...
Extraídas 80000 linhas...
Extraídas 90000 linhas...
Extraídas 100000 linhas...
Extraídas 110000 linhas...
Extraídas 120000 linhas...
Extraídas 130000 linhas...
Extraídas 140000 linhas...
Extraídas 150000 linhas...
Extraídas 160000 linhas...
Extraídas 170000 linhas...
Extraídas 180000 linhas...
Extraídas 190000 linhas...
Extraídas 200000 linhas...
Extraídas 210000 linhas...
Extraídas 214746 linhas...
Extraídas 10000 linhas...
Extraídas 20000 linhas...
Extraídas 30000 linhas...
Extraídas 40000 linhas...
Extraídas 50000 linhas...
Extraídas 53671 linhas...
Total de parcelas extraídas: 268417


## Revisão Anual

In [ ]:
%%time
import pandas as pd
import numpy as np
from datetime import datetime
import gc

# Parâmetros iniciais
ini = pd.to_datetime('1994-01-01')
fim = pd.Timestamp(datetime.today().replace(day=1))

# ======== 1. Tabela de Parcelas ==========================

# Preparar dados de parcelas
# Selecione parcelas
try:
    # Assumindo que 'extrair_parcelas_em_lotes' e 'supabase' estão definidos no ambiente
    df_parcela = extrair_parcelas_em_lotes(supabase, tamanho_lote=10000)
    # Colunas a serem usadas no EC
    colunas_revisao_anual = ['ID', 'idInventario', 'numeroParcela', 'valor', 'previsaoPagamento', 'dataPagamento', 'nomeTela']
    # Filtrar colunas do dataframe
    df_parcela = df_parcela[colunas_revisao_anual]
    if df_parcela.empty:
        print('Nenhum registro encontrado na extração inicial do Supabase.')
        df_parcela = pd.DataFrame() # Garante que df_parcela seja um DataFrame vazio para evitar erros posteriores
    else:
        print(f"Foram encontrados {len(df_parcela)} registros na extração completa das parcelas.")

        # Aplicar a filtragem de nomeTela e idInventario.notna() no DataFrame
        mascara = (
            (df_parcela['nomeTela'] == 'Patrimônio') & 
            (df_parcela['idInventario'].notna())
        )
        df_parcela = df_parcela.loc[mascara].reset_index(drop=True)

        if not df_parcela.empty:
            print(f"Após filtragem, foram encontrados {len(df_parcela)} registros válidos.")
        else:
            print('Nenhum registro válido encontrado após filtragem.')

except Exception as e:
    print(f"Ocorreu um erro ao preparar df_parcela: {e}")
    df_parcela = pd.DataFrame() # Garante que df_parcela seja um DataFrame vazio em caso de erro

# Se df_parcela estiver vazio, não há necessidade de continuar
if df_parcela.empty:
    print("DataFrame de parcelas vazio. Encerrando processamento.")
    # Você pode adicionar um 'sys.exit()' ou 'break' aqui se quiser parar o script
    # import sys; sys.exit() # Exemplo para parar o script
else:
    # ======== 2. Inserir Valor Aquisicao nas Parcelas ============

    # Renomear a coluna ID para não gerar ID_x e ID_y
    df_parcela = df_parcela.rename(columns={'ID':'idParcela'})

    # Mesclar com informações do inventário (df_inventario)
    # df_inventario deve conter 'ID', 'idFazenda', 'dataAquisicao', 'classificacao',
    # 'quantidade', 'valorAquisicao', 'novoValor', 'dataRevisao', 'dataFim'.
    # A coluna 'quantidade' é necessária para calcular o valor total do bem.
    df_parcela = (
        df_parcela
        .merge(
            df_inventario[['ID', 'idFazenda', 'dataAquisicao', 'classificacao', 'quantidade', 'valorAquisicao', 'novoValor', 'dataRevisao', 'dataFim']],
            left_on= 'idInventario',
            right_on = 'ID',
            how='left'
        )
    )

    # Retirar parcelas sem dono (idFazenda) que podem ter surgido de merges sem correspondência
    df_parcela = df_parcela.loc[df_parcela['idFazenda'].notna()]

    # ======= 3. Gerar proporção de cada parcela ==================
    # Criar valor de Compra (valor total do bem na aquisição)
    df_parcela['valorCompra'] = df_parcela['quantidade'] * df_parcela['valorAquisicao']

    # Criar proporção da parcela -- Quanto a parcela paga do item?
    df_parcela['proporcaoParcela'] = df_parcela['valor']/df_parcela['valorCompra']

    # Transformar datas em Datetime
    datas_patrimonio = ['dataAquisicao', 'dataRevisao', 'dataPagamento', 'previsaoPagamento', 'dataFim']
    for col in datas_patrimonio:
        df_parcela[col] = (
            pd.to_datetime(
                pd.to_datetime(
                    df_parcela[col],
                    errors='coerce'
                ).dt.date,
                format='%Y-%m-%d'
            )
        )

    # Criar nova tabela (df_ec_parcela)
    df_ec_parcela = df_parcela.copy()
    # Coluna de primeiro dia do mês para servir como referência mensal
    df_ec_parcela['mesReferencia'] = df_ec_parcela['dataPagamento'].dt.to_period('M').dt.start_time
    # Conforme memória do usuário, mesReferencia deve ser YYYY-MM-DD sem horário
    df_ec_parcela['mesReferencia'] = df_ec_parcela['mesReferencia'].dt.strftime('%Y-%m-%d')
    df_ec_parcela['mesReferencia'] = pd.to_datetime(df_ec_parcela['mesReferencia']) # Converte de volta para datetime para operações

    # Calcular acumulado de proporção
    df_parcelas_acum = (
        df_ec_parcela
        .sort_values(['idInventario', 'mesReferencia'])
        .groupby(['idInventario', 'mesReferencia'])['proporcaoParcela']
        .sum()
        .groupby('idInventario')
        .cumsum()
        .reset_index(name='proporcaoParcelaAcumulado')
    )

    # =========== 4. Gerar meses para cada idInventario =========
    # 1. OBTER PRIMEIRA DATA DE PAGAMENTO POR BEM - DIRETO
    primeira_data_pagamento = (
        df_ec_parcela
        .groupby('idInventario')['mesReferencia']
        .min()
        .reset_index()
        .rename(columns={'mesReferencia': 'primeiroPagamento'})
    )

    # 2. PREPARAR BASE PATRIMÔNIO - OTIMIZADO
    df_patrimonio_base = (
        df_inventario
        .merge(primeira_data_pagamento, left_on='ID', right_on='idInventario', how='left')
        .drop_duplicates(subset=['ID'])
        [['ID', 'idFazenda', 'classificacao', 'dataAquisicao', 'dataFim', 'primeiroPagamento', 'dataRevisao', 'valorAquisicao', 'novoValor', 'quantidade']]
    )

    # Lidar com NaT em 'primeiroPagamento' e 'dataFim' antes de usar
    df_patrimonio_base['primeiroPagamento_clean'] = df_patrimonio_base['primeiroPagamento'].fillna(ini)
    df_patrimonio_base['dataFim_clean'] = df_patrimonio_base['dataFim'].fillna(fim)

    # Criação dos ranges de datas
    all_periods = []
    for idx, row in df_patrimonio_base.iterrows():
        data_inicio = max(row['primeiroPagamento_clean'], ini)
        data_fim = min(row['dataFim_clean'], fim)
        if data_inicio <= data_fim: # Garante que o range é válido
            periods = pd.date_range(start=data_inicio, end=data_fim, freq='MS')
            for p in periods:
                all_periods.append({'ID': row['ID'], 'mesAtivo': p})

    df_base_temporal = pd.DataFrame(all_periods)

    # Mesclar de volta as informações do inventário
    df_base_temporal = df_base_temporal.merge(
        df_patrimonio_base.drop(columns=['primeiroPagamento_clean', 'dataFim_clean', 'primeiroPagamento', 'dataFim']),
        on='ID',
        how='left'
    )

    # Definir valor unitário atualizado do bem
    df_base_temporal['valorUnitarioAtualizado'] = np.where(
        (df_base_temporal['dataRevisao'].notna()) & (df_base_temporal['mesAtivo'] >= df_base_temporal['dataRevisao']),
        df_base_temporal['novoValor'],
        df_base_temporal['valorAquisicao']
    )

    # Calcular valor total do patrimônio (quantidade * valor unitário)
    df_base_temporal['valorPatrimonio'] = df_base_temporal['quantidade'] * df_base_temporal['valorUnitarioAtualizado']
    df_base_temporal.drop('quantidade', axis=1, inplace=True) # Remover 'quantidade' após uso

    # Trazer a proporção da parcela
    df_ec_mensal = df_base_temporal.merge(
        df_parcelas_acum,
        left_on=['ID', 'mesAtivo'],
        right_on=['idInventario', 'mesReferencia'],
        how='left'
    )

    # --- INÍCIO DO NOVO BLOCO DE CORREÇÃO MONETÁRIA DO PATRIMÔNIO ---

    # 1. Determinar a data base para o índice (dataAquisicao ou dataRevisao)
    # Esta lógica é idêntica à usada na depreciação para alinhar a correção.
    condicao_revisao_ativa_ec = (
        (df_ec_mensal['dataRevisao'].notna()) & 
        (df_ec_mensal['mesAtivo'] >= df_ec_mensal['dataRevisao'])
    )

    df_ec_mensal['data_base_indice'] = pd.Series(np.where(
        condicao_revisao_ativa_ec,
        df_ec_mensal['dataRevisao'],
        df_ec_mensal['dataAquisicao']
    )).astype('datetime64[ns]').dt.to_period('M').dt.to_timestamp()

    # 2. Merge para obter o IGP-DI da Época (igpdi_epoca)
    # Traz o índice do mês corrente (mesAtivo) para o cálculo do fator.
    # Assumimos que df_igpdi possui as colunas 'data' e 'igpdi'.
    df_ec_mensal = df_ec_mensal.merge(
        df_igpdi[['data', 'igpdi']],
        left_on='mesAtivo',
        right_on='data',
        how='left'
    ).rename(columns={'igpdi': 'igpdi_epoca'}).drop(columns=['data'])

    # 3. Merge para obter o IGP-DI da Base (igpdi_aquisicao)
    # Traz o índice da data de aquisição/revisão (data_base_indice) para o denominador do fator.
    # Assumimos que df_igpdi possui as colunas 'data' e 'igpdi'.
    df_ec_mensal = df_ec_mensal.merge(
        df_igpdi[['data', 'igpdi']],
        left_on='data_base_indice',
        right_on='data',
        how='left'
    ).rename(columns={'igpdi': 'igpdi_aquisicao'}).drop(columns=['data'])

    # 4. Calcular o Fator de Correção e o Valor do Patrimônio Corrigido
    df_ec_mensal['fator_correcao'] = df_ec_mensal['igpdi_epoca'] / df_ec_mensal['igpdi_aquisicao']

    # Tratar NaNs no fator de correção (ex: IGP-DI faltando para alguma data)
    # Assumir fator 1.0 (sem correção) se faltar índice para evitar divisões por zero ou NaNs.
    df_ec_mensal['fator_correcao'] = df_ec_mensal['fator_correcao'].ffill() 

    # Aplicar a correção monetária ao valor do patrimônio
    # O valorPatrimonio já é o valor atualizado (aquisição ou novoValor) * quantidade.
    # Agora, ele é corrigido monetariamente pela inflação acumulada.
    df_ec_mensal['valorPatrimonioCorrigido'] = df_ec_mensal['valorPatrimonio'] * df_ec_mensal['fator_correcao']

    # --- FIM DO NOVO BLOCO DE CORREÇÃO MONETÁRIA DO PATRIMÔNIO ---

    # Ordenar dataframe e preencher proporção
    df_ec_mensal = df_ec_mensal.sort_values(['ID','mesAtivo'])
    # Aplica ffill apenas na coluna 'proporcaoParcelaAcumulado'
    df_ec_mensal['proporcaoParcelaAcumulado'] = df_ec_mensal['proporcaoParcelaAcumulado'].ffill() 
    # Em seguida, preenche quaisquer NaN restantes (se houver) com 0
    df_ec_mensal['proporcaoParcelaAcumulado'] = df_ec_mensal['proporcaoParcelaAcumulado'].fillna(0)

    # Criar coluna de ECParcela
    # Agora usa 'valorPatrimonioCorrigido' em vez de 'def_valorPatrimonio'
    df_ec_mensal['ECParcela'] = df_ec_mensal['proporcaoParcelaAcumulado'] * df_ec_mensal['valorPatrimonioCorrigido']

    # # 6. AGREGAR POR FAZENDA E MÊS - SIMPLIFICADO
    print("Agregando por fazenda e mês...")

    # Agrupar e somar ECParcela
    df_ec_final = (
        df_ec_mensal
        .groupby(['idFazenda', 'mesAtivo', 'classificacao'])['ECParcela']
        .sum()
        .reset_index(name='EC_Mensal_Bruto')
    )
    
    # Aplicar a divisão por 24
    df_ec_final['EC_Mensal'] = df_ec_final['EC_Mensal_Bruto'] / 24
    df_ec_final.drop(columns=['EC_Mensal_Bruto'], inplace=True)

    # Corrigir nomes desconfigurados na coluna 'classificacao'
    # Use .loc para atribuir de volta à coluna específica
    df_ec_final.loc[df_ec_final['classificacao'] == 'MÃ¡quinas e equipamentos', 'classificacao'] = 'Máquinas e equipamentos'


    # Pivotar para ter as classificações como colunas
    df_ec_final = (
        df_ec_final
        .pivot_table(
            index=['idFazenda', 'mesAtivo'],
            columns='classificacao',
            values='EC_Mensal',
            aggfunc='sum',
            fill_value=0
        )
        .reset_index()
    )

    # Flatten columns e renomear de uma vez
    df_ec_final.columns.name = None
    rename_mapping = {
        'Benfeitorias': 'EC_Benfeitorias',
        'Máquinas e equipamentos': 'EC_Maquinas' # Corrigido o nome aqui também para o pivot
    }

    # Renomear colunas existentes
    for old_col, new_col in rename_mapping.items():
        if old_col in df_ec_final.columns:
            df_ec_final = df_ec_final.rename(columns={old_col: new_col})

    # Garantir colunas essenciais existem
    for col in ['EC_Benfeitorias', 'EC_Maquinas']:
        if col not in df_ec_final.columns:
            df_ec_final[col] = 0

    # Criar EC Médio
    df_ec_final['EC_MaquinasBenfeitorias_Medio'] = df_ec_final[['EC_Benfeitorias', 'EC_Maquinas']].sum(axis=1)


    # 7. RESUMO FINAL - CONSOLIDADO
    print(f"""
=== RESUMO DO PROCESSAMENTO ===
Período: {ini.strftime('%Y-%m')} a {fim.strftime('%Y-%m')}
Bens processados: {len(df_base_temporal['ID'].unique()):,}
Meses gerados: {len(df_ec_final):,}
Fazendas: {df_ec_final['idFazenda'].nunique()}
Total EC: R$ {df_ec_final['EC_MaquinasBenfeitorias_Medio'].sum():,.2f}
Shape final: {df_ec_final.shape}
""")

    display(df_ec_final.loc[(df_ec_final['mesAtivo']>=datetime(2025,1,1)) & (df_ec_final['idFazenda']==252)])


Extraídas 10000 linhas...
Extraídas 20000 linhas...
Extraídas 30000 linhas...
Extraídas 40000 linhas...
Extraídas 50000 linhas...
Extraídas 60000 linhas...
Extraídas 70000 linhas...
Extraídas 80000 linhas...
Extraídas 90000 linhas...
Extraídas 100000 linhas...
Extraídas 110000 linhas...
Extraídas 120000 linhas...
Extraídas 130000 linhas...
Extraídas 140000 linhas...
Extraídas 150000 linhas...
Extraídas 160000 linhas...
Extraídas 170000 linhas...
Extraídas 180000 linhas...
Extraídas 190000 linhas...
Extraídas 200000 linhas...
Extraídas 210000 linhas...
Extraídas 214746 linhas...
Foram encontrados 214746 registros na extração completa das parcelas.
Após filtragem, foram encontrados 28963 registros válidos.
Agregando por fazenda e mês...

=== RESUMO DO PROCESSAMENTO ===
Período: 1994-01 a 2026-07
Bens processados: 19,360
Meses gerados: 235,969
Fazendas: 943
Total EC: R$ 9,516,147,512.43
Shape final: (235969, 5)



,idFazenda,mesAtivo,EC_Benfeitorias,EC_Maquinas,EC_MaquinasBenfeitorias_Medio
62067,252,2025-01-01,144957.50,119833.08,264790.58
62068,252,2025-02-01,146380.58,121187.47,267568.05
62069,252,2025-03-01,145662.36,120737.47,266399.83
62070,252,2025-04-01,146087.06,121251.89,267338.95
62071,252,2025-05-01,144867.48,120375.72,265243.20
62072,252,2025-06-01,142300.28,118203.15,260503.43
62073,252,2025-07-01,142208.58,118125.54,260334.12
62074,252,2025-08-01,142493.55,118366.71,260860.27
62075,252,2025-09-01,142990.58,118787.33,261777.91
62076,252,2025-10-01,142946.65,118750.16,261696.81


CPU times: total: 1min 33s
Wall time: 1min 39s


## Estoque de Capital de Animais

In [55]:
df_estoque_animais = df_rebanho.copy()
# Mesclar com IGPDI
df_estoque_animais = (
    df_estoque_animais
    .merge(df_igpdi[['data','deflator']],
           left_on='mesReferencia',
           right_on='data',
           how='left')
)
col_deflacionar = ['valorunitVacasEmLactacao', 'valorunitVacasSecas', 'valorunitAnimaisAleitamento', 'valorunitAnimaisRecria', 'valorunitMacho', 'valorunitOutrasCategorias']
# Deflacionar preços
df_estoque_animais = deflacionar_colunas(df_estoque_animais, col_deflacionar)

# Valor das Vacas em Lactação
df_estoque_animais['valorVacasLactacao'] = df_estoque_animais['qtdeVacasEmLactacao'] * df_estoque_animais['def_valorunitVacasEmLactacao']
# Valors das Vacas Secas
df_estoque_animais['valorVacasSecas'] = df_estoque_animais['qtdeVacasSecas'] * df_estoque_animais['def_valorunitVacasSecas']
# Valor das vacas em aleitamento
df_estoque_animais['valorAleitamento'] = df_estoque_animais['qtdeAnimaisAleitamento'] * df_estoque_animais['def_valorunitAnimaisAleitamento']
# Valor dos animais em Recria
df_estoque_animais['valorRecria'] = df_estoque_animais['qtdeAnimaisRecria'] * df_estoque_animais['def_valorunitAnimaisRecria']
# Valor dos Machos
df_estoque_animais['valorMachos'] = df_estoque_animais['qtdeMacho'] * df_estoque_animais['def_valorunitMacho']
# Valor de Outras Categorias
df_estoque_animais['valorOutrasCategorias'] = df_estoque_animais['qtdeOutrasCategorias'] * df_estoque_animais['def_valorunitOutrasCategorias']/2
# Colunas para estoque de capital
coluna_estoquecapital = ['valorVacasLactacao', 'valorVacasSecas', 'valorAleitamento', 'valorRecria', 'valorMachos', 'valorOutrasCategorias']
# Estoque de capital de animais
df_estoque_animais['estoqueCapitalAnimais'] = df_estoque_animais.loc[:, coluna_estoquecapital].sum(axis=1)
# Reduzir dataframe
# df_estoque_animais = df_estoque_animais[['idFazenda','mesReferencia','estoqueCapitalAnimais']]
df_estoque_animais['estoqueCapitalAnimais_Mensal'] = df_estoque_animais['estoqueCapitalAnimais'] / 12
# df_estoque_animais.loc[df_estoque_animais['idFazenda']==99]

## Estoque de Capital em Terra

In [56]:
# Trazer Área
df_estoque_area = df_area_ativa[['idFazenda','mesReferencia','def_estoqueTerra','def_precoTerraNua']].copy()

df_estoque_area['def_estoqueTerra_Mensal'] = df_estoque_area['def_estoqueTerra'] / 12

# df_estoque_area.loc[df_estoque_area['idFazenda']==99]

## Itens Cultura

In [57]:
# Itens Cultura
tab_itenscultura = (
    supabase.table('tab_itenscultura')
    .select("*")
    .execute()
)
# Importar df itens cultura
df_itenscultura = pd.DataFrame(tab_itenscultura.data)
# df_itenscultura.head()

In [58]:
# Copiar custo de forrageira
df_forrageira_naoanual = df_custoforrageira.loc[
    df_custoforrageira['idCultura'].notna() &
    df_custoforrageira['idCultura'] != 0 &
    df_custoforrageira['idArea'].notna()].copy()
df_forrageira_naoanual = df_forrageira_naoanual.loc[df_forrageira_naoanual['etapa']=='Plantio']
# Trazer Itens Cultura
df_forrageira_naoanual = (
    df_forrageira_naoanual
    .merge(df_itenscultura,
           left_on='culturasPlantadas',
           right_on='cultura',
           how='left')
)
# Filtrar apenas forrageiras não anuais
df_forrageira_naoanual = df_forrageira_naoanual.loc[df_forrageira_naoanual['tipoForrageira']=='Não anual']
# Criar coluna de custoTotal
df_forrageira_naoanual['custoPlantio'] = df_forrageira_naoanual['qtdeConsumida'] * df_forrageira_naoanual['def_valorUnitario']
# Depreciação
df_forrageira_naoanual['depreciacaoPlantioMensal'] = df_forrageira_naoanual['custoPlantio'] / (df_forrageira_naoanual['vidaUtil'] * 12)
# Criar estoque de capital
df_forrageira_naoanual['estoqueCapitalPlantio'] = df_forrageira_naoanual['custoPlantio'] / 24
# Criar colunas para reduzir o dataframe
manter_colunas = ['idFazenda', 'priDiaMes', 'estoqueCapitalPlantio', 'depreciacaoPlantioMensal']
# Agregar por idFazenda e mês
df_forrageira_naoanual = df_forrageira_naoanual[manter_colunas].groupby(['idFazenda','priDiaMes']).sum().reset_index()
# Descobrir menor e maior data para cada fazenda
min_dates = df_forrageira_naoanual.groupby('idFazenda')['priDiaMes'].min()
max_dates = df_forrageira_naoanual.groupby('idFazenda')['priDiaMes'].max()

# Criar todos os meses para cada fazenda
linhas = []
for fazenda in df_forrageira_naoanual['idFazenda'].unique():
    start = min_dates.loc[fazenda]
    end = max_dates.loc[fazenda]
    meses = pd.date_range(start, end, freq='MS')
    for mes in meses:
        linhas.append({'idFazenda': fazenda, 'priDiaMes': mes})

base = pd.DataFrame(linhas)
# Junta os lançamentos à base, um mês para cada fazenda, sem ainda preencher valores
base = base.merge(df_forrageira_naoanual, on=['idFazenda', 'priDiaMes'], how='left')

# Ordena para garantir
base = base.sort_values(['idFazenda', 'priDiaMes']).reset_index(drop=True)
def acumulador(sub):
    # Acumular “eventos” (novos lançamentos) mês a mês
    sub = sub.copy()
    sub['estoqueCapitalPlantio_acumulado'] = sub['estoqueCapitalPlantio'].fillna(0).cumsum()
    sub['depreciacaoPlantioMensal_acumulado'] = sub['depreciacaoPlantioMensal'].fillna(0).cumsum()
    return sub

df_forrageira_naoanual = base.groupby('idFazenda', group_keys=False).apply(acumulador, include_groups=True).reset_index(drop=True)

# df_forrageira_naoanual.head()

C:\Users\analy\AppData\Local\Temp\ipykernel_54632\3429669377.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sub['depreciacaoPlantioMensal_acumulado'] = sub['depreciacaoPlantioMensal'].fillna(0).cumsum()
C:\Users\analy\AppData\Local\Temp\ipykernel_54632\3429669377.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sub['depreciacaoPlantioMensal_acumulado'] = sub['depreciacaoPlantioMensal'].fillna(0).cumsum()
C:\Users\analy\AppData\Local\Temp\ipykernel_54632\3429669377.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill 

## Culturas

In [59]:
# Cultura
tab_cultura = (
    supabase.table('tab_culturas')
    .select('ID','idFazenda','idArea','identificacaoCultura','idEspecificacaoCultura','especificacaoCultura',
            'areaha','Plantio','Excluido','finalizado')
    .eq('Excluido',0)
    .execute()
)
# Dataframe 
df_cultura = pd.DataFrame(tab_cultura.data)
df_cultura = (
    df_cultura
    .merge(df_itenscultura,
           left_on='especificacaoCultura',
           right_on='cultura',
           how='left')
)
# Transformar mesReferencia em Datetime
df_cultura['Plantio'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_cultura['Plantio'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)
# df_cultura.head()

## Área de Forrageira não anuais

In [60]:
# DEFINIÇÕES PRÉVIAS
ini = pd.to_datetime('2022-01-01')
fim = pd.Timestamp(datetime.today().replace(day=1))

def gerar_periodos_forrageira(row):
    data_inicio = max(row['Plantio'], ini)
    data_fim = min(row['dataFim'] if pd.notna(row['dataFim']) else fim, fim)
    return pd.date_range(start=data_inicio, end=data_fim, freq='MS')

# PASSOS
df_area_forrageira = df_cultura[['idFazenda','idArea','especificacaoCultura','Plantio','areaha','tipoForrageira','vidaUtil']].copy()
df_area_forrageira = df_area_forrageira.loc[df_area_forrageira['tipoForrageira']=='Não anual']
df_area_forrageira = (
    df_area_forrageira
    .merge(df_area, left_on='idArea', right_on='ID', how='left')
)

df_area_forrageira.dtypes
df_area_forrageira_expand = (
    df_area_forrageira
    .assign(mesReferencia=df_area_forrageira.apply(gerar_periodos_forrageira, axis=1))
    .explode('mesReferencia')
    .dropna(subset=['mesReferencia'])
    .reset_index(drop=True)
)
df_area_forrageira_expand = df_area_forrageira_expand[
    (df_area_forrageira_expand['mesReferencia'] >= ini) & 
    (df_area_forrageira_expand['mesReferencia'] <= fim)
]
# Renomar coluna de fazenda
df_area_forrageira_expand = df_area_forrageira_expand.rename(columns={'idFazenda_x':'idFazenda'})
# Reduzir o dataframe 
df_area_forrageira_expand = df_area_forrageira_expand[['idFazenda','mesReferencia','areaha']]
# Agrupar somando as áreas 
df_area_forrageira_expand = df_area_forrageira_expand.groupby(['idFazenda','mesReferencia']).sum().reset_index().sort_values(['idFazenda', 'mesReferencia'])
# df_area_forrageira_expand.head()

## Estoque de Capital Final (✅)

1. Revisão Anual (✅)
2. Animais (✅)
3. Forrageiras Não Anuais (✅)
4. Terra (✅)

In [61]:

# Para fazer a combinação entre idFazenda e meses, prefiro, primeiro ter a lista dos idFazendas de todos os estoques de capital
id_maquinas      = pd.Index(df_ec_final['idFazenda'].unique())
id_benfeitorias  = pd.Index(df_ec_final['idFazenda'].unique())
id_animais       = pd.Index(df_estoque_animais['idFazenda'].unique())
id_terra         = pd.Index(df_estoque_area['idFazenda'].unique())
id_forrageira    = pd.Index(df_forrageira_naoanual['idFazenda'].unique())
# Separa os valores únicos de datas
todos_idfazendas = id_maquinas.union(id_benfeitorias).union(id_animais).union(id_terra).union(id_forrageira)
# Coleta todas as datas de interesse
datas_maquinas     = pd.Index(df_ec_final['mesAtivo'].unique())
datas_benfeitorias = pd.Index(df_ec_final['mesAtivo'].unique())
datas_animais      = pd.Index(df_estoque_animais['mesReferencia'].unique())
datas_terra        = pd.Index(df_estoque_area['mesReferencia'].unique())
datas_forrageira   = pd.Index(df_forrageira_naoanual['priDiaMes'].unique())
# Une as datas em um único vetor
todas_datas = datas_maquinas.union(datas_benfeitorias)\
    .union(datas_animais)\
    .union(datas_terra)\
    .union(datas_forrageira)\
    .sort_values()

# Gera o produto cartesiano
df_base = pd.MultiIndex.from_product(
    [todos_idfazendas, todas_datas],
    names=['idFazenda', 'mesReferencia']
).to_frame(index=False)
# Criar tabela de Estoque de Capital
df_list = [
    df_estoquecapital_maquinas.rename(columns={'mesAtivo': 'mesReferencia', 'def_depreciacao':'def_depreciacaoMaquinas'})[['idFazenda', 'mesReferencia', 'def_depreciacaoMaquinas']],
    df_estoquecapital_benfeitorias.rename(columns={'mesAtivo': 'mesReferencia', 'def_depreciacao':'def_depreciacaoBenfeitorias'})[['idFazenda', 'mesReferencia', 'def_depreciacaoBenfeitorias']],
    df_estoque_animais[['idFazenda', 'mesReferencia', 'estoqueCapitalAnimais_Mensal']],
    df_estoque_area[['idFazenda', 'mesReferencia', 'def_estoqueTerra_Mensal','def_precoTerraNua']],
    df_forrageira_naoanual.rename(columns={'priDiaMes': 'mesReferencia'}), 
    df_ec_final.rename(columns={'mesAtivo':'mesReferencia', 'EC_Maquinas':'def_estoqueCapitalMaquinas', 'EC_Benfeitorias':'def_estoqueCapitalBenfeitorias'})[['idFazenda', 'mesReferencia', 'def_estoqueCapitalMaquinas','def_estoqueCapitalBenfeitorias']]
]
# Concatena todas as tabelas
df_estoquecapital = pd.concat(df_list, ignore_index=True).sort_values(['idFazenda','mesReferencia']).reset_index(drop=True)
# Agrupar as linhaspor meio de soma
df_estoquecapital = df_estoquecapital.groupby(['idFazenda','mesReferencia']).sum().reset_index()
# Cria Estoque de Capital sem terra
df_estoquecapital['estoqueCapital_semTerra'] = df_estoquecapital[['def_estoqueCapitalBenfeitorias', 'def_estoqueCapitalMaquinas', 'estoqueCapitalAnimais_Mensal', 'estoqueCapitalPlantio']].sum(axis=1)
# Cria Estoque de Capital com Terra
df_estoquecapital['estoqueCapital_comTerra'] = df_estoquecapital[['def_estoqueCapitalBenfeitorias', 'def_estoqueCapitalMaquinas', 'estoqueCapitalAnimais_Mensal', 'estoqueCapitalPlantio', 'def_estoqueTerra_Mensal']].sum(axis=1)
df_estoquecapital['depreciacaoEstoqueCapital'] = df_estoquecapital[['def_depreciacaoMaquinas','def_depreciacaoBenfeitorias','depreciacaoPlantioMensal']].sum(axis=1)

# Retirar tabelas de ETL 
del df_estoque_area, df_estoquecapital_benfeitorias, df_estoquecapital_maquinas, df_estoque_animais, df_base, df_list
gc.collect() 

df_estoquecapital.groupby(['idFazenda','mesReferencia']).sum().reset_index()

,idFazenda,mesReferencia,def_depreciacaoMaquinas,def_depreciacaoBenfeitorias,estoqueCapitalAnimais_Mensal,def_estoqueTerra_Mensal,def_precoTerraNua,estoqueCapitalPlantio,depreciacaoPlantioMensal,estoqueCapitalPlantio_acumulado,depreciacaoPlantioMensal_acumulado,def_estoqueCapitalMaquinas,def_estoqueCapitalBenfeitorias,estoqueCapital_semTerra,estoqueCapital_comTerra,depreciacaoEstoqueCapital
0,1,2001-01-01,189.58,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,1958.33,0.00,1958.33,1958.33,189.58
1,1,2001-02-01,190.23,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,1964.96,0.00,1964.96,1964.96,190.23
2,1,2001-03-01,191.75,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,1980.75,0.00,1980.75,1980.75,191.75
3,1,2001-04-01,193.92,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,2003.08,0.00,2003.08,2003.08,193.92
4,1,2001-05-01,194.77,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,2011.89,0.00,2011.89,2011.89,194.77
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242843,1216,2026-03-01,450.53,611.81,0.00,0.00,0.00,0.00,0,0.00,0.00,3379.01,9177.10,12556.11,12556.11,1062.34
242844,1216,2026-04-01,461.41,626.58,0.00,0.00,0.00,0.00,0,0.00,0.00,3460.58,9398.66,12859.24,12859.24,1087.99
242845,1216,2026-05-01,465.43,632.04,0.00,0.00,0.00,0.00,0,0.00,0.00,3490.74,9480.56,12971.30,12971.30,1097.47
242846,1216,2026-06-01,461.77,627.07,0.00,0.00,0.00,0.00,0,0.00,0.00,3463.28,9405.99,12869.27,12869.27,1088.84


# Cálculo de Indicadores

## Indicadores Mensais

In [ ]:
# Usar tabela de leite como base para os indicadores mensais
df_ind_mensais = df_leite[['idFazenda', 'mesReferencia', 'CCS', 'CPP', 'Gordura', 'Proteina', 'leiteProduzido', 'leiteDiario', 'def_precoLeite']].reset_index(drop=True).copy()
# Incluir as colunas necessárias de MDO
df_ind_mensais = df_ind_mensais.merge(df_mdo[['idFazenda','mesReferencia','MDOContratadaDiaria','MDOTotalDiaria']], on=['idFazenda','mesReferencia'], how='left')
# Incluir as colunas de Rebanho
df_ind_mensais = df_ind_mensais.merge(df_rebanho[['idFazenda', 'mesReferencia', 'qtdeVacasEmLactacao', 'totalVacas', 'totalAnimais', 'vacasLactacao_totalVacas', 'vacasLactacao_totalAnimais']],
                                      on=['idFazenda', 'mesReferencia'],
                                      how='left')


# Alterar os 0s para NaN e evitar valores Infinitos
df_ind_mensais = df_ind_mensais.replace([np.inf, -np.inf, 0], np.nan)
# Criar coluna de Produção/VL
df_ind_mensais['producaoVacasLactacao'] = df_ind_mensais['leiteDiario'] / df_ind_mensais['qtdeVacasEmLactacao']

# Criar coluna de Produção/MDOContratada
df_ind_mensais['producaoMDOTotal'] = df_ind_mensais['leiteDiario'] / df_ind_mensais['MDOTotalDiaria']
# Criar coluna de VL/MDOTotal
df_ind_mensais['vacasLactacaoMDOTotal'] = df_ind_mensais['qtdeVacasEmLactacao'] / df_ind_mensais['MDOTotalDiaria']
# Inserir custo com concentrado, volumoso, alimentação e mdo
df_ind_mensais = df_ind_mensais.merge(df_componentescustos[['idFazenda', 'mesReferencia', 'custoAlimentacao_Concentrado_Minerais', 'custoAlimentacao_Volumoso', 'custoMDOContratada']], on=['idFazenda', 'mesReferencia'], how='left')
# Custo com Alimentação (R$/litro)
df_ind_mensais['custoAlimentacao_RSLitro'] = (df_ind_mensais['custoAlimentacao_Concentrado_Minerais'] + df_ind_mensais['custoAlimentacao_Volumoso']) / df_ind_mensais['leiteProduzido']
# Custo com Concentrado, Volumoso e MDO Contratada (R$/litro)
coluna_rslitro = ['custoAlimentacao_Concentrado_Minerais', 'custoAlimentacao_Volumoso', 'custoMDOContratada']

for col in coluna_rslitro:
    nome_col = col+'_RSLitro'
    df_ind_mensais[nome_col] = df_ind_mensais[col] / df_ind_mensais['leiteProduzido']
# Criar proporção do custo com alimentação por preço do leite
df_ind_mensais['alimentacao_precoLeite'] = df_ind_mensais['custoAlimentacao_RSLitro'] / df_ind_mensais['def_precoLeite']
# Inserir o estoque de capital total
df_ind_mensais = df_ind_mensais.merge(df_estoquecapital[['idFazenda','mesReferencia','estoqueCapital_comTerra','def_precoTerraNua']], on=['idFazenda','mesReferencia'], how='left')
df_ind_mensais['estoqueCapitalcomTerra_leiteDiario'] = df_ind_mensais['estoqueCapital_comTerra'] / df_ind_mensais['leiteDiario']
# Definir regras de consistencia mensais
regras_consistencia_mensais = (
    (df_ind_mensais['CCS'] > 50) & (df_ind_mensais['CPP'] > 1) & (df_ind_mensais['Gordura'] > 2.5) & (df_ind_mensais['Gordura'] < 5.5) & (df_ind_mensais['Proteina'] > 2.4) & (df_ind_mensais['Proteina'] < 4.5) &
    (df_ind_mensais['vacasLactacao_totalVacas'] > 0.2) & (df_ind_mensais['vacasLactacao_totalVacas'] < 0.99) & (df_ind_mensais['vacasLactacao_totalAnimais'] > 0.15) & (df_ind_mensais['vacasLactacao_totalAnimais'] < 0.99) &
    (df_ind_mensais['producaoVacasLactacao'] > 3) & (df_ind_mensais['producaoVacasLactacao'] < 45) & (df_ind_mensais['producaoMDOTotal'] > 20) & (df_ind_mensais['producaoMDOTotal'] < 1500) & (df_ind_mensais['vacasLactacaoMDOTotal'] < 70) &
    (df_ind_mensais['alimentacao_precoLeite'] > 0.15) & (df_ind_mensais['alimentacao_precoLeite'] < 1.5) & (df_ind_mensais['custoAlimentacao_Volumoso_RSLitro'] < 3) & 
    (df_ind_mensais['custoAlimentacao_Concentrado_Minerais_RSLitro'] > 0.3) & (df_ind_mensais['custoAlimentacao_Concentrado_Minerais_RSLitro'] < 3.5)  & (df_ind_mensais['custoMDOContratada_RSLitro'] < 1) & (df_ind_mensais['estoqueCapitalcomTerra_leiteDiario'] < 30000)
)
df_ind_mensais['Consistencia'] = np.where(regras_consistencia_mensais,'Consistente','Inconsistente')
df_ind_mensais['idConsistencia'] = np.where(regras_consistencia_mensais,0,1)
# Ordenar indicadores mensais 
df_ind_mensais = df_ind_mensais.sort_values(['idFazenda','mesReferencia']).reset_index(drop=True)
# Criar indicadores de lançamento
total_linhas = df_ind_mensais.shape[0]
linhas_consistentes = df_ind_mensais.loc[df_ind_mensais['Consistencia']=='Consistente'].shape[0]
proporcao_consistencia = linhas_consistentes/total_linhas
print(f"Total de linhas: {total_linhas}")
print(f"Total de fazendas: {df_ind_mensais.idFazenda.nunique()}")
print(f"Meses e fazendas consistentes: {linhas_consistentes}")
print(f"Proporção de consistência: {proporcao_consistencia:.2%}")

# Criar colunas individuais para cada critério
print("Criando indicadores individuais de consistência...")

# Critérios de qualidade do leite
df_ind_mensais['cons_CCS'] = (df_ind_mensais['CCS'] > 50)
df_ind_mensais['cons_CPP'] = (df_ind_mensais['CPP'] > 1)
df_ind_mensais['cons_Gordura'] = (df_ind_mensais['Gordura'] > 2.5) & (df_ind_mensais['Gordura'] < 5.5)
df_ind_mensais['cons_Proteina'] = (df_ind_mensais['Proteina'] > 2.4) & (df_ind_mensais['Proteina'] < 4.5)

# Critérios de rebanho
df_ind_mensais['cons_vacasLactacao_totalVacas'] = (df_ind_mensais['vacasLactacao_totalVacas'] > 0.2) & (df_ind_mensais['vacasLactacao_totalVacas'] < 0.99)
df_ind_mensais['cons_vacasLactacao_totalAnimais'] = (df_ind_mensais['vacasLactacao_totalAnimais'] > 0.15) & (df_ind_mensais['vacasLactacao_totalAnimais'] < 0.99)

# Critérios de produtividade
df_ind_mensais['cons_producaoVacasLactacao'] = (df_ind_mensais['producaoVacasLactacao'] > 3) & (df_ind_mensais['producaoVacasLactacao'] < 45)
df_ind_mensais['cons_producaoMDOTotal'] = (df_ind_mensais['producaoMDOTotal'] > 20) & (df_ind_mensais['producaoMDOTotal'] < 1500)
df_ind_mensais['cons_vacasLactacaoMDOTotal'] = (df_ind_mensais['vacasLactacaoMDOTotal'] < 70)

# Critérios de custos
df_ind_mensais['cons_alimentacao_precoLeite'] = (df_ind_mensais['alimentacao_precoLeite'] > 0.15) & (df_ind_mensais['alimentacao_precoLeite'] < 1.5)
df_ind_mensais['cons_custoVolumoso'] = (df_ind_mensais['custoAlimentacao_Volumoso_RSLitro'] < 3)
df_ind_mensais['cons_custoConcentrado'] = (df_ind_mensais['custoAlimentacao_Concentrado_Minerais_RSLitro'] > 0.3) & (df_ind_mensais['custoAlimentacao_Concentrado_Minerais_RSLitro'] < 3.5)
df_ind_mensais['cons_custoMDO'] = (df_ind_mensais['custoMDOContratada_RSLitro'] < 1)

# Critério de capital
df_ind_mensais['cons_estoqueCapital'] = (df_ind_mensais['estoqueCapitalcomTerra_leiteDiario'] < 30000)

# Contar quantos critérios cada linha viola
colunas_criterios = [col for col in df_ind_mensais.columns if col.startswith('cons_')]
df_ind_mensais['total_criterios_ok'] = df_ind_mensais[colunas_criterios].sum(axis=1)
df_ind_mensais['total_criterios_violados'] = len(colunas_criterios) - df_ind_mensais['total_criterios_ok']

# Criar lista dos critérios violados
def listar_criterios_violados(row):
    violados = []
    if not row['cons_CCS']: violados.append('CCS')
    if not row['cons_CPP']: violados.append('CPP')
    if not row['cons_Gordura']: violados.append('Gordura')
    if not row['cons_Proteina']: violados.append('Proteina')
    if not row['cons_vacasLactacao_totalVacas']: violados.append('VL/TotalVacas')
    if not row['cons_vacasLactacao_totalAnimais']: violados.append('VL/TotalAnimais')
    if not row['cons_producaoVacasLactacao']: violados.append('Prod/VL')
    if not row['cons_producaoMDOTotal']: violados.append('Prod/MDO')
    if not row['cons_vacasLactacaoMDOTotal']: violados.append('VL/MDOTotal')
    if not row['cons_alimentacao_precoLeite']: violados.append('Alim/Preço')
    if not row['cons_custoVolumoso']: violados.append('CustoVolumoso')
    if not row['cons_custoConcentrado']: violados.append('CustoConcentrado')
    if not row['cons_custoMDO']: violados.append('CustoMDO')
    if not row['cons_estoqueCapital']: violados.append('EstoqueCapital')
    
    return '; '.join(violados) if violados else 'Nenhum'

df_ind_mensais['criterios_violados'] = df_ind_mensais.apply(listar_criterios_violados, axis=1)

# ==== INÍCIO DA CRIAÇÃO DE COLUNA COM DADOS ===
# Trazer os dados das outras tabelas que compõe Com dados
'''
Novas colunas importantes: 
Quantidade de Energia: qtdeEnergia, qtdeEnergiaSolar, qtdeAlcoolgasolina, qtdeDiesel
Custo de Energia: def_custoEnergia def_custoCombustivel
Despesas: [
    'def_gastoSucedaneo',
    'def_gastoMaterialOrdenha',
    'def_gastoReproducao',
    'def_gastoHormonios',
    'def_gastoMedicamentosVacinas',
    'def_gastoAssistenciaTecnica',
    'def_gastoImpostoTaxas',
    'def_gastoArrendamento',
    'def_gastoReparosConsertos',
    'def_gastoAdministrativo',
    'def_gastoAcessoriosDespesasGerais',
    'def_gastoCamaAreia'
]
'''
# Colunas importantes para Energia
colunas_energia = ['mesReferencia', 'idFazenda', 'qtdeEnergia', 'qtdeEnergiaSolar', 'qtdeAlcoolgasolina', 'qtdeDiesel', 'def_custoEnergia', 'def_custoCombustivel']
# Colunas importantes para Despesas
colunas_despesas = [
    'mesReferencia',
    'idFazenda',
    'def_gastoSucedaneo',
    'def_gastoMaterialOrdenha',
    'def_gastoReproducao',
    'def_gastoHormonios',
    'def_gastoMedicamentosVacinas',
    'def_gastoAssistenciaTecnica',
    'def_gastoImpostoTaxas',
    'def_gastoArrendamento',
    'def_gastoReparosConsertos',
    'def_gastoAdministrativo',
    'def_gastoAcessoriosDespesasGerais',
    'def_gastoCamaAreia'
]
# Incluir as colunas de Energia
df_ind_mensais = df_ind_mensais.merge(df_energiacombustivel[colunas_energia],
                                      on=['idFazenda', 'mesReferencia'],
                                      how='left')
# Incluir as colunas de Despesas
df_ind_mensais = df_ind_mensais.merge(df_componentescustos[colunas_despesas],
                                      on=['idFazenda', 'mesReferencia'],
                                      how='left')
# Definir colunas
colunas_com_dados = [
    'CCS', 'CPP', 'Gordura', 'Proteina',
    'leiteProduzido', 'def_precoLeite',
    'MDOTotalDiaria', 
    'qtdeVacasEmLactacao',
    'estoqueCapital_comTerra',
    'def_precoTerraNua'
]
# Criar coluna de custo com alimentação
df_ind_mensais['custo_alimentacao'] = df_ind_mensais['custoAlimentacao_Concentrado_Minerais'].fillna(0) + \
                                           df_ind_mensais['custoAlimentacao_Volumoso'].fillna(0)
# Criar coluna de qtdEnerigaCombustivel
df_ind_mensais['qtd_energia_combustivel'] = df_ind_mensais[['qtdeEnergia', 'qtdeEnergiaSolar', 'qtdeAlcoolgasolina', 'qtdeDiesel']].sum(axis=1)

# Criar coluna de custoEnergiaCombustivel
df_ind_mensais['custo_energia_combustivel'] = df_ind_mensais[['def_custoEnergia', 'def_custoCombustivel']].sum(axis=1)

# Criar coluna de Despesas
df_ind_mensais['despesas'] = (
    df_ind_mensais[[
    'def_gastoSucedaneo',
    'def_gastoMaterialOrdenha',
    'def_gastoReproducao',
    'def_gastoHormonios',
    'def_gastoMedicamentosVacinas',
    'def_gastoAssistenciaTecnica',
    'def_gastoImpostoTaxas',
    'def_gastoArrendamento',
    'def_gastoReparosConsertos',
    'def_gastoAdministrativo',
    'def_gastoAcessoriosDespesasGerais',
    'def_gastoCamaAreia'
    ]]
    .sum(axis=1)
)
# Criar a condição
condicao_com_dados = (
    df_ind_mensais[colunas_com_dados].notna().all(axis=1) &
    (df_ind_mensais['custo_alimentacao'].notna()) &
    (df_ind_mensais['custo_alimentacao'] > 0) &
    (df_ind_mensais['qtd_energia_combustivel'] > 0) &
    (df_ind_mensais['custo_energia_combustivel'] > 0) &
    (df_ind_mensais['despesas'] >0)
)

# Aplicar condição
df_ind_mensais['status_dados'] = np.where(condicao_com_dados, 'Com dados', 'Sem dados')
# Dropar as colunas desnecessárias
df_ind_mensais = (
    df_ind_mensais
    .drop(
        [
            'qtdeEnergia', 'qtdeEnergiaSolar', 'qtdeAlcoolgasolina', 'qtdeDiesel',
            'def_custoEnergia', 'def_custoCombustivel',
            'def_gastoSucedaneo', 'def_gastoMaterialOrdenha', 'def_gastoReproducao',
            'def_gastoHormonios', 'def_gastoMedicamentosVacinas', 'def_gastoAssistenciaTecnica',
            'def_gastoImpostoTaxas', 'def_gastoArrendamento', 'def_gastoReparosConsertos', 
            'def_gastoAdministrativo', 'def_gastoAcessoriosDespesasGerais', 'def_gastoCamaAreia'
        ],
        axis=1
    )
)
            
# Verificar resultados
print("\n=== RESUMO DOS CRITÉRIOS ===")
print(f"Total de critérios avaliados: {len(colunas_criterios)}")
print(f"Distribuição de critérios violados:")
print(df_ind_mensais['total_criterios_violados'].value_counts().sort_index())

print(f"\nCritérios mais violados:")
for col in colunas_criterios:
    nome_criterio = col.replace('cons_', '')
    violacoes = (~df_ind_mensais[col]).sum()
    total_valido = df_ind_mensais[col].notna().sum()
    if total_valido > 0:
        perc = violacoes / total_valido * 100
        print(f"{nome_criterio}: {violacoes}/{total_valido} ({perc:.1f}%)")

# Mostrar exemplo de linhas inconsistentes
print(f"\nExemplo de linhas inconsistentes:")
inconsistentes = df_ind_mensais[df_ind_mensais['Consistencia'] == 'Inconsistente']
if len(inconsistentes) > 0:
    print(inconsistentes[['idFazenda', 'mesReferencia', 'total_criterios_violados', 'criterios_violados']].head())

# Criar arquivo em Excel de indicadores mensais
nome_arquivo_mensais = 'indicadores_mensais.xlsx'
# Inserir nome do produtor e do consultor no Ind Mensais
df_ind_mensais_export = df_ind_mensais.merge(d_fazenda[['idFazenda', 'codAgroindustria', 'nomeFazenda',  'nomeprodutor', 'regiaoLeiteira', 'nomeagroindustria', 'nomeconsultor']], 
                            on='idFazenda', 
                            how='left').copy()
df_ind_mensais_export = df_ind_mensais_export.loc[df_ind_mensais_export['nomeconsultor'].notna()]
df_ind_mensais_export.reset_index(drop=True, inplace=True)
# Ver amostra 
print(f"Número de linhas: {df_ind_mensais_export.shape[0]}")

df_ind_mensais_export['fazenda-produtor'] = df_ind_mensais_export['nomeFazenda'] + ' - ' + df_ind_mensais_export['nomeprodutor']
    
df_ind_mensais_export.to_excel(nome_arquivo_mensais, index=False)
print(f"Arquivo {nome_arquivo_mensais} criado.")

df_ind_mensais_export.head()

Total de linhas: 20444
Total de fazendas: 1048
Meses e fazendas consistentes: 11058
Proporção de consistência: 54.09%
Criando indicadores individuais de consistência...

=== RESUMO DOS CRITÉRIOS ===
Total de critérios avaliados: 14
Distribuição de critérios violados:
total_criterios_violados
0     11070
1       651
2       819
3       507
4      5084
5       506
6       542
7       222
8       554
9       191
10      165
11       47
12       70
13       66
14       77
Name: count, dtype: int64

Critérios mais violados:
CCS: 6415/20571 (31.2%)
CPP: 6392/20571 (31.1%)
Gordura: 6418/20571 (31.2%)
Proteina: 6400/20571 (31.1%)
vacasLactacao_totalVacas: 1496/20571 (7.3%)
vacasLactacao_totalAnimais: 1430/20571 (7.0%)
producaoVacasLactacao: 1705/20571 (8.3%)
producaoMDOTotal: 1695/20571 (8.2%)
vacasLactacaoMDOTotal: 2525/20571 (12.3%)
alimentacao_precoLeite: 2614/20571 (12.7%)
custoVolumoso: 1530/20571 (7.4%)
custoConcentrado: 2566/20571 (12.5%)
custoMDO: 1032/20571 (5.0%)
estoqueCapital: 358/

,idFazenda,mesReferencia,CCS,CPP,Gordura,Proteina,leiteProduzido,leiteDiario,def_precoLeite,MDOContratadaDiaria,...,custo_energia_combustivel,despesas,status_dados,codAgroindustria,nomeFazenda,nomeprodutor,regiaoLeiteira,nomeagroindustria,nomeconsultor,fazenda-produtor
0,2.00,2022-05-01,NaN,NaN,NaN,NaN,68682.00,2257.79,2.75,5.92,...,15408.25,23957.31,Sem dados,LR05021,BARREIRO,GABRIEL DE CASTRO ALVES SAVASSI,Patos de Minas - 9188,Nestlé,João Paulo Alves Mendonça,BARREIRO - GABRIEL DE CASTRO ALVES SAVASSI
1,2.00,2022-05-01,NaN,NaN,NaN,NaN,68682.00,2257.79,2.75,5.92,...,15408.25,23957.31,Sem dados,LR05021,BARREIRO,GABRIEL DE CASTRO ALVES SAVASSI,Patos de Minas - 9188,Nestlé,João Vitor Carneiro Maciel Melo Quintão,BARREIRO - GABRIEL DE CASTRO ALVES SAVASSI
2,2.00,2022-06-01,NaN,NaN,NaN,NaN,81278.00,2671.86,3.29,5.92,...,15384.27,20778.27,Sem dados,LR05021,BARREIRO,GABRIEL DE CASTRO ALVES SAVASSI,Patos de Minas - 9188,Nestlé,João Paulo Alves Mendonça,BARREIRO - GABRIEL DE CASTRO ALVES SAVASSI
3,2.00,2022-06-01,NaN,NaN,NaN,NaN,81278.00,2671.86,3.29,5.92,...,15384.27,20778.27,Sem dados,LR05021,BARREIRO,GABRIEL DE CASTRO ALVES SAVASSI,Patos de Minas - 9188,Nestlé,João Vitor Carneiro Maciel Melo Quintão,BARREIRO - GABRIEL DE CASTRO ALVES SAVASSI
4,2.00,2022-07-01,NaN,NaN,NaN,NaN,90096.00,2961.74,3.50,5.92,...,15267.44,21655.27,Sem dados,LR05021,BARREIRO,GABRIEL DE CASTRO ALVES SAVASSI,Patos de Minas - 9188,Nestlé,João Paulo Alves Mendonça,BARREIRO - GABRIEL DE CASTRO ALVES SAVASSI


## Gerar Arquivo de Fazendas Consistentes para BI de Gestor

In [63]:
df_gestor_consistente = (
    df_ind_mensais[['idFazenda','Consistencia','mesReferencia']]
    .merge(d_fazenda[['idFazenda','nomeprodutor','codAgroindustria']],
           on='idFazenda').iloc[:,[0,-1,-2,1,2]]
)
df_gestor_consistente['mesReferencia'] = df_gestor_consistente['mesReferencia'].dt.strftime('%d/%m/%Y')
df_gestor_consistente.columns = ['ID da Fazenda', 'Código LR', 'Produtor', 'Consistência da Fazenda', 'Mês/Ano']
nome_arquivo_gestor = 'C:\\Users\\analy\\LABOR RURAL\\Analytics - Departamento Analytics\\POWER_BI\\PROJETOS\\ELABORE\\GESTOR\\BD_CONSISTENTES_GESTOR.xlsx'

try:
    os.remove(nome_arquivo_gestor)
    print("Planilha antiga excluída.")
    df_gestor_consistente.to_excel(nome_arquivo_gestor, index=False)
    print("BI Gestor atualizado")

except:
    print("Planilha de consistência não exportada")
df_gestor_consistente.head()

Planilha antiga excluída.
BI Gestor atualizado


,ID da Fazenda,Código LR,Produtor,Consistência da Fazenda,Mês/Ano
0,1.00,LR05024,GERALDO EUSTAQUIO MARQUES,Inconsistente,01/05/2022
1,1.00,LR05024,GERALDO EUSTAQUIO MARQUES,Inconsistente,01/06/2022
2,1.00,LR05024,GERALDO EUSTAQUIO MARQUES,Inconsistente,01/07/2022
3,1.00,LR05024,GERALDO EUSTAQUIO MARQUES,Inconsistente,01/08/2022
4,1.00,LR05024,GERALDO EUSTAQUIO MARQUES,Inconsistente,01/09/2022


## Gerar Upsert de Fazendas Consistentes para BI LED

In [64]:
# Criar dataframe consistente
df_consistente = (
    df_ind_mensais[['idFazenda','Consistencia','mesReferencia', 'status_dados']].copy()
    .merge(d_fazenda[['idFazenda','codAgroindustria']],
           on='idFazenda').iloc[:,[0,-1,-2,1,2]]
)
# Criar coluna de referencia led
df_consistente['mesReferencia'] = pd.to_datetime(df_consistente['mesReferencia'])
df_consistente['mes_led'] = df_consistente['mesReferencia'] + pd.DateOffset(months=1)
# Renomear colunas
colunas_consistente_supabase = ['idfazenda', 'codigo_lr', 'status_code', 'consistencia_mensal', 'mes_elabore', 'mes_referencia']
df_consistente.columns = colunas_consistente_supabase

# Remover duplicatas
df_consistente.drop_duplicates(subset=['idfazenda', 'mes_referencia'], keep='last', inplace=True)
print(f"Após remover duplicatas, {len(df_consistente)} registros únicos para upsert.")

# Para garantir, podemos formatar explicitamente:
df_consistente['mes_elabore'] = df_consistente['mes_elabore'].dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')
df_consistente['mes_referencia'] = df_consistente['mes_referencia'].dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')

data_to_upsert = df_consistente.to_dict(orient='records')

# Nome da tabela
TABLE_NAME = 'tab_consistencia_mensal'
# --- 3. Realizar o Upsert em Chunks ---
print(f"Iniciando processo de UPSERT para a tabela '{TABLE_NAME}' no Supabase.")

if data_to_upsert:
    print(f"   - Processando {len(data_to_upsert)} registros para upsert...")

    chunk_size = 1000 # Tamanho do chunk, ajuste conforme a necessidade e limites da API
    for i in range(0, len(data_to_upsert), chunk_size):
        chunk = data_to_upsert[i:i + chunk_size]
        try:
            # O método .upsert() é usado aqui. Ele identificará conflitos
            # com base na chave primária (idFazenda, mes_referencia) e atualizará ou inserirá.
            response = supabase.table(TABLE_NAME) \
                .upsert(chunk) \
                .execute()

            if response.data:
                print(f"     - Upserted {len(response.data)} registros em chunk {i//chunk_size + 1}.")
            else:
                print(f"     - Nenhum registro upserted para chunk {i//chunk_size + 1} ou erro na resposta.")
                # Para depuração, você pode querer imprimir o erro da resposta
                # if response.error:
                #     print(f"       Erro Supabase: {response.error}")

        except Exception as e:
            print(f"❌ Erro ao realizar upsert no Supabase (chunk {i//chunk_size + 1}): {str(e)}")
            # Para depuração, você pode querer imprimir mais detalhes do erro
            # print(f"Detalhes do erro: {e.args}")

    print(f"   - UPSERTs concluídos. Total de registros processados: {len(data_to_upsert)}")


print("\nProcesso de upsert concluído.")
print(df_consistente.dtypes)
df_consistente.head()

Após remover duplicatas, 15908 registros únicos para upsert.
Iniciando processo de UPSERT para a tabela 'tab_consistencia_mensal' no Supabase.
   - Processando 15908 registros para upsert...
❌ Erro ao realizar upsert no Supabase (chunk 1): {'code': '22P02', 'details': None, 'hint': None, 'message': 'invalid input syntax for type integer: "1.0"'}
❌ Erro ao realizar upsert no Supabase (chunk 2): {'code': '22P02', 'details': None, 'hint': None, 'message': 'invalid input syntax for type integer: "39.0"'}
❌ Erro ao realizar upsert no Supabase (chunk 3): {'code': '22P02', 'details': None, 'hint': None, 'message': 'invalid input syntax for type integer: "63.0"'}
❌ Erro ao realizar upsert no Supabase (chunk 4): {'code': '22P02', 'details': None, 'hint': None, 'message': 'invalid input syntax for type integer: "82.0"'}
❌ Erro ao realizar upsert no Supabase (chunk 5): {'code': '22P02', 'details': None, 'hint': None, 'message': 'invalid input syntax for type integer: "112.0"'}
❌ Erro ao realizar 

,idfazenda,codigo_lr,status_code,consistencia_mensal,mes_elabore,mes_referencia
0,1.00,LR05024,Sem dados,Inconsistente,2022-05-01T00:00:00.000000Z,2022-06-01T00:00:00.000000Z
1,1.00,LR05024,Sem dados,Inconsistente,2022-06-01T00:00:00.000000Z,2022-07-01T00:00:00.000000Z
2,1.00,LR05024,Sem dados,Inconsistente,2022-07-01T00:00:00.000000Z,2022-08-01T00:00:00.000000Z
3,1.00,LR05024,Sem dados,Inconsistente,2022-08-01T00:00:00.000000Z,2022-09-01T00:00:00.000000Z
4,1.00,LR05024,Sem dados,Inconsistente,2022-09-01T00:00:00.000000Z,2022-10-01T00:00:00.000000Z


## Sistema de Produção

In [65]:
# Sistema de Produção
tab_sistema = (
    supabase.table('tab_sistemaproducao')
    .select('ID','idFazenda','dataInicio','dataFim','percentualVacasCompostBarn','percentualVacasFreeStall','percentualVacasSemConfinado','percentualVacasPasto','percentualVacasConfinadoSemEstrutura','Excluido')
    .eq('Excluido', 0)
    .execute()
)
# Definir dataframe
df_sistema = pd.DataFrame(tab_sistema.data)
# Demonstra tipos das colunas
print(f"Tipo de colunas antes do ELT:\n{df_sistema.dtypes}")
# Transformar em data
df_sistema['dataInicio'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_sistema['dataInicio'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)
df_sistema['dataFim'] = (
    pd.to_datetime(
        pd.to_datetime(
            df_sistema['dataFim'],
            errors='coerce'
        )
        .dt
        .date,
        format='%Y-%m-%d'
    )
)
colunas_numericas = ['percentualVacasConfinadoSemEstrutura']
# Transformar em numérico
df_sistema = tratar_colunas_numericas(df_sistema, colunas_numericas)
#Função para mapear coluna percentual para nome do sistema
def sistema(row):
    col_map = {
        'percentualVacasCompostBarn': 'Compost Barn',
        'percentualVacasFreeStall': 'Free Stall',
        'percentualVacasSemConfinado': 'Semi Confinado',
        'percentualVacasPasto': 'Vacas a Pasto',
        'percentualVacasConfinadoSemEstrutura': 'Confinado sem Estrutura'
    }
    for col, nome in col_map.items():
        if row[col] == 100:
            return nome
    # Se nenhuma for 100%, retorna maior percentual
    max_col = max(col_map, key=lambda c: row[c])
    return col_map[max_col]
df_sistema['sistemaProducao'] = df_sistema.apply(sistema, axis=1)

# Defina o "hoje"
hoje = pd.Timestamp.today().replace(day=1)  # primeiro dia do mês atual

def expandir_periodo(row):
    inicio = row['dataInicio']
    fim = row['dataFim'] if pd.notna(row['dataFim']) else hoje
    inicio_mes = inicio.replace(day=1)
    meses = pd.date_range(inicio_mes, fim, freq='MS')  # MS = month start
    return pd.DataFrame({
        'idFazenda': row['idFazenda'],
        'mesReferencia': meses,
        'sistema': row['sistemaProducao'],
        'dataInicio': row['dataInicio'],
        'dataFim': row['dataFim'],
        'dias_inicio': row['dataInicio'].day if inicio.year == meses[0].year and inicio.month == meses[0].month else 1,
        'dias_fim': row['dataFim'].day if pd.notna(row['dataFim']) and fim.year == meses[-1].year and fim.month == meses[-1].month else pd.NaT,
    })
# Aplicar a explosão dos períodos
periodos = pd.concat([expandir_periodo(row) for idx, row in df_sistema.iterrows()], ignore_index=True)

def dias_no_mes(dt):
    # Número de dias do mês
    next_month = (dt + pd.DateOffset(months=1)).replace(day=1)
    return (next_month - dt).days

periodos['dias_no_mes'] = periodos['mesReferencia'].apply(dias_no_mes)

# Calcular dias efetivos em cada mês para cada registro:
def dias_sistema(row):
    if row['dataInicio'].year == row['mesReferencia'].year and row['dataInicio'].month == row['mesReferencia'].month:
        if pd.isna(row['dataFim']) or (row['dataFim'].year != row['mesReferencia'].year or row['dataFim'].month != row['mesReferencia'].month):
            return row['dias_no_mes'] - row['dataInicio'].day + 1
        else:
            return row['dataFim'].day - row['dataInicio'].day + 1
    elif pd.notna(row['dataFim']) and row['dataFim'].year == row['mesReferencia'].year and row['dataFim'].month == row['mesReferencia'].month:
        return row['dataFim'].day
    else:
        return row['dias_no_mes']

periodos['dias_sistema'] = periodos.apply(dias_sistema, axis=1)

# Soma total de dias por sistema, fazenda e mês:
agg = (periodos.groupby(['idFazenda', 'mesReferencia', 'sistema'])['dias_sistema']
               .sum()
               .reset_index())

# Para cada idFazenda e mesReferencia, pega o sistema com mais dias
idx = agg.groupby(['idFazenda','mesReferencia'])['dias_sistema'].idxmax()
df_sistema = agg.loc[idx].reset_index(drop=True)
# Demonstra tipos das colunas
print(f"Tipo de colunas depois do ELT:\n{df_sistema.dtypes}")
print(f"Número de linhas: {df_sistema.shape[0]}")
df_sistema.head()

Tipo de colunas antes do ELT:
ID                                       int64
idFazenda                                int64
dataInicio                              object
dataFim                                 object
percentualVacasCompostBarn               int64
percentualVacasFreeStall                 int64
percentualVacasSemConfinado              int64
percentualVacasPasto                     int64
percentualVacasConfinadoSemEstrutura     int64
Excluido                                 int64
dtype: object
Tipo de colunas depois do ELT:
idFazenda                 int64
mesReferencia    datetime64[ns]
sistema                  object
dias_sistema              int64
dtype: object
Número de linhas: 29175


,idFazenda,mesReferencia,sistema,dias_sistema
0,1,2025-03-01,Compost Barn,12
1,1,2025-04-01,Compost Barn,30
2,1,2025-05-01,Compost Barn,31
3,1,2025-06-01,Compost Barn,30
4,1,2025-07-01,Compost Barn,31


## Indicadores Mensais

In [66]:
# Criar lista do ID com UF das fazendas
uf_fazenda = d_fazenda[['idFazenda','codAgroindustria','ufFazenda','nomeagroindustria']].drop_duplicates()



# Selecionar colunas finais
colunas_mensais = [
    'idFazenda', 'codAgroindustria', 'ufFazenda', 'nomeagroindustria', 'sistema', 'mesReferencia', 'Consistencia',
    'CCS', 'CPP', 'Gordura', 'Proteina',
    'leiteProduzido', 'leiteDiario', 'def_precoLeite',
    'MDOContratadaDiaria', 'MDOTotalDiaria', 'qtdeVacasEmLactacao',
    'totalVacas', 'totalAnimais', 'vacasLactacao_totalVacas',
    'vacasLactacao_totalAnimais', 'producaoVacasLactacao',
    'custoAlimentacao_Concentrado_Minerais', 'custoAlimentacao_Volumoso',
    'custoMDOContratada', 'estoqueCapital_comTerra',
    'custo_energia_combustivel', 'despesas'
]

bd_ind_mensais = (
    df_ind_mensais
    .merge(
        uf_fazenda,
        on='idFazenda',
        how='left'
    )
    .merge(
        df_sistema[['idFazenda', 'mesReferencia', 'sistema']], 
        on=['idFazenda', 'mesReferencia'],
        how='left'
    )
    .loc[df_ind_mensais['leiteProduzido'].notna(), colunas_mensais] # Filtrar apenas as colunas que quero
    .sort_values(['idFazenda', 'mesReferencia'], ascending=True)
    .reset_index(drop=True)
    .copy()
)



# Rename columns
nome_colunas = [
    'ID Fazenda', 'Codigo LR', 'UF', 'Agroindústria', 'Sistema de Produção', 'Mês de Referência', 'Consistência',
    'CCS (células/mL)', 'CPP (UFC/mL)', 'Gordura (%)', 'Proteína (%)',
    'Leite Produzido (L)', 'Leite Diário (L)', 'Preço do Leite (R$/L)',
    'MDO Contratada (dias)', 'MDO Total (dias)', 'Vacas em Lactação',
    'Total de Vacas', 'Total de Animais', 'Taxa Vacas Lactação / Total Vacas (%)',
    'Taxa Vacas Lactação / Total Animais (%)', 'Produção por Vaca em Lactação (L/vaca/dia)',
    'Custo Alimentação - Concentrado e Minerais (R$)', 'Custo Alimentação - Volumoso (R$)',
    'Custo MDO Contratada (R$)', 'Estoque de Capital com Terra (R$)',
    'Custo Energia e Combustível (R$)', 'Outros Custos (R$)'
]

bd_ind_mensais.columns = nome_colunas

# Transformar NaN de UF em Não Informado
bd_ind_mensais.loc[bd_ind_mensais['UF'].isna(), 'UF'] = "Não Informado"

# Transformar NaN de UF em Não Informado
bd_ind_mensais.loc[bd_ind_mensais['Sistema de Produção'].isna(), 'Sistema de Produção'] = "Não Informado"

# Exportar
bd_ind_mensais.to_excel('indicadores_mensais_tratados.xlsx', index=False)

bd_ind_mensais.tail(20)

,ID Fazenda,Codigo LR,UF,Agroindústria,Sistema de Produção,Mês de Referência,Consistência,CCS (células/mL),CPP (UFC/mL),Gordura (%),...,Total de Animais,Taxa Vacas Lactação / Total Vacas (%),Taxa Vacas Lactação / Total Animais (%),Produção por Vaca em Lactação (L/vaca/dia),Custo Alimentação - Concentrado e Minerais (R$),Custo Alimentação - Volumoso (R$),Custo MDO Contratada (R$),Estoque de Capital com Terra (R$),Custo Energia e Combustível (R$),Outros Custos (R$)
20319,1194.00,LR07828,MG,Laticínios Porto Alegre,Não Informado,2026-02-01,Consistente,1180.00,264.00,3.74,...,310.00,0.61,0.25,17.29,64122.24,12699.01,23404.54,699917.86,10021.65,2534.62
20320,1194.00,LR07828,MG,Laticínios Porto Alegre,Não Informado,2026-03-01,Consistente,1332.00,22.00,3.80,...,296.00,0.60,0.25,18.98,65773.75,13580.47,23140.10,694374.63,13990.82,4527.98
20321,1194.00,LR07828,MG,Laticínios Porto Alegre,Não Informado,2026-04-01,Consistente,820.50,18.00,3.84,...,319.00,0.66,0.25,17.59,68388.80,14673.94,22594.61,689520.14,12394.66,1000.78
20322,1195.00,NaN,Não Informado,NaN,Confinado sem Estrutura,2026-02-01,Consistente,216.00,64.00,3.63,...,95.00,0.90,0.38,24.21,17072.14,9122.56,2591.64,212660.03,1313.44,5016.99
20323,1196.00,LR11799,MG,Alvoar,Semi Confinado,2026-02-01,Consistente,613.00,111.00,4.10,...,52.00,0.62,0.31,14.86,6965.28,4664.94,4208.78,338976.75,1431.00,5155.47
20324,1196.00,LR11799,MG,Alvoar,Semi Confinado,2026-03-01,Consistente,480.00,42.00,3.91,...,63.00,0.76,0.41,17.18,7476.95,7687.06,4172.25,336226.39,1457.36,5685.16
20325,1197.00,LR11715,MG,Alvoar,Semi Confinado,2026-02-01,Consistente,488.00,37.00,3.90,...,178.00,0.83,0.41,13.63,18025.34,0.00,8677.54,495482.49,7122.64,5761.60
20326,1197.00,LR11715,MG,Alvoar,Semi Confinado,2026-03-01,Inconsistente,506.00,25.00,3.99,...,178.00,0.77,0.38,15.99,NaN,NaN,0.00,491380.32,0.00,1340.91
20327,1198.00,LR11815,MG,Alvoar,Semi Confinado,2026-03-01,Inconsistente,264.00,82.00,3.63,...,126.00,0.77,0.29,16.27,5406.87,6918.35,5198.87,218743.22,132.52,2374.99
20328,1200.00,LR11890,MG,Alvoar,Não Informado,2026-01-01,Consistente,311.00,168.00,3.33,...,100.00,0.85,0.40,24.16,33265.05,9765.88,6400.30,402371.36,4164.65,8936.20


## Indicadores Anuais

In [67]:
def movel_leite(df, cols_media=None, cols_soma=None, janela=12, periodos=12):
    if cols_media is None: cols_media = []
    if cols_soma is None: cols_soma = []
    res = []
    for nome, grupo in df.groupby('idFazenda'):
        grupo = grupo.sort_values('mesReferencia').copy()
        # Rolling para médias
        for col in cols_media:
            grupo[f'{col}_mediaMovel'] = grupo[col].rolling(window=janela, min_periods=periodos).mean()
        # Rolling para somas
        for col in cols_soma:
            grupo[f'{col}_somaMovel'] = grupo[col].rolling(window=janela, min_periods=periodos).sum()
        grupo['num_meses_movel'] = grupo['mesReferencia'].rolling(window=janela, min_periods=periodos).count()
        # Calcula janela de datas para cada linha
        grupo['dt_ini_movel'] = grupo['mesReferencia'].shift(janela-1).dt.strftime('%b/%y')
        grupo['dt_fim_movel'] = grupo['mesReferencia'].dt.strftime('%b/%y')
        grupo['intervalo_movel'] = grupo['dt_ini_movel'] + '-' + grupo['dt_fim_movel']
        res.append(grupo)
    return pd.concat(res, ignore_index=True)


def movel_grupo(df, cols_media=None, cols_soma=None, janela=12, periodos=1):
    if cols_media is None: cols_media = []
    if cols_soma is None: cols_soma = []
    res = []
    for nome, grupo in df.groupby('idFazenda'):
        grupo = grupo.sort_values('mesReferencia').copy()
        # Rolling para médias
        for col in cols_media:
            grupo[f'{col}_mediaMovel'] = grupo[col].rolling(window=janela, min_periods=periodos).mean()
        # Rolling para somas
        for col in cols_soma:
            grupo[f'{col}_somaMovel'] = grupo[col].rolling(window=janela, min_periods=periodos).sum()
        res.append(grupo)
    return pd.concat(res, ignore_index=True)



In [68]:
%%time
# Cria a série de fazendas
idFazenda = pd.Index(d_fazenda['idFazenda'].unique())

# Primeira data da série 
start = pd.to_datetime('2021-01-01')
# Data atual para servir de fim da série
end = pd.Timestamp(datetime.today()).replace(day=1) 
# Série de meses
mesReferencia = pd.Index(pd.date_range(start=start, end=end, freq='MS')) 
# Dataframe com a combinação de Fazenda e Meses
df_base_anuais = pd.MultiIndex.from_product([idFazenda, mesReferencia], names=['idFazenda', 'mesReferencia']).to_frame(index=False)
# Mostrar número de linhas
print(f"Mostrar o número de linhas na base: {df_base_anuais.shape[0]}")
# Trazer leite mensal
df_base_anuais = df_base_anuais.merge(df_leite[['idFazenda','mesReferencia','leiteProduzido', 'def_precoLeite', 'absCCS', 'absCPP', 'absGordura','absProteina']],
                                      on=['idFazenda','mesReferencia'],
                                      how='left')
# Trazer Consistencia
df_base_anuais = df_base_anuais.merge(df_ind_mensais[['idFazenda','mesReferencia','idConsistencia']],
                                      on=['idFazenda','mesReferencia'],
                                      how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Leite: {df_base_anuais.shape[0]}")
# Trazer área "
df_base_anuais = df_base_anuais.merge(df_area_ativa, on=['idFazenda','mesReferencia'], how='left') # AREA ATIVA
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Áreas: {df_base_anuais.shape[0]}")
df_base_anuais = df_base_anuais.merge(df_leiteconsumido[['idFazenda','mesReferencia','consumoLeiteDescartado']], on=['idFazenda','mesReferencia'], how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Leite Consumido: {df_base_anuais.shape[0]}")
# Trazer os dados da Mão de Obra 
df_base_anuais = df_base_anuais.merge(df_mdo[['idFazenda','mesReferencia','MDOContratadaDiaria', 'MDOTotalDiaria', 'MDOFamiliarDiaria','def_familiarValortotal']], # MDO
                                            on=['idFazenda','mesReferencia'], how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após MDO: {df_base_anuais.shape[0]}")
# Trazer Renda Bruta 
df_base_anuais = df_base_anuais.merge(df_rendabruta[['idFazenda','mesReferencia','def_rendaLeite','def_rendaAtividade']],on=['idFazenda','mesReferencia'],how='left') 
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Renda: {df_base_anuais.shape[0]}")
# Trazer Estoque de Capital 
df_base_anuais = df_base_anuais.merge(df_estoquecapital[['idFazenda', 'mesReferencia', 'def_depreciacaoMaquinas', 'def_depreciacaoBenfeitorias',  'depreciacaoPlantioMensal', 'depreciacaoPlantioMensal_acumulado',  'depreciacaoEstoqueCapital', 
                                                         'estoqueCapitalAnimais_Mensal', 'def_estoqueTerra_Mensal', 'estoqueCapitalPlantio', 'estoqueCapitalPlantio_acumulado', 'def_estoqueCapitalMaquinas', 'def_estoqueCapitalBenfeitorias',
                                                         'estoqueCapital_semTerra', 'estoqueCapital_comTerra']],
                                      on=['idFazenda','mesReferencia'],how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Estoque de Capital: {df_base_anuais.shape[0]}")
# Trazer os dados de Forrageira 
df_base_anuais = df_base_anuais.merge(df_area_forrageira_expand[['idFazenda','mesReferencia', 'areaha']], # Área de Forrageira
                                            on=['idFazenda','mesReferencia'], how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Área de Forrageira: {df_base_anuais.shape[0]}")
# Colunas para trazer do componentes de custos 
colunas_manter = ['idFazenda', 'mesReferencia', 'def_gastoSucedaneo', 'def_gastoMaterialOrdenha', 'def_gastoReproducao', 'def_gastoHormonios',
                  'def_gastoMedicamentosVacinas', 'def_gastoAssistenciaTecnica', 'def_gastoImpostoTaxas', 'def_gastoArrendamento', 'def_gastoReparosConsertos',
                  'def_gastoAdministrativo', 'def_gastoAcessoriosDespesasGerais', 'def_gastoCamaAreia', 'def_compra_Animais', 'def_compra_Terras', 'def_gastoEmprestimosJurosPagos',
                  'gastoconsumoLeiteBezerro', 'gastoconsumoMaoObraFamiliar', 'custoAlimentacao_Concentrado_Minerais', 
                  'custoAlimentacao_Volumoso', 'def_custoEnergia', 'def_custoCombustivel', 'custoReceitasConcentrado', 'custoReceitasVolumoso', 'custoMDOContratada', 'coe']
# Trazer os dados de Forrageira 
df_base_anuais = df_base_anuais.merge(df_componentescustos[colunas_manter], 
                                            on=['idFazenda','mesReferencia'], how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Custos: {df_base_anuais.shape[0]}")
# Trazer os dados de Forrageira 
df_base_anuais = df_base_anuais.merge(df_custo_alimentacao[['idFazenda', 'mesReferencia', 'custoAlimentacao_Concentrado', 'custoAlimentacao_Minerais',
                                                            'quantidadeConsumida_Concentrado', 'quantidadeConsumida_Minerais', 'quantidadeConsumida_Volumoso']], 
                                            on=['idFazenda','mesReferencia'], how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Alimentação: {df_base_anuais.shape[0]}")

# Trazer tabela de Rebanho 
df_base_anuais = df_base_anuais.merge(df_rebanho[['idFazenda', 'mesReferencia', 'qtdeVacasEmLactacao', 'totalVacas', 'totalAnimais']], 
                                            on=['idFazenda','mesReferencia'], how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Rebanho: {df_base_anuais.shape[0]}")

# Trazer dados de investimento
df_base_anuais = df_base_anuais.merge(df_investimento_agg[['idFazenda', 'mesReferencia', 'investimentoBenfeitorias',
                                                           'investimentoMaquinas', 'investimentoAnimais', 'investimentoTerras','investimento']], 
                                            on=['idFazenda','mesReferencia'], how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Investimento: {df_base_anuais.shape[0]}")

# Custo da MDO Familiar = Custo da MDO Familiar + Leite Consumido da MDO Familiar 
df_base_anuais['def_familiarValortotal'] = df_base_anuais[['def_familiarValortotal', 'gastoconsumoMaoObraFamiliar']].sum(axis=1)

df_base_anuais['valorTerraNua'] = df_base_anuais['def_precoTerraNua'] * df_base_anuais['areaTotal']
df_base_anuais['areaTotalSoma'] = df_base_anuais['areaTotal']
df_base_anuais['areaPropriaSoma'] = df_base_anuais['areaPropria']


    # Remova duplicatas antes de aplicar as funções de média móvel
print(f"Número de linhas antes de remover duplicatas: {df_base_anuais.shape[0]}")
df_base_anuais.drop_duplicates(subset=['idFazenda', 'mesReferencia'], keep='first', inplace=True)
print(f"Número de linhas após remover duplicatas: {df_base_anuais.shape[0]}")


# Colunas a realizar a soma 
soma_movel = ['absCCS', 'absCPP', 'absGordura', 'absProteina', 'consumoLeiteDescartado', # Qualidade do Leite
              'valorTerraNua', 'areaTotalSoma','areaArrendada', 'areaPropriaSoma', # Área 
              'def_familiarValortotal', # MDO Familiar
              'def_rendaLeite', 'def_rendaAtividade', # Renda
              'def_depreciacaoBenfeitorias', 'def_depreciacaoMaquinas', 'depreciacaoPlantioMensal_acumulado',  'depreciacaoEstoqueCapital', # Depreciação
              'def_estoqueCapitalMaquinas', 'def_estoqueCapitalBenfeitorias','estoqueCapitalPlantio_acumulado', 'estoqueCapitalAnimais_Mensal', 'def_estoqueTerra_Mensal', 'estoqueCapital_semTerra', 'estoqueCapital_comTerra', # Estoque Capital
              'def_gastoSucedaneo', 'def_gastoMaterialOrdenha', 'def_gastoReproducao', 'def_gastoHormonios', 'def_gastoMedicamentosVacinas', 'def_gastoAssistenciaTecnica', # COE
              'def_gastoImpostoTaxas', 'def_gastoArrendamento', 'def_gastoReparosConsertos', 'def_gastoAdministrativo', 'def_gastoAcessoriosDespesasGerais', # COE
              'def_gastoCamaAreia', 'def_gastoEmprestimosJurosPagos', 'gastoconsumoLeiteBezerro', # COE
              'custoAlimentacao_Volumoso', 'def_custoEnergia', 'def_custoCombustivel', 'custoReceitasConcentrado', 'custoReceitasVolumoso', 'custoMDOContratada', 'coe',
              'custoAlimentacao_Concentrado', 'custoAlimentacao_Minerais', 'custoAlimentacao_Concentrado_Minerais', # Custo Alimentação
              'quantidadeConsumida_Concentrado', 'quantidadeConsumida_Minerais', 'quantidadeConsumida_Volumoso', # Quantidade Alimentação
              'investimentoBenfeitorias', 'investimentoMaquinas', 'investimentoAnimais', 'investimentoTerras','investimento', # Investimento
              'idConsistencia'] # Consistência
media_movel = ['areaArrendada', 'areaAtividadeSemReserva', 'areaAtividadeReserva', 'areaForrageira', 'areaPropria', 'areaTotal', 'areaha', # Área
               'MDOContratadaDiaria', 'MDOTotalDiaria', 'MDOFamiliarDiaria', # MDO
               'qtdeVacasEmLactacao', 'totalVacas', 'totalAnimais'] # Rebanho
# Combine as colunas das duas listas (evita duplicidade)
cols_to_fill = list(set(soma_movel + media_movel))

# Mantém só as colunas que existem no df_anuais
cols_to_fill = [col for col in cols_to_fill if col in df_base_anuais.columns]

# Criar df para indicadores anusis
df_anuais = df_base_anuais.copy() 
# Preenche NaN com 0 nessas colunas
df_anuais[cols_to_fill] = df_anuais[cols_to_fill].fillna(0)
# # Realizar a média móvel 
df_anuais = movel_leite(df_anuais, cols_media=None,  cols_soma=['leiteProduzido'], janela=12, periodos=12)
# # Realizar a média móvel 
df_anuais = movel_grupo(df_anuais, cols_media=media_movel,  cols_soma=soma_movel, janela=12, periodos=1)
# Adicionar sistema de produção
df_anuais = df_anuais.merge(df_sistema[['idFazenda', 'mesReferencia', 'sistema']], on=['idFazenda', 'mesReferencia'], how='left')
# Mostrar número de linhas
print(f"Mostrar o número de linhas após Média Móvel: {df_anuais.shape[0]}")
# Dropar NaN 
df_anuais = df_anuais.loc[df_anuais['leiteProduzido_somaMovel'].notna()]
# Mostrar número de linhas
print(f"Mostrar o número de linhas após retirar NaN: {df_anuais.shape[0]}")
# Inserir Cama em Acessórios
df_anuais['def_gastoAcessoriosDespesasGerais_somaMovel'] = df_anuais[['def_gastoAcessoriosDespesasGerais_somaMovel', 'def_gastoCamaAreia_somaMovel']].sum(axis=1)
# Inserir aleitamento como soma de leite de bezerro e Sucedâneo
df_anuais['def_aleitaento_somaMovel'] = df_anuais[['gastoconsumoLeiteBezerro_somaMovel', 'def_gastoSucedaneo_somaMovel']].sum(axis=1)
# Criar coluna de Preço da Terra dos Indicadores Anuais 
df_anuais['precoTerraNua_somaMovel'] = df_anuais['valorTerraNua_somaMovel'] / df_anuais['areaPropriaSoma_somaMovel']
# Criar coluna de Percentual da área arrendada dos Indicadores Anuais 
df_anuais['percentualAreaArrendada'] = (df_anuais['areaArrendada_somaMovel'] / df_anuais['areaTotalSoma_somaMovel']) * 100
# Criar VL/VACAS 
df_anuais['VL_TV'] = df_anuais['qtdeVacasEmLactacao_mediaMovel'] / df_anuais['totalVacas_mediaMovel']
# Criar VL/VACAS * 100 
df_anuais['VL_TV_100'] = df_anuais['VL_TV'] * 100
# Criar VL/ANIMAIS 
df_anuais['VL_TA'] = df_anuais['qtdeVacasEmLactacao_mediaMovel'] / df_anuais['totalAnimais_mediaMovel']
# Criar VL/ANIMAIS * 100
df_anuais['VL_TA_100'] = df_anuais['VL_TA'] * 100
# Criar VL/AREA SEM RESERVA
df_anuais['VL_areaSemReserva'] = df_anuais['qtdeVacasEmLactacao_mediaMovel'] / df_anuais['areaAtividadeSemReserva_mediaMovel']
# Criar VL/AREA COM RESERVA
df_anuais['VL_areaComReserva'] = df_anuais['qtdeVacasEmLactacao_mediaMovel'] / df_anuais['areaAtividadeReserva_mediaMovel']
# Criar CCS 
df_anuais['CCS_mediaMovel'] = df_anuais['absCCS_somaMovel'] / df_anuais['leiteProduzido_somaMovel']
# Criar CPP 
df_anuais['CPP_mediaMovel'] = df_anuais['absCPP_somaMovel'] / df_anuais['leiteProduzido_somaMovel']
# Criar Gordura 
df_anuais['Gordura_mediaMovel'] = df_anuais['absGordura_somaMovel'] / df_anuais['leiteProduzido_somaMovel']
# Criar Proteina 
df_anuais['Proteina_mediaMovel'] = df_anuais['absProteina_somaMovel'] / df_anuais['leiteProduzido_somaMovel']
# Criar Gordura/VL/Dia
df_anuais['Gordura_VL'] = (df_anuais['leiteProduzido_somaMovel']/365) * 1.032 * (df_anuais['Gordura_mediaMovel']/100) / df_anuais['qtdeVacasEmLactacao_mediaMovel']
# Criar Proteina 
df_anuais['Proteina_VL'] = (df_anuais['leiteProduzido_somaMovel']/365) * 1.032 * (df_anuais['Proteina_mediaMovel']/100) / df_anuais['qtdeVacasEmLactacao_mediaMovel']
# Criar Gordura/VL/Dia
df_anuais['Gordura_VL_Dia'] = (df_anuais['leiteProduzido_somaMovel']/365) * 1.032 * (df_anuais['Gordura_mediaMovel']/100) / df_anuais['qtdeVacasEmLactacao_mediaMovel']
# Criar Proteina 
df_anuais['Proteina_VL_Dia'] = (df_anuais['leiteProduzido_somaMovel']/365) * 1.032 * (df_anuais['Proteina_mediaMovel']/100) / df_anuais['qtdeVacasEmLactacao_mediaMovel']
# Criar VL/MDO Total
df_anuais['VL_MDO'] = df_anuais['qtdeVacasEmLactacao_mediaMovel'] / df_anuais['MDOTotalDiaria_mediaMovel']
# Criar Produção Diária dos indicadores anuais 
df_anuais['producaoDiaria'] = df_anuais['leiteProduzido_somaMovel'] / (30.42*12)
# Criar Produção por Vacas em Lactação 
df_anuais['producaoDiaria_VL'] = df_anuais['producaoDiaria'] / df_anuais['qtdeVacasEmLactacao_mediaMovel']
# Criar Produção por Total de Vacas 
df_anuais['producaoDiaria_TV'] = df_anuais['producaoDiaria'] / df_anuais['totalVacas_mediaMovel']
# Criar Produção por MDO Total  
df_anuais['producaoDiaria_MDO'] = df_anuais['producaoDiaria'] / df_anuais['MDOTotalDiaria_mediaMovel']
# Criar Produção por Área da Atividade Sem Reserva  
df_anuais['producaoAnual_AreaSemReserva'] = df_anuais['leiteProduzido_somaMovel'] / df_anuais['areaAtividadeSemReserva_mediaMovel']
# Criar Produção por Área da Atividade Com Reserva  
df_anuais['producaoAnual_AreaComReserva'] = df_anuais['leiteProduzido_somaMovel'] / df_anuais['areaAtividadeReserva_mediaMovel']
# Criar Produção por Área de Forrageira
df_anuais['producaoAnual_AreaForrageira'] = df_anuais['leiteProduzido_somaMovel'] / df_anuais['areaForrageira_mediaMovel']
# Criar Preço do Leite 
df_anuais['precoLeiteAnual'] = df_anuais['def_rendaLeite_somaMovel'] / df_anuais['leiteProduzido_somaMovel']
# Criar RBL/RBA 
df_anuais['rbl_rba'] = (df_anuais['def_rendaLeite_somaMovel'] / df_anuais['def_rendaAtividade_somaMovel']) * 100
# Concentrado + Minerais 
df_anuais['quantidade_Concentrado_Minerais_Anual'] = df_anuais[['quantidadeConsumida_Concentrado_somaMovel', 'quantidadeConsumida_Minerais_somaMovel']].sum(axis=1) 

# Criar Preço do Concentrado 
df_anuais['precoConcentradoAnual'] = df_anuais['custoAlimentacao_Concentrado_Minerais_somaMovel'] / df_anuais['quantidade_Concentrado_Minerais_Anual']
# Criar Relação de Troca Leite/Concentrado 
df_anuais['relacaoTroca'] = df_anuais['precoLeiteAnual'] / df_anuais['precoConcentradoAnual']
# Criar Estoque de Capital em Benfeitoria, Máquinas, Animais, Terra e Forrageira sobre EC Total com Terra 
colunas_EC = ['def_estoqueCapitalBenfeitorias_somaMovel', 'def_estoqueCapitalMaquinas_somaMovel',
              'estoqueCapitalAnimais_Mensal_somaMovel', 'def_estoqueTerra_Mensal_somaMovel', 'estoqueCapitalPlantio_acumulado_somaMovel']
df_anuais[[col + '_ECTotalcomTerra' for col in colunas_EC]] = pd.DataFrame(
    {col + '_ECTotalcomTerra': (df_anuais[col] / df_anuais['estoqueCapital_comTerra_somaMovel']) * 100 for col in colunas_EC}
)
df_anuais = df_anuais.copy()
# Criar EC Total com Terra por Litro 
df_anuais['estoqueCapital_comTerra_somaMovel_litro'] = df_anuais['estoqueCapital_comTerra_somaMovel'] / df_anuais['producaoDiaria']
# Criar EC Total com Terra por VL 
df_anuais['estoqueCapital_comTerra_somaMovel_VL'] = df_anuais['estoqueCapital_comTerra_somaMovel'] / df_anuais['qtdeVacasEmLactacao_mediaMovel']
# Criar COE 
df_anuais['cotAnual'] = df_anuais[['coe_somaMovel', 'depreciacaoPlantioMensal_acumulado_somaMovel', 'def_depreciacaoBenfeitorias_somaMovel',
                                   'def_depreciacaoMaquinas_somaMovel', 'def_familiarValortotal_somaMovel']].sum(axis=1)
# Criar o custo de oportunidade do capital 
df_anuais['custoOportunidadeCapital'] = df_anuais['estoqueCapital_semTerra_somaMovel'] * 0.06
# Criar COT 
df_anuais['ctAnual'] = df_anuais[['cotAnual', 'custoOportunidadeCapital']].sum(axis=1)
# Criar Margem Bruta Anual 
df_anuais['margemBrutaAnual'] = df_anuais['def_rendaAtividade_somaMovel'] - df_anuais['coe_somaMovel']
# Criar Margem Líquida Anual 
df_anuais['margemLiquidaAnual'] = df_anuais['def_rendaAtividade_somaMovel'] - df_anuais['cotAnual']
# Criar Lucro Anual 
df_anuais['lucroAnual'] = df_anuais['def_rendaAtividade_somaMovel'] - df_anuais['ctAnual']
# Criar RCMA 
df_anuais['RCMA'] = df_anuais['def_rendaAtividade_somaMovel'] - df_anuais[['custoAlimentacao_Concentrado_Minerais_somaMovel', 'custoAlimentacao_Volumoso_somaMovel']].sum(axis=1)
# Criar RCMA por VL 
df_anuais['RCMA_VL'] = df_anuais['RCMA'] / (df_anuais['qtdeVacasEmLactacao_mediaMovel']*365)
# Criar taxa de giro 
df_anuais['taxadegiro'] = (df_anuais['def_rendaAtividade_somaMovel'] / df_anuais['estoqueCapital_comTerra_somaMovel']) * 100
# Criar Lucratividade 
df_anuais['lucratividade'] = (df_anuais['margemLiquidaAnual']/df_anuais['def_rendaAtividade_somaMovel']) * 100
# Criar taxa de retorno sobre capital sem terra 
df_anuais['taxaRetornoCapitalSemTerra'] = np.where(
    (df_anuais['margemLiquidaAnual'] / df_anuais['estoqueCapital_semTerra_somaMovel']) * 100 < 0,
    0,
    (df_anuais['margemLiquidaAnual'] / df_anuais['estoqueCapital_semTerra_somaMovel']) * 100
)
# Criar taxa de retorno sobre capital com terra 
df_anuais['taxaRetornoCapitalComTerra'] = np.where(
    (df_anuais['margemLiquidaAnual'] / df_anuais['estoqueCapital_comTerra_somaMovel']) * 100 <0,
    0,
    (df_anuais['margemLiquidaAnual'] / df_anuais['estoqueCapital_comTerra_somaMovel']) * 100
)
# Criar a taxa de retorno sem filtro para ordenação 
df_anuais['taxaRetornoCapitalComTerra_Geral'] = (df_anuais['margemLiquidaAnual'] / df_anuais['estoqueCapital_comTerra_somaMovel']) * 100
# Criar PCOT 
df_anuais['pcot'] = (df_anuais['cotAnual'] / df_anuais['precoLeiteAnual']) / (12 * 30.42)
# Criar PCT  
df_anuais['pct'] = df_anuais['ctAnual'] / df_anuais['precoLeiteAnual'] / (12 * 30.42)
# Criar RB COE, COT e CT, MB, ML, Lucro e RCMA por litro 
colunas_economicas = ['def_rendaAtividade_somaMovel', 'coe_somaMovel', 'cotAnual', 'ctAnual', 'margemBrutaAnual',
                       'margemLiquidaAnual', 'lucroAnual']
df_anuais[[col + '_litro' for col in colunas_economicas]] = pd.DataFrame(
    {col + '_litro': df_anuais[col] / df_anuais['leiteProduzido_somaMovel'] for col in colunas_economicas}
)
# Criar RB COE, COT e CT, MB, ML, Lucro e RCMA por preço do Leite 
df_anuais[[col + '_precoLeite' for col in colunas_economicas]] = pd.DataFrame(
    {col + '_precoLeite': (df_anuais[col] / df_anuais['def_rendaAtividade_somaMovel']) * 100 for col in colunas_economicas}
)
# Criar RB COE, COT e CT, MB, ML, Lucro e RCMA em litros equivalentes 
df_anuais[[col + '_litrosEquivalente' for col in colunas_economicas]] = pd.DataFrame(
    {col + '_litrosEquivalente': df_anuais[col] / df_anuais['precoLeiteAnual'] for col in colunas_economicas}
)
# Criar RB COE, COT e CT, MB, ML, Lucro e RCMA por VL 
df_anuais[[col + '_VL' for col in colunas_economicas]] = pd.DataFrame(
    {col + '_VL': df_anuais[col] / df_anuais['qtdeVacasEmLactacao_mediaMovel'] for col in colunas_economicas}
)
# Criar RB COE, COT e CT, MB, ML, Lucro e RCMA por TV 
df_anuais[[col + '_TotalVacas' for col in colunas_economicas]] = pd.DataFrame(
    {col + '_TotalVacas': df_anuais[col] / df_anuais['totalVacas_mediaMovel'] for col in colunas_economicas}
)
# Criar RB COE, COT e CT, MB, ML, Lucro e RCMA por Área  
df_anuais[[col + '_AreaSemReserva' for col in colunas_economicas]] = pd.DataFrame(
    {col + '_AreaSemReserva': df_anuais[col] / df_anuais['areaAtividadeSemReserva_mediaMovel'] for col in colunas_economicas}
)
# Criar RB COE, COT e CT, MB, ML, Lucro e RCMA por Área com reserva 
df_anuais[[col + '_AreaComReserva' for col in colunas_economicas]] = pd.DataFrame(
    {col + '_AreaComReserva': df_anuais[col] / df_anuais['areaAtividadeReserva_mediaMovel'] for col in colunas_economicas}
)
df_anuais = df_anuais.copy()
# Criar Energia + Combustível 
df_anuais['custoEnergiaCombustivel'] = df_anuais[['def_custoEnergia_somaMovel', 'def_custoCombustivel_somaMovel']].sum(axis=1)

# Investimento em animais / estoque de capital em animais 
df_anuais['investimentoAnimais_ECAnimais'] = (df_anuais['investimentoAnimais_somaMovel'] / df_anuais['estoqueCapitalAnimais_Mensal_somaMovel']) * 100
# Investimento em máquinas/estoque de capital em máquinas 
df_anuais['investimentoMaquinas_ECMaquinas'] = (df_anuais['investimentoMaquinas_somaMovel'] / df_anuais['def_estoqueCapitalMaquinas_somaMovel']) * 100
# Investimento em Benfeitorias / Estoque de capital em Benfeitorias 
df_anuais['investimentoBenfeitorias_ECBenfeitorias'] = (df_anuais['investimentoBenfeitorias_somaMovel'] / df_anuais['def_estoqueCapitalBenfeitorias_somaMovel']) * 100
# Investimento anual / estoque de capital com terra 
df_anuais['investimento_ECcomTerra'] = (df_anuais['investimento_somaMovel'] / df_anuais['estoqueCapital_comTerra_somaMovel']) * 100
# Investimento anual / RBA 
df_anuais['investimento_RBA'] = (df_anuais['investimento_somaMovel'] / df_anuais['def_rendaAtividade_somaMovel']) * 100
# Investimento anual / MB 
df_anuais['investimento_MB'] = (df_anuais['investimento_somaMovel'] / df_anuais['margemBrutaAnual']) * 100
# Criar coluna de Vacas em Lactação por Área destinada à atividade
df_anuais['VL_AASR'] = df_anuais['qtdeVacasEmLactacao_mediaMovel'] / df_anuais['areaAtividadeSemReserva_mediaMovel']
# Criar coluna de Vacas em Lactação por Área destinada à atividade considerando reserva
df_anuais['VL_AACR'] = df_anuais['qtdeVacasEmLactacao_mediaMovel'] / df_anuais['areaAtividadeReserva_mediaMovel']


# =================== DEFINIÇÃO DE CONSISTÊNCIA ==================================================================
# 1. Definir as regras de consistência ANUAIS
# Esta é a sua Regra 1. O resultado será True se TODAS as condições forem satisfeitas,
# e False se pelo menos uma falhar.
regras_consistencia_anuais = (
    (df_anuais['MDOTotalDiaria'] > 0.5) & (df_anuais['MDOTotalDiaria'] <= 40) &
    (df_anuais['CCS_mediaMovel'] > 50) & (df_anuais['CCS_mediaMovel'] < 9999) & (df_anuais['CPP_mediaMovel'] > 1) &
    (df_anuais['Gordura_mediaMovel'] > 2.5) & (df_anuais['Gordura_mediaMovel'] < 5.5) & (df_anuais['Proteina_mediaMovel'] > 2.4) & (df_anuais['Proteina_mediaMovel'] < 4.5) &
    (df_anuais['VL_TV'] > 0.2) & (df_anuais['VL_TV'] < 0.99) & (df_anuais['VL_TA'] > 0.15) & (df_anuais['VL_TA'] < 0.99) &
    (df_anuais['VL_AASR'] < 20) & (df_anuais['VL_AACR'] < 20) &
    (df_anuais['producaoDiaria_VL'] > 3) & (df_anuais['producaoDiaria_VL'] < 45) &
    (df_anuais['producaoDiaria_MDO'] < 1500) &
    (df_anuais['VL_MDO'] < 70) &
    (df_anuais['estoqueCapital_comTerra_somaMovel_litro'] > 800) & (df_anuais['estoqueCapital_comTerra_somaMovel_litro'] < 30000) &
    (df_anuais['cotAnual_precoLeite'] >= 50) & (df_anuais['cotAnual_precoLeite'] <= 150) &
    (df_anuais['rbl_rba'] >= 60) & (df_anuais['rbl_rba'] <= 100) &
    (df_anuais['taxadegiro'] <= 180) &
    (df_anuais['lucratividade'] >= -50) & (df_anuais['lucratividade'] <= 50) &
    (df_anuais['taxaRetornoCapitalComTerra_Geral'] <= 40)
)

# 2. Definir as condições e os valores correspondentes em ordem de prioridade
# A função np.select avalia as condições na ordem em que são fornecidas.
# A primeira condição que for True para uma linha, define o valor para essa linha,
# e as condições subsequentes para aquela linha são ignoradas.

condicoes = [
    # Condição 1 (Prioridade Máxima): Se a linha FALHAR nas regras anuais, é 'Inconsistente'.
    # Usamos '~' para negar 'regras_consistencia_anuais', ou seja, True onde as regras falham.
    ~regras_consistencia_anuais,

    # Condição 2: Se idConsistencia_somaMovel for maior que 2, é 'Inconsistente'.
    # Esta condição só será avaliada para as linhas que *passaram* na 'regras_consistencia_anuais'.
    (df_anuais['idConsistencia_somaMovel'] > 2),

    # Condição 3: Se idConsistencia_somaMovel for 1 ou 2, é 'Parcialmente consistente'.
    # Esta condição só será avaliada para as linhas que *passaram* na 'regras_consistencia_anuais'
    # E que não tiveram 'idConsistencia_somaMovel' maior que 2.
    (df_anuais['idConsistencia_somaMovel'].isin([1, 2]))
]

# Os valores que serão atribuídos para cada condição, na mesma ordem das 'condicoes'.
valores = [
    'Inconsistente',           # Corresponde à primeira condição (~regras_consistencia_anuais)
    'Inconsistente',           # Corresponde à segunda condição (idConsistencia_somaMovel > 2)
    'Parcialmente consistente'  # Corresponde à terceira condição (idConsistencia_somaMovel.isin([1, 2]))
]

# Aplica as condições. O 'default' é o valor para as linhas que não se encaixam em nenhuma das 'condicoes'.
# Ou seja, são as linhas que passaram nas regras anuais E cujo idConsistencia_somaMovel não é 1, 2 ou > 2.
df_anuais['consistenciaAnual'] = np.select(condicoes, valores, default='Consistente')
# =================== DEFINIÇÃO DE CONSISTÊNCIA ==================================================================

# Ver amostra 
print(f"Número de linhas: {df_anuais.shape[0]}")

df_anuais = df_anuais.merge(d_fazenda[['idFazenda', 'codAgroindustria', 'nomeFazenda',  'nomeprodutor', 'regiaoLeiteira', 'nomeagroindustria', 'nomeconsultor']], 
                            on='idFazenda', 
                            how='left').copy()
# df_anuais = df_anuais.loc[df_anuais['nomeconsultor'].notna()]
df_anuais.reset_index(drop=True, inplace=True)
# Ver amostra 
print(f"Número de linhas: {df_anuais.shape[0]}")

df_anuais['fazenda-produtor'] = df_anuais['nomeFazenda'] + ' - ' + df_anuais['nomeprodutor']

df_anuais['Status_Mensais'] = None 
# Cria a série de fazendas
# Filtro 1 será contagem de idFazenda
# Aplicar a contagem acumulada
df_anuais['Filtro1'] = df_anuais.groupby(['idFazenda', 'intervalo_movel']).cumcount() + 1
df_anuais['Filtro2'] = None
df_anuais['Filtro3'] = None
df_anuais['Filtro4'] = None
df_anuais['Exames_Laboratoriais'] = None
df_anuais['Qualidade_Leite'] = None
df_anuais['Terceirizacao'] = None
df_anuais['Transporte'] = None
df_anuais['Vacinas'] = None

df_anuais.head()

Mostrar o número de linhas na base: 58558
Mostrar o número de linhas após Leite: 58708
Mostrar o número de linhas após Áreas: 58708
Mostrar o número de linhas após Leite Consumido: 58708
Mostrar o número de linhas após MDO: 58852
Mostrar o número de linhas após Renda: 59141
Mostrar o número de linhas após Estoque de Capital: 59141
Mostrar o número de linhas após Área de Forrageira: 59141
Mostrar o número de linhas após Custos: 59717
Mostrar o número de linhas após Alimentação: 60885
Mostrar o número de linhas após Rebanho: 60889
Mostrar o número de linhas após Investimento: 60889
Número de linhas antes de remover duplicatas: 60889
Número de linhas após remover duplicatas: 58558


<timed exec>:124: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


Mostrar o número de linhas após Média Móvel: 58558
Mostrar o número de linhas após retirar NaN: 6813
Número de linhas: 6813
Número de linhas: 9149
CPU times: total: 58.6 s
Wall time: 1min 1s


,idFazenda,mesReferencia,leiteProduzido,def_precoLeite,absCCS,absCPP,absGordura,absProteina,idConsistencia,areaArrendada,...,Status_Mensais,Filtro1,Filtro2,Filtro3,Filtro4,Exames_Laboratoriais,Qualidade_Leite,Terceirizacao,Transporte,Vacinas
0,1,2023-04-01,12811.00,2.96,0.00,0.00,0.00,0.00,1.00,0.00,...,None,1,None,None,None,None,None,None,None,None
1,1,2023-05-01,13001.50,3.13,0.00,0.00,0.00,0.00,1.00,0.00,...,None,1,None,None,None,None,None,None,None,None
2,1,2023-06-01,16743.00,3.07,0.00,0.00,0.00,0.00,1.00,0.00,...,None,1,None,None,None,None,None,None,None,None
3,1,2023-07-01,20218.00,2.97,0.00,0.00,0.00,0.00,1.00,0.00,...,None,1,None,None,None,None,None,None,None,None
4,1,2023-08-01,19504.00,2.87,5714672.00,292560.00,86597.76,70214.40,0.00,0.00,...,None,1,None,None,None,None,None,None,None,None


In [69]:
# ============================================================
# DIAGNÓSTICO — pode remover depois de resolver o problema
# ============================================================

# Passo 1 — Fazendas candidatas
periodo_inicio = pd.Timestamp('2023-04-01')
periodo_fim = pd.Timestamp('2024-03-01')

fazendas_candidatas = (
    df_base_anuais[
        (df_base_anuais['mesReferencia'] >= periodo_inicio) &
        (df_base_anuais['mesReferencia'] <= periodo_fim) &
        (df_base_anuais['leiteProduzido'].notna()) &
        (df_base_anuais['leiteProduzido'] > 0)
    ]
    .groupby('idFazenda')['mesReferencia']
    .count()
    .reset_index()
    .rename(columns={'mesReferencia': 'meses_com_leite'})
)
fazendas_12_meses = fazendas_candidatas[fazendas_candidatas['meses_com_leite'] == 12]
print(f"Fazendas com 12 meses de leite no período: {len(fazendas_12_meses)}")

# Passo 2 — Quem está ausente em df_anuais
ids_candidatas = set(fazendas_12_meses['idFazenda'])
ids_anuais_marco2024 = set(
    df_anuais[df_anuais['mesReferencia'] == periodo_fim]['idFazenda']
)

fazendas_ausentes = ids_candidatas - ids_anuais_marco2024
print(f"✅ Presentes nos anuais em Mar/2024: {len(ids_candidatas & ids_anuais_marco2024)}")
print(f"❌ Ausentes nos anuais em Mar/2024: {len(fazendas_ausentes)}")
print(f"IDs ausentes: {fazendas_ausentes}")

# Passo 3 — Investigar um ID ausente
if fazendas_ausentes:
    id_teste = list(fazendas_ausentes)[0]
    print(f"\n=== Investigando idFazenda: {id_teste} ===")

    print("\n--- df_leite ---")
    print(df_leite[df_leite['idFazenda'] == id_teste]
          [['idFazenda','mesReferencia','leiteProduzido']].head(15))

    print("\n--- df_ind_mensais ---")
    print(df_ind_mensais[df_ind_mensais['idFazenda'] == id_teste]
          [['idFazenda','mesReferencia','leiteProduzido']].head(15))

    print("\n--- df_base_anuais ---")
    print(df_base_anuais[
        (df_base_anuais['idFazenda'] == id_teste) &
        (df_base_anuais['mesReferencia'] >= periodo_inicio) &
        (df_base_anuais['mesReferencia'] <= periodo_fim)
    ][['idFazenda','mesReferencia','leiteProduzido']])

# ============================================================
# FIM DO DIAGNÓSTICO
# ============================================================

Fazendas com 12 meses de leite no período: 84
✅ Presentes nos anuais em Mar/2024: 84
❌ Ausentes nos anuais em Mar/2024: 0
IDs ausentes: set()


## Gerar Arquivo de Consistência Anual

In [70]:
# Selecionar colunas
df_cons_ano = df_anuais[['idFazenda','mesReferencia','consistenciaAnual']].copy()
# Retirar duplicadas
df_cons_ano.drop_duplicates(subset=['idFazenda', 'mesReferencia'], inplace=True)
# Retirar duplicadas de d_fazenda
merge_fazenda = d_fazenda[['idFazenda', 'codAgroindustria']].drop_duplicates()
# # # Juntar codigo LR ao dataframe
df_cons_ano = df_cons_ano.merge(merge_fazenda, on='idFazenda', how='left')
# Garantir que mesReferencia está em datetime
df_cons_ano['mesReferencia'] = pd.to_datetime(df_cons_ano['mesReferencia'])
# Rename columns
df_cons_ano.columns = ['idfazenda','mes_elabore', 'consistencia_anual', 'codigo_lr']
# Adicionar mes_referencia
df_cons_ano['mes_referencia'] = df_cons_ano['mes_elabore'] + pd.DateOffset(months=1)
# # Transformar em string
# df_cons_ano['mesReferencia'] = df_cons_ano['mesReferencia'].dt.strftime('%d/%m/%Y')
df_cons_ano.to_excel('C:\\Users\\analy\\LABOR RURAL\\Analytics - Departamento Analytics\\POWER_BI\\PROJETOS\\ELABORE\\GESTOR\\BD_CONSISTENTES_IND_ANUAIS.xlsx', index=False)

# --- CORREÇÃO: Remover duplicatas com base na chave primária antes do upsert ---
# A chave primária da tab_consistencia_anual é (idfazenda, mes_elabore)
df_cons_ano.drop_duplicates(subset=['idfazenda', 'mes_elabore'], keep='last', inplace=True)
print(f"Após remover duplicatas, {len(df_cons_ano)} registros únicos para upsert na tabela anual.")

# Converter as colunas datetime para string ISO 8601
# O formato '%Y-%m-%dT%H:%M:%S.%fZ' é comum para UTC.
# Se suas datas tiverem fuso horário, use '%Y-%m-%dT%H:%M:%S.%f%z'.
# Se suas datas são apenas ano-mês-dia e você quer evitar a parte da hora,
# pode usar '%Y-%m-%d' e o Supabase deve interpretar como início do dia.
# Para TIMESTAMP WITH TIME ZONE, é mais seguro incluir a hora e fuso horário.
df_cons_ano['mes_elabore'] = df_cons_ano['mes_elabore'].dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')
df_cons_ano['mes_referencia'] = df_cons_ano['mes_referencia'].dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')

# Converter DataFrame para uma lista de dicionários
data_to_upsert_anual = df_cons_ano.to_dict(orient='records')

# Definir tabela
TABLE_NAME_ANUAL = 'tab_consistencia_anual'

# --- 3. Realizar o Upsert em Chunks ---
print(f"\nIniciando processo de UPSERT para a tabela '{TABLE_NAME_ANUAL}' no Supabase.")

if data_to_upsert_anual:
    print(f"   - Processando {len(data_to_upsert_anual)} registros para upsert...")

    chunk_size = 1000 # Tamanho do chunk, ajuste conforme a necessidade e limites da API
    for i in range(0, len(data_to_upsert_anual), chunk_size):
        chunk = data_to_upsert_anual[i:i + chunk_size]
        try:
            # O método .upsert() usará a chave primária (idfazenda, mes_elabore)
            # para identificar conflitos e atualizar ou inserir.
            response = supabase.table(TABLE_NAME_ANUAL) \
                .upsert(chunk) \
                .execute()

            if response.data:
                print(f"     - Upserted {len(response.data)} registros em chunk {i//chunk_size + 1}.")
            else:
                print(f"     - Nenhum registro upserted para chunk {i//chunk_size + 1} ou erro na resposta.")
                if response.error:
                    print(f"       Erro Supabase: {response.error}")

        except Exception as e:
            print(f"❌ Erro ao realizar upsert no Supabase (chunk {i//chunk_size + 1}): {str(e)}")

    print(f"   - UPSERTs concluídos. Total de registros processados: {len(data_to_upsert_anual)}")
else:
    print("   - Nenhum registro para upsert.")

print("\nProcesso de upsert anual finalizado.")


df_cons_ano.tail(20)

Após remover duplicatas, 6813 registros únicos para upsert na tabela anual.

Iniciando processo de UPSERT para a tabela 'tab_consistencia_anual' no Supabase.
   - Processando 6813 registros para upsert...
     - Upserted 1000 registros em chunk 1.
     - Upserted 1000 registros em chunk 2.
     - Upserted 1000 registros em chunk 3.
     - Upserted 1000 registros em chunk 4.
     - Upserted 1000 registros em chunk 5.
     - Upserted 1000 registros em chunk 6.
     - Upserted 813 registros em chunk 7.
   - UPSERTs concluídos. Total de registros processados: 6813

Processo de upsert anual finalizado.


,idfazenda,mes_elabore,consistencia_anual,codigo_lr,mes_referencia
6793,922,2024-12-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-01-01T00:00:00.000000Z
6794,922,2025-01-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-02-01T00:00:00.000000Z
6795,922,2025-02-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-03-01T00:00:00.000000Z
6796,922,2025-03-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-04-01T00:00:00.000000Z
6797,922,2025-04-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-05-01T00:00:00.000000Z
6798,922,2025-05-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-06-01T00:00:00.000000Z
6799,922,2025-06-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-07-01T00:00:00.000000Z
6800,922,2025-07-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-08-01T00:00:00.000000Z
6801,922,2025-08-01T00:00:00.000000Z,Inconsistente,Treinamento,2025-09-01T00:00:00.000000Z
6802,922,2025-09-01T00:00:00.000000Z,Parcialmente consistente,Treinamento,2025-10-01T00:00:00.000000Z


## Arquivo Final

In [71]:
# Selecionar colunas para trabalhar
colunas_finais = ['idFazenda', 'fazenda-produtor', 'codAgroindustria', 'regiaoLeiteira', 'nomeagroindustria', 'nomeconsultor','intervalo_movel', 'consistenciaAnual', 'Status_Mensais', 'sistema', # Dimensão
                  'Filtro1', 'Filtro2', 'Filtro3', 'Filtro4', 'taxaRetornoCapitalComTerra_Geral',
                  'areaAtividadeSemReserva_mediaMovel', 'areaAtividadeReserva_mediaMovel', 'areaPropria_mediaMovel', # Área
                  'areaArrendada_mediaMovel', 'percentualAreaArrendada', 'precoTerraNua_somaMovel', # Área
                  'qtdeVacasEmLactacao_mediaMovel', 'totalVacas_mediaMovel', 'totalAnimais_mediaMovel', # Rebanho
                  'quantidade_Concentrado_Minerais_Anual', # Concentrado
                  'MDOTotalDiaria_mediaMovel', 'MDOFamiliarDiaria_mediaMovel', 'MDOContratadaDiaria_mediaMovel', # MDO
                  'CCS_mediaMovel', 'CPP_mediaMovel', 'Gordura_mediaMovel', 'Proteina_mediaMovel', 'absCCS_somaMovel', # Qualidade do Leite
                  'absCPP_somaMovel', 'absGordura_somaMovel', 'absProteina_somaMovel', 'Gordura_VL', 'Proteina_VL', # Qualidade do Leite
                  'VL_TV_100', 'VL_TA_100', 'VL_areaSemReserva', 'VL_areaComReserva', 'VL_MDO', # Rebanho 
                  'leiteProduzido_somaMovel', 'consumoLeiteDescartado_somaMovel', 'producaoDiaria', 'producaoDiaria_VL', 'producaoDiaria_TV', # Produção de Leite 
                  'producaoDiaria_MDO', 'producaoAnual_AreaSemReserva', 'producaoAnual_AreaComReserva', 'producaoAnual_AreaForrageira','areaForrageira_mediaMovel', # Produção de Leite
                  'def_rendaAtividade_somaMovel', 'def_rendaLeite_somaMovel', 'precoLeiteAnual', 'def_rendaLeite_somaMovel', # Receita 
                  'precoConcentradoAnual', 'relacaoTroca', # Alimentação x Leite
                  'estoqueCapital_semTerra_somaMovel', 'estoqueCapital_comTerra_somaMovel', 'def_estoqueCapitalBenfeitorias_somaMovel', # Estoque de Capital
                  'def_estoqueCapitalMaquinas_somaMovel', 'estoqueCapitalAnimais_Mensal_somaMovel', 'def_estoqueTerra_Mensal_somaMovel', 'estoqueCapitalPlantio_acumulado_somaMovel', # Estoque de Capital
                  'def_estoqueCapitalBenfeitorias_somaMovel_ECTotalcomTerra', 'def_estoqueCapitalMaquinas_somaMovel_ECTotalcomTerra', 'estoqueCapitalAnimais_Mensal_somaMovel_ECTotalcomTerra', # Estoque de Capital 
                  'def_estoqueTerra_Mensal_somaMovel_ECTotalcomTerra', 'estoqueCapitalPlantio_acumulado_somaMovel_ECTotalcomTerra', # Estoque de Capital 
                  'coe_somaMovel_litrosEquivalente', 'cotAnual_litrosEquivalente', 'ctAnual_litrosEquivalente', # COE, COT e CT 
                  'coe_somaMovel_precoLeite', 'cotAnual_precoLeite', 'ctAnual_precoLeite', # COE, COT e CT 
                  'margemBrutaAnual', 'margemBrutaAnual_litro', 'margemBrutaAnual_AreaSemReserva', 'margemBrutaAnual_AreaComReserva', 'margemBrutaAnual_VL', 'margemBrutaAnual_TotalVacas', # Margem Bruta 
                  'margemLiquidaAnual', 'margemLiquidaAnual_litro', 'margemLiquidaAnual_AreaSemReserva', 'margemLiquidaAnual_AreaComReserva', 'margemLiquidaAnual_VL', 'margemLiquidaAnual_TotalVacas', # Margem Líquida 
                  'lucroAnual', 'lucroAnual_litro', # Lucro
                  'RCMA', 'RCMA_VL', # RCMA
                  'rbl_rba', # RBL/RBA
                  'estoqueCapital_comTerra_somaMovel_VL', 'estoqueCapital_comTerra_somaMovel_litro', # Estoque de Capital
                  'investimentoAnimais_ECAnimais', 'investimentoMaquinas_ECMaquinas', 'investimentoBenfeitorias_ECBenfeitorias', 'investimento_ECcomTerra', # Investimento
                  'investimento_RBA', 'investimento_MB', 'investimentoAnimais_somaMovel', 'investimentoMaquinas_somaMovel', 'investimentoBenfeitorias_somaMovel', 'investimento_somaMovel', # Investimento
                  'taxadegiro', 'lucratividade', 'taxaRetornoCapitalSemTerra', 'taxaRetornoCapitalComTerra', 'pcot', 'pct', # Econômico
                  'coe_somaMovel', 'def_gastoAcessoriosDespesasGerais_somaMovel', 'def_aleitaento_somaMovel', 'def_gastoArrendamento_somaMovel', # COE
                  'custoAlimentacao_Concentrado_Minerais_somaMovel', 'custoReceitasConcentrado_somaMovel', 'def_gastoAdministrativo_somaMovel', # COE
                  'def_gastoAssistenciaTecnica_somaMovel', 'custoEnergiaCombustivel', 'Exames_Laboratoriais', 'def_gastoHormonios_somaMovel', 'def_gastoImpostoTaxas_somaMovel', # COE
                  'custoMDOContratada_somaMovel', 'def_gastoMaterialOrdenha_somaMovel', 'def_gastoMedicamentosVacinas_somaMovel', 'Qualidade_Leite', 'def_gastoReparosConsertos_somaMovel', # COE
                  'def_gastoReproducao_somaMovel', 'Terceirizacao', 'Transporte', 'Vacinas', 'custoAlimentacao_Volumoso_somaMovel', 'custoReceitasVolumoso_somaMovel', # COE
                  'cotAnual', 'depreciacaoEstoqueCapital_somaMovel', 'def_familiarValortotal_somaMovel', 'ctAnual', 'custoOportunidadeCapital' # COT, CT, Depreciação, MDO Familiar, Custo Oportunidade
                 ]
df_final = df_anuais[colunas_finais].copy() 
df_final.tail(10)

,idFazenda,fazenda-produtor,codAgroindustria,regiaoLeiteira,nomeagroindustria,nomeconsultor,intervalo_movel,consistenciaAnual,Status_Mensais,sistema,...,Terceirizacao,Transporte,Vacinas,custoAlimentacao_Volumoso_somaMovel,custoReceitasVolumoso_somaMovel,cotAnual,depreciacaoEstoqueCapital_somaMovel,def_familiarValortotal_somaMovel,ctAnual,custoOportunidadeCapital
9139,922,Fazenda treinamento correção - Produtor treina...,Treinamento,TESTE,NaN,Andreza Martins,Dec/24-Nov/25,Consistente,None,Compost Barn,...,None,None,None,1148048.35,22762.59,4181207.51,240324.35,101238.11,4691021.82,509814.31
9140,922,Fazenda treinamento correção - Produtor treina...,Treinamento,TESTE,NaN,Herick Lucca Costa Viana,Jan/25-Dec/25,Inconsistente,None,Compost Barn,...,None,None,None,1168567.61,22762.59,4054352.61,254698.27,93361.59,4574597.69,520245.08
9141,922,Fazenda treinamento correção - Produtor treina...,Treinamento,TESTE,NaN,Andreza Martins,Jan/25-Dec/25,Inconsistente,None,Compost Barn,...,None,None,None,1168567.61,22762.59,4054352.61,254698.27,93361.59,4574597.69,520245.08
9142,1002,Fazenda Cachoeira - Juscelio Lopes de Miranda,LR02562,Minas Gerais,Alvoar,Paulo Sergio da Silva Lopes,Jan/25-Dec/25,Consistente,None,Semi Confinado,...,None,None,None,57805.13,0.00,480842.22,54798.24,74003.97,538695.96,57853.74
9143,1002,Fazenda Cachoeira - Juscelio Lopes de Miranda,LR02562,Minas Gerais,Alvoar,Paulo Sergio da Silva Lopes,Feb/25-Jan/26,Consistente,None,Semi Confinado,...,None,None,None,54484.69,0.00,472097.81,55808.22,74128.54,530256.63,58158.82
9144,1002,Fazenda Cachoeira - Juscelio Lopes de Miranda,LR02562,Minas Gerais,Alvoar,Paulo Sergio da Silva Lopes,Mar/25-Feb/26,Consistente,None,Semi Confinado,...,None,None,None,50584.44,0.00,469667.07,56669.55,74359.18,528190.60,58523.53
9145,1002,Fazenda Cachoeira - Juscelio Lopes de Miranda,LR02562,Minas Gerais,Alvoar,Paulo Sergio da Silva Lopes,Apr/25-Mar/26,Consistente,None,Semi Confinado,...,None,None,None,47479.68,0.00,470657.16,57610.74,74507.01,529586.42,58929.26
9146,1193,Estância Sobreiro - José Maria Silva Sobreiro,None,None,Independente,Winston Robert Wilt,Mar/25-Feb/26,Inconsistente,None,NaN,...,None,None,None,301938.49,0.00,2191709.91,0.00,0.00,2375396.15,183686.24
9147,1193,Estância Sobreiro - José Maria Silva Sobreiro,None,None,Independente,Winston Robert Wilt,Apr/25-Mar/26,Inconsistente,None,NaN,...,None,None,None,324778.76,0.00,2279993.17,0.00,0.00,2465236.65,185243.48
9148,1193,Estância Sobreiro - José Maria Silva Sobreiro,None,None,Independente,Winston Robert Wilt,May/25-Apr/26,Inconsistente,None,NaN,...,None,None,None,342830.66,0.00,2336042.91,0.00,0.00,2520464.76,184421.85


In [72]:
# Lista original sem idFazenda
colunas_originais = [col for col in colunas_finais if col != 'idFazenda']

# Novos nomes das colunas
novos_nomes = [
    'IDFazenda',
    'Fazenda - Produtor',
    'Código LR',
    'Região',
    'Agroindústria', 
    'Consultor',
    'Período',
    'Status - Ind. Anuais',
    'Status - Ind. Mensais',
    'Sistema de produção atual',
    'Filtro 1',
    'Filtro 2',
    'Filtro 3',
    'Filtro 4',
    'AUXILIAR PARA ESTRATIFICAÇÃO - TRCCT CALCULADA',
    'Área destinada à atividade (hectare)',
    'Área destinada à atividade considerando reserva (hectare)',
    'Área própria considerando reserva (hectare)',
    'Área arrendada considerando reserva (hectare)',
    'Percentual de área arrendada (%)',
    'Preço médio da terra própria (R$/hectare)',
    'Vacas em lactação (animais/mês)',
    'Total de vacas (animais/mês)',
    'Total de animais (animais/mês)',
    'Consumo de concentrado anual (Kg/Ano)',
    'Mão de obra total (trabalhador)',
    'Mão de obra familiar (trabalhador)',
    'Mão de obra contratada (trabalhador)',
    'CCS (Contagem de células somáticas) (x1000 células/ml)',
    'CPP (Contagem padrão em placas) (x1000 UFC/ml)',
    'Gordura (%)',
    'Proteína (%)',
    'AUXILIAR PARA MÉDIAS - CCS (Contagem de células somáticas) (x1000 células/ml)',
    'AUXILIAR PARA MÉDIAS - CPP (Contagem padrão em placas) (x1000 UFC/ml)',
    'AUXILIAR PARA MÉDIAS - Gordura (%)',
    'AUXILIAR PARA MÉDIAS - Proteína (%)',
    'Gordura (kg/vaca em lactação/dia)',
    'Proteína (kg/vaca em lactação/dia)',
    'Vacas em lactação/total de vacas (%)',
    'Vacas em lactação/total de animais (%)',
    'Vacas em lactação/área destinada à atividade (animais/hectare)',
    'Vacas em lactação/área destinada à atividade considerando reserva (animais/hectare)',
    'Vacas em lactação/mão de obra total (animais/trabalhador/dia)',
    'Produção anual de leite (litros/ano)',
    'Leite descartado (litros/ano)',
    'Produção diária de leite (litros/dia)',
    'Produção/vacas em lactação (litros/animal/dia)',
    'Produção/total de vacas (litros/animal/dia)',
    'Produção/mão de obra total (litros/trabalhador/dia)',
    'Produção/área destinada à atividade (litros/hectare/ano)',
    'Produção/área destinada à atividade considerando reserva (litros/hectare/ano)',
    'Produção / área de produção de forrageira (litros/hectare/ano)',
    'AUXILIAR PARA MÉDIAS - Produção / área de produção de forrageira (litros/hectare/ano)',
    'Renda bruta da atividade leiteira (R$/ano)',
    'Renda bruta do leite (R$/ano)',
    'Preço médio do leite (R$/litro)',
    'AUXILIAR PARA MÉDIAS - Preço médio do leite (R$/litro)',
    'Preço médio do concentrado (R$/Kg)',
    'Relação de troca leite/concentrado (Kg/L)',
    'Estoque de capital total sem terra (R$)',
    'Estoque de capital total com terra (R$)',
    'Estoque de capital em benfeitorias (R$)',
    'Estoque de capital em máquinas (R$)',
    'Estoque de capital em animais (R$)',
    'Estoque de capital em terra (R$)',
    'Estoque de capital em forrageiras não-anuais (R$)',
    'Estoque de capital em benfeitorias/estoque de capital total com terra (%)',
    'Estoque de capital em máquinas/estoque de capital total com terra (%)',
    'Estoque de capital em animais/estoque de capital total com terra (%)',
    'Estoque de capital em terra/estoque de capital total com terra (%)',
    'Estoque de capital em forrageiras não-anuais/estoque de capital total com terra (%)',
    'COE da atividade leiteira em equivalentes litros de leite (litros/ano)',
    'COT da atividade leiteira em equivalentes litros de leite (litros/ano)',
    'CT da atividade leiteira em equivalentes litros de leite (litros/ano)',
    'COE do leite/preço do leite (%)',
    'COT do leite/preço do leite (%)',
    'CT do leite/preço do leite (%)',
    'Margem bruta da atividade (R$/ano)',
    'Margem bruta unitária (R$/litro)',
    'Margem bruta/área destinada à atividade (R$/hectare/ano)',
    'Margem bruta/área destinada à atividade considerando reserva (R$/hectare/ano)',
    'Margem bruta/vacas em lactação (R$/animal/ano)',
    'Margem bruta/total de vacas (R$/animal/ano)',
    'Margem líquida da atividade (R$/ano)',
    'Margem líquida unitária (R$/litro)',
    'Margem líquida/área destinada à atividade (R$/hectare/ano)',
    'Margem líquida/área destinada à atividade considerando reserva (R$/hectare/ano)',
    'Margem líquida/vacas em lactação (R$/animal/ano)',
    'Margem líquida/total de vacas (R$/animal/ano)',
    'Lucro total (R$/ano)',
    'Lucro unitário (R$/litro)',
    'RMCA (Receita Menos Custo com Alimentação) (R$/ano)',
    'RMCA (Receita Menos Custo com Alimentação) (R$/vaca em lactação/dia)',
    'Renda do leite/renda atividade (%)',
    'Estoque de capital total com terra/vaca em lactação (R$/animal)',
    'Estoque de capital total com terra/produção diária de leite (R$/litro/dia)',
    'Investimento em animais/estoque de capital em animais (%)',
    'Investimento em máquinas e equipamento/estoque de capital em máquinas e equipamentos (%)',
    'Investimento em benfeitorias/estoque de capital em benfeitorias (%)',
    'Investimento anual/estoque de capital total com terra (%)',
    'Investimento anual/renda bruta da atividade (%)',
    'Investimento anual/margem bruta (%)',
    'AUXILIAR PARA MÉDIAS - Investimento em animais',
    'AUXILIAR PARA MÉDIAS - Investimento em máquinas e equipamento',
    'AUXILIAR PARA MÉDIAS - Investimento em benfeitorias',
    'AUXILIAR PARA MÉDIAS - Investimento anual',
    'Taxa de giro do estoque de capital total (%)',
    'Lucratividade operacional (%)',
    'Taxa de remuneração do capital sem terra (% ao ano)',
    'Taxa de remuneração do capital com terra (% ao ano)',
    'Ponto de cobertura operacional total da atividade (litros/dia)',
    'Ponto de cobertura total da atividade (litros/dia)',
    'Custo operacional efetivo - R$/ano (Atividade)',
    'Acessórios e despesas em geral - R$/ano (Atividade)',
    'Aleitamento - R$/ano (Atividade)',
    'Arrendamento/Aluguel - R$/ano (Atividade)',
    'Concentrados e Minerais de ingestão livre - R$/ano (Atividade)',
    'Concentrados (Vendido) - R$/ano (Atividade)',
    'Despesas administrativas - R$/ano (Atividade)',
    'Despesas com assistência técnica - R$/ano (Atividade)',
    'Energia e combustível - R$/ano (Atividade)',
    'Exames laboratoriais - R$/ano (Atividade)',
    'Hormônios - R$/ano (Atividade)',
    'Impostos e taxas - R$/ano (Atividade)',
    'Mão de obra contratada - R$/ano (Atividade)',
    'Material de ordenha - R$/ano (Atividade)',
    'Medicamentos - R$/ano (Atividade)',
    'Qualidade do leite - R$/ano (Atividade)',
    'Reparos e consertos de máquinas e benfeitorias - R$/ano (Atividade)',
    'Reprodução - R$/ano (Atividade)',
    'Terceirização de recria - R$/ano (Atividade)',
    'Transporte e descontos no leite - R$/ano (Atividade)',
    'Vacinas - R$/ano (Atividade)',
    'Volumosos - R$/ano (Atividade)',
    'Volumosos (Vendido) - R$/ano (Atividade)',
    'Custo operacional total - R$/ano (Atividade)',
    'Depreciação - R$/ano (Atividade)',
    'Mão de obra familiar - R$/ano (Atividade)',
    'Custo total - R$/ano (Atividade)',
    'Remuneração do capital - R$/ano (Atividade)'
]

# Verificar se as quantidades batem
print(f"Colunas originais (sem idFazenda): {len(colunas_originais)}")
print(f"Novos nomes: {len(novos_nomes)}")

# Renomear por posição ao invés de usar dicionário
df_final_renamed = df_final.copy()


# Renomear todas as colunas restantes por posição
df_final_renamed.columns = novos_nomes


Colunas originais (sem idFazenda): 139
Novos nomes: 140


In [73]:
df_export = df_final_renamed[novos_nomes].copy()
df_export.to_excel('calculo_medias.xlsx', index=False)
print("Arquivos salvos!")

Arquivos salvos!


## Enviar Email

In [74]:
import win32com.client as win32
import os
# destinatarios_labor = 'hugo.lopes@laborrural.com; vanessa.martins@laborrural.com; lorena.carneiro@laborrural.com; mateus.carnielli@laborrural.com; matheus.goncalves@laborrural.com; talita.fontes@laborrural.com; analistasdedados@laborrural.com'
equipe_tecnica = 'hugo.lopes@laborrural.com; lorena.carneiro@laborrural.com; mateus.carnielli@laborrural.com; matheus.goncalves@laborrural.com; talita.fontes@laborrural.com; nadia.domingues@laborrural.com; bruno.fontes@laborrural.com; analistasdedados@laborrural.com'
# # Definir dstinatários 
destinatarios = equipe_tecnica
assunto = '[ELABORE] BD -> CALCULO MEDIAS' 
corpo = '''
Olá,

Segue a base da planilha de médias com os lançamentos até 29 de maio, data de quando foi interrompido o lançamento.

Este é o último envio com o banco de dados do Elabore antigo.
Daqui em diante, precisaremos construir a consulta ao novo banco de dados.

Fico à disposição

Atenciosamente,

Filipe Dalboni
'''
# # Caminho do arquivo Excel pra anexar
# caminho_arquivo = r'C:\\Users\\analy\\LABOR RURAL\\Analytics - Departamento Analytics\\DEMANDAS\\ELABORE\\ELABORE_CONSISTENCIA_FAZENDAS/bd_elabore_teste.xlsx'
# assert os.path.exists(caminho_arquivo), 'Arquivo Excel não encontrado!'

caminho_arquivo_consistencia = r'C:/Users/analy/LABOR RURAL/Analytics - Departamento Analytics/FERRAMENTAS/ELABORE/DEMANDAS_GERAIS/ELABORE_CONSISTENCIA_FAZENDAS/indicadores_mensais.xlsx'
assert os.path.exists(caminho_arquivo_consistencia), 'Arquivo Excel não encontrado!'

caminho_arquivo_calculo_medias = r'C:/Users/analy/LABOR RURAL/Analytics - Departamento Analytics/FERRAMENTAS/ELABORE/DEMANDAS_GERAIS/ELABORE_CONSISTENCIA_FAZENDAS/calculo_medias.xlsx'
assert os.path.exists(caminho_arquivo_calculo_medias), 'Arquivo Excel não encontrado!'

# Abre o Outlook
outlook = win32.Dispatch('outlook.application')
mail = outlook.CreateItem(0)

# Configura
mail.To = destinatarios
mail.Subject = assunto
mail.Body = corpo   # ou, se quiser corpo em HTML: mail.HTMLBody = '<b>Olá!</b> ...'

# # Adiciona anexo
# mail.Attachments.Add(caminho_arquivo)

# Adiciona anexo
mail.Attachments.Add(caminho_arquivo_consistencia)

# Adiciona anexo
mail.Attachments.Add(caminho_arquivo_calculo_medias)

# Envia
#mail.Send()   # Para abrir e revisar antes de enviar, use mail.Display()
print('Email enviado!')

Email enviado!


In [75]:
fim = time.time()
tempo_total_segundos = fim - inicio

minutos, segundos = divmod(int(tempo_total_segundos), 60)
print(f"Tempo total de execução: {minutos} minutos e {segundos} segundos")

Tempo total de execução: 7 minutos e 49 segundos
